# Bioleft — Estandarización de las 4 campañas

Auditoría, limpieza y exploración del dataset de observaciones agronómicas
(formato largo / EAV) de los ensayos participativos de tomate criollo.

## Setup

In [152]:
import pandas as pd
import re
import unicodedata
import numpy as np
from IPython.display import display
import io
from functools import partial

## 1. Auditoría de datos

Primer vistazo al dataset: cuántas filas, cuántas campañas y cuántas
variables únicas hay antes de empezar a estandarizar.

### 1.1 Leemos el dataset y hacemos una vista previa
El dataset que leemos es uno preprocesado manualmente, ya que había unos pocos evaluadores que habían realizado dos ensayos para una misma variedad (en una misma campaña y sitio), pero al generar el conjunto de datos no se los había distinguido. Se arreglaron manualmente porque eran pocos casos y para no ensuciar con más código innecesario. Estos casos se detectaron más tarde en la limpieza, pero acá leemos directamente ese archivo.

In [153]:
df = pd.read_csv('data/interim/df_limpieza_manual_observaciones_agronomicas.csv', encoding='utf-8-sig')
print(df.shape)
df.head(3)

(10313, 9)


,campaña,id_participante,id_sitio,variedad_id,variedad_original,etapa,variable,valor,nota
0,2024-2025,AA023T,AA023T,V172,172,Siembra y emergencia,Método de siembra,Siembra en almácigos,NaN
1,2024-2025,AA023T,AA023T,V172,172,Siembra y emergencia,Fecha de siembra,2024-09-02,NaN
2,2024-2025,AA023T,AA023T,V172,172,Siembra y emergencia,Cantidad de semillas sembradas,10,NaN


### 1.2 Observaciones que ocupan 2 o más líneas porque tienen un salto de línea en su contenido
Esto explica porque al ver el csv vemos 10317 líneas (sin contar la de los nombres de las columnas) y porque pandas al leerlo nos dijo que tenía 10313.

In [154]:
multilinea = df[df.apply(lambda r: r.astype(str).str.contains('\n').any(), axis=1)]
for i, row in multilinea.iterrows():
    print(i, '|', repr(row['valor'][:150]))

1436 | 'Growmix de Terra Fertil | Turba de Musgo Sphagnum de fibras finas. Compost de Corteza Fina. Perlita. Corrector de pH. Fertilizante. Humectante.\r\n\r\n'
7276 | 'Fertilicé con humus de lombriz casero y té de compost aireado 24 hs 1 vez por semana\r\n'
9777 | '\r\nPoco sabor'


### 1.3 Observaciones por campaña

In [155]:
temporadas = df['campaña'].value_counts().sort_index()
print(f"Total de campañas: {df['campaña'].nunique()}")
temporadas

Total de campañas: 4


campaña
2022-2023    4156
2023-2024    2956
2024-2025    1495
2025-2026    1706
Name: count, dtype: int64

### 1.4 Variables únicas y su frecuencia (sin estandarizar todavía)

In [156]:
variables = df['variable'].value_counts().sort_index()
print(f"Total de variables únicas: {df['variable'].nunique()}")
variables

Total de variables únicas: 123


variable
#emergidas                                                                                                           116
#implantados                                                                                                          30
#plantines trasplantados                                                                                              73
#por contenedor                                                                                                       62
#semillas sembradas                                                                                                  128
                                                                                                                    ... 
comentarios                                                                                                           22
¿A qué crees que se debe la uniformidad observada?                                                                    36
¿Cuánto te gusta este t

## 2. Limpieza y preprocesamiento de datos 

Objetivos de esta sección:
1. Eliminar o tratar observaciones duplicadas, si es que hay

2. Estandarizar los nombres de variables: mapear variables duplicadas (miden lo mismo pero tienen distinto nombre según la campaña) a un mismo nombre corto. 

3. Limpieza y estandarización de los valores que toman las variables: Unificar categorías corrigiendo errores de tipeo, asegurar consistencia en los valores arreglando datos ingresados en la variable incorrecta o tipos de datos inconsistentes dentro de una misma variable.

**Valores faltantes**: el tratamiento se realizó a la par de la fase anterior. Muchas veces se buscó en las planillas y se recuperaron datos en variables que tenían "nan" y también se agregaron observaciones que no se habían guardado. Sin embargo, hubo algunos casos que fue imposible recuperar el dato por lo que se decidió eliminar esas observaciones. 

**Valores atípicos**: por temas de tiempo, no se pudo hacer mucho hincapié en observar relaciones entre variables y la detección de estos casos atípicos (solamente mediante las visualizaciones finales del tablero).

**Criterio de nombre estándar:** uso el nombre de la campaña más
reciente como estándar, y convierto los nombres de campañas
anteriores a ese estándar cuando difieren.

**Metodología:** para cada par de variables candidatas a ser la
misma, comparo los valores que toman (y en algunos casos la etapa
del cultivo en la que aparecen, porque puede haber nombres parecidos
que en realidad se refieren a etapas distintas) para confirmar que
realmente representan lo mismo antes de fusionarlas.

### 2.1 Tratamiento de observaciones duplicadas y de incompatibilidad

No solo se observó que no hubiera duplicados, sino también la consistencia y que no haya múltiples respuestas para una misma variable que sean incompatibles. Esto se verificó de esta forma debido a que se entendía que era un conjunto de datos muy heterogeneo y no bastaba con simplemente mirar duplicados totales. Por eso no agregamos la columna "valor" en esta búsqueda.

Tras la primera revisión se comprendió que no había duplicados, sino que las filas que parecían supuestos duplicados correspondían a 2 casos: 
- Pertenecían a experimentos distintos que realizó un usuario durante una misma campaña para una variedad y que no se distinguieron al momento de armar el conjunto de datos. *Se resolvieron manualmente agregando "_1" y "_2" (porque eran pocos casos) al final de id_sitio para separar los experimentos distintos. Posteriormente, usando el id_sitio modificado, se agrego una columna "sub_ensayo" para identificarlos y que se suban a la base de datos cada uno con su propio ensayo_id y se elimaron esos sufijos en id_sitio ("_1" y "_2").*
- Todas las variables del dataset venían sin explotar (los distintos valores que toman no estaban expresados en distintas filas), a excepción de la variable sustrato utilizado en la campaña 2024-2025, lo que dejó muchas dudas sobre la integridad de los datos.

In [157]:
def procesar_sitios_y_subensayos(df_in: pd.DataFrame) -> pd.DataFrame:
    """Extraemos el sufijo de id_sitio, sí no tiene se le asina 1"""
    df_out = df_in.copy()
    
    sub_num = df_out['id_sitio'].astype(str).str.extract(r'_(\d+)$')[0]
    df_out['sub_ensayo'] = sub_num.fillna(1).astype(int)
    df_out['id_sitio'] = df_out['id_sitio'].astype(str).str.replace(r'_\d+$', '', regex=True)

    cols = df_out.columns.tolist()
    if 'sub_ensayo' in cols:
        cols.remove('sub_ensayo')
    idx_sitio = cols.index('id_sitio')
    cols.insert(idx_sitio + 1, 'sub_ensayo')

    return df_out[cols]

df = procesar_sitios_y_subensayos(df)

##### Resultados:
- Las 54 filas restantes se deben a la variable "sustrato" explotada.

In [158]:
# Definimos las columnas que determinan una observación única
all_cols = [
    'campaña', 
    'id_participante', 
    'id_sitio', 
    'sub_ensayo',
    'variedad_id', 
    'variedad_original', 
    'etapa', 
    'variable',
]

duplicados = df[df.duplicated(subset=all_cols, keep=False)]

print(f"Total de filas involucradas en duplicados: {len(duplicados)}")

if not duplicados.empty:
    duplicados_ordenados = duplicados.sort_values(by=all_cols)
    
    cols_mostrar = all_cols + ['valor', 'nota']
    print("\n--- Vista previa de registros duplicados ---")
    print(duplicados_ordenados[cols_mostrar].to_string(index=False))
else:
    print("No se encontraron registros duplicados con esa combinación de columnas.")

Total de filas involucradas en duplicados: 54

--- Vista previa de registros duplicados ---
  campaña id_participante id_sitio  sub_ensayo variedad_id                                              variedad_original               etapa                                                                variable                   valor nota
2024-2025          AA001T   AA001T           1        V172                                                            172 Trasplante a maceta Sustrato utilizado (marcá todo lo que hayas utilizado en tu contenedor)                  Tierra  NaN
2024-2025          AA001T   AA001T           1        V172                                                            172 Trasplante a maceta Sustrato utilizado (marcá todo lo que hayas utilizado en tu contenedor)                 Compost  NaN
2024-2025          AA001T   AA001T           1        V172                                                            172 Trasplante a maceta Sustrato utilizado (marcá todo lo que

### 2.2 Estandarización de los nombres de las variables

#### 2.2.1. Para detectar algunas de las candidatas a fusionarse, vamos a normalizar sus nombres (minúsculas, sin tildes, sin paréntesis, sin símbolos sueltos).

Con esto podemos ver que hay varias variables duplicadas (cambiaron de nombre entre campañas). Al final de todo las vamos a unificar.

In [159]:
def normalizar(texto):
    texto = texto.lower().strip()
    texto = re.sub(r'\(.*?\)', '', texto)  # sacamos lo que está entre paréntesis, ej: "(KG)"
    texto = re.sub(r'[#°ª]', '', texto)     # sacamos símbolos sueltos tipo "#"
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')  # sacamos tildes
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df['variable_norm'] = df['variable'].apply(normalizar)

grupos = df.groupby('variable_norm')['variable'].unique()
candidatos_duplicados = grupos[grupos.apply(len) > 1]
candidatos_duplicados

variable_norm
cantidad de plantulas emergidas        [Cantidad de plantulas emergidas, Cantidad de ...
comentarios                                                   [Comentarios, comentarios]
conduccion                                                      [Conducción, Conduccion]
cuanto te gusta este tomate?           [¿Cuánto te gusta este tomate? (1-9), ¿Cuánto ...
densidad                                               [Densidad (plantas/m²), Densidad]
fecha de emergencia                    [Fecha de emergencia, Fecha de emergencia (cua...
fecha de primera floracion             [Fecha de primera floración , Fecha de primera...
fertilizacion                                             [Fertilización, Fertilizacion]
peso de todos los frutos cosechados    [Peso de todos los frutos cosechados, Peso de ...
sanidad                                 [Sanidad (1 = nada sana, 5 = muy sana), Sanidad]
sustrato utilizado                     [Sustrato utilizado , Sustrato utilizado (marc...
uniform

#### 2.2.2. Confirmación de candidatas a fusionarse.

- Para los casos cuyos nombres no lo dejan tan claro, vamos a revisar y comparar de manera manual para asegurarnos de que realmente representen lo mismo a lo largo de las campañas.
- También se comparó con la toma de datos de las planillas originales, principalmente para asegurarnos de que la pregunta y la escala se correspondan y para algunos nombres se consultó al equipo de Bioleft para asegurar una correcta unificación. Por ejemplo de "Fecha floracion" y "Fecha de primera floración (día en la que al menos la mitad de las plantas están en su primera floración)" y "Motivo" con "¿A qué crees que se debe la uniformidad observada?"

In [160]:
def comparar_valores(variables, top=None, dropna=True):
    """Muestra la frecuencia de valores de cada variable ORIGINAL (columna 'variable') para decidir si miden lo mismo."""
    for var in variables:
        print(f"\n=== {var} ===")
        conteo = df.loc[df['variable'] == var, 'valor'].value_counts(dropna=dropna)
        print(conteo.head(top) if top else conteo)

Cobertura vs. Cobertura de suelo

In [161]:
comparar_valores(["Cobertura", "Cobertura de suelo"], dropna=False)


=== Cobertura ===
valor
vegetal              27
sin cobertura        27
cobertura vegetal     7
plástica              3
sin                   1
Name: count, dtype: int64

=== Cobertura de suelo ===
valor
cobertura vegetal    78
sin cobertura        40
0.17                  1
0.08                  1
Name: count, dtype: int64


Motivo vs. ¿A qué crees que se debe la uniformidad observada?

In [162]:
comparar_valores(['Motivo', '¿A qué crees que se debe la uniformidad observada?'])


=== Motivo ===
valor
genética                                                                                                                                                       5
Tal vez el contenedor no fue el adecuado                                                                                                                       3
quizás no estaba bien tutorada                                                                                                                                 3
por su naturaleza y alimentación                                                                                                                               3
en su mayoria todas tienen un mismo crecimiento,en cuanto a tamaño y sanidad.                                                                                  2
Por la orientación algunas plantas reciben menos sol directo que otras.\t                                                                                      2
a las enferm

Implantados / trasplantados y Ubicación — comparación por etapa

Acá comparo por `etapa` en vez de por `valor`, justamente para
descartar que sean variables de etapas distintas.

In [163]:
for var in ['#implantados', 'Cantidad de plantines implantados', 'Cantidad de plantas implantadas',
            'Ubicación del contenedor', 'Ubicación', 'Ubicación de la parcela']:
    print(var, '->', df[df['variable']==var]['etapa'].unique())

#implantados -> ['Trasplante a maceta']
Cantidad de plantines implantados -> ['Trasplante a maceta']
Cantidad de plantas implantadas -> ['Trasplante a suelo']
Ubicación del contenedor -> ['Trasplante a maceta']
Ubicación -> ['Trasplante a suelo']
Ubicación de la parcela -> ['Trasplante a suelo']


Malezas / Plagas / Brote apical

In [164]:
comparar_valores(['Malezas', 'Control de malezas', 'Plagas', 'Control de plagas', 'Plagas observadas',
                   'Eliminación del brote apical', 'Brote apical'], top=8)


=== Malezas ===
valor
control mecánico         74
sin control              16
otro                     11
herbicidas sintéticos     2
Name: count, dtype: int64

=== Control de malezas ===
valor
control mecánico    108
sin control          13
Control Mecánico      4
control manual        2
manual                1
Name: count, dtype: int64

=== Plagas ===
valor
biopreparados              46
sin control                24
control mecánico           20
otro                        5
preparados biodinámicos     5
plaguicidas sintéticos      3
Name: count, dtype: int64

=== Control de plagas ===
valor
control mecánico           41
sin control                31
biopreparados              30
preparados biodinámicos    13
plaguicidas sintéticos      3
Manualmente                 3
otro                        3
Name: count, dtype: int64

=== Plagas observadas ===
valor
ninguna                                       8
Babosas                                       5
caracoles                        

Sustrato vs. 'Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.)'

In [165]:
comparar_valores(['Sustrato', 'Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.)'], top=8)


=== Sustrato ===
valor
tierra, compost                    19
tierra, compost, perlita           16
tierra, perlita                     8
tierra, compost, turba, suelo       6
tierra, compost, perlita, suelo     5
tierra, compost, turba              5
compost                             4
compost, perlita, turba             4
Name: count, dtype: int64

=== Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.) ===
valor
Mezcla de tierra, compost y turba en proporciones iguales.    3
tierra, compost, perlita                                      3
tierra del patio, ceniza, y tierra negra comprada.            2
sustrato comercial grow y compost                             2
tierra                                                        1
Tierra + Compost                                              1
sustrato comercial grow y compost                             1
Name: count, dtype: int64


Tolerancia al sol

In [166]:
comparar_valores(['Tolerancia sol', 'Tolerancia a la quemadura del sol (1 = nada tolerante, 5 = muy tolerante)'])


=== Tolerancia sol ===
valor
5      27
4      22
3      12
4.5    11
2.5     7
1       6
3.5     4
2       4
1.5     4
Name: count, dtype: int64

=== Tolerancia a la quemadura del sol (1 = nada tolerante, 5 = muy tolerante) ===
valor
4             41
3             23
5             20
4.5            7
2              6
04/05/2024     5
2.5            2
02/05/2024     2
03/05/2023     2
1              1
03/05/2024     1
01/05/2024     1
04/05/2023     1
Name: count, dtype: int64


Tolerancia a la sequía

In [167]:
comparar_valores(['Tolerancia sequia', 'Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante)'])


=== Tolerancia sequia ===
valor
5      31
4      20
3      17
2.5    14
3.5     6
2       5
1       4
1.5     1
Name: count, dtype: int64

=== Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante) ===
valor
4             44
3             34
5             12
3.5            6
03/05/2024     4
02/05/2024     3
4.5            3
02/05/2023     2
2.5            2
1              2
04/05/2024     2
2              1
01/05/2024     1
03/05/2023     1
04/05/2023     1
Name: count, dtype: int64


#### 2.2.3. Diccionarios de unificación de variables

Se armó en base a las deducciones del apartado anterior.

Se escribe como `{nombre estándar: [nombres que aparecen en las campañas]}` y se invierte a `{alias: nombre estándar}`, así cada variable estándar aparece una sola vez. Acá también se definen las constantes `VAR_*` con los nombres estándares (de la última campaña) que se reutilizan en otras secciones.

En este momento no se tenía conocimiento de que había sido generado por Claude (a partir de las planillas) todo el conjunto de datos y se pensaba que se habían validado estos nombres. Por eso se trabajo de esta forma, ahora comprendo que era mejor pasar a slugs desde el principio.

In [168]:
VAR_EMERGENCIA = 'Fecha de emergencia (cuando emerge el 70% de las plántulas)'
VAR_SUSTRATO = 'Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.)'
VAR_DENSIDAD = 'Densidad (plantas/m²)'
VAR_FLORACION = 'Fecha de primera floración (día en la que al menos la mitad de las plantas están en su primera floración)'
VAR_ESTRES_HIDRICO = 'Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante)'
VAR_SOL = 'Tolerancia a la quemadura del sol (1 = nada tolerante, 5 = muy tolerante)'
VAR_SANIDAD = 'Sanidad (1 = nada sana, 5 = muy sana)'
VAR_UNIFORMIDAD = 'Uniformidad (1 = nada uniforme, 5 = muy uniforme)'
VAR_CAUSAS = '¿A qué crees que se debe la uniformidad observada?'
VAR_POSCOSECHA = 'Duración poscosecha (días que dura una muestra de 10 tomates en estado para ser consumido)'
VAR_PESO = 'Peso de todos los frutos cosechados (KG)'
VAR_GUSTO = '¿Cuánto te gusta este tomate? (1 = me disgusta muchísimo, 5 = ni me gusta ni me disgusta, 9 = me gusta muchísimo)'

equivalencias = {
    # --- Siembra y emergencia ---
    'Cantidad de semillas sembradas': ['#semillas sembradas'],
    'Cantidad de plántulas emergidas': ['Cantidad de plantulas emergidas', '#emergidas'],
    VAR_EMERGENCIA: ['Fecha emergencia', 'Fecha de emergencia'],

    # --- Trasplante a maceta ---
    'Profundidad del contenedor (cm)': ['Profundidad'],
    VAR_SUSTRATO: [
        'Sustrato utilizado ', 
        'Sustrato utilizado (marcá todo lo que hayas utilizado en tu contenedor)',
        'Sustrato',
    ],  
    'Cantidad de plantines trasplantados': ['#plantines trasplantados'],
    'Cantidad de plantines implantados': ['#implantados'],

    # --- Trasplante a suelo ---
    'Ubicación de la parcela': ['Ubicación'],
    'Superficie de la parcela (m²)': ['Superficie'],
    VAR_DENSIDAD: ['Densidad'],
    'Cobertura de suelo': ['Cobertura'],

    # --- Labores ---
    'Conducción': ['Conduccion'],
    'Fertilización': ['Fertilizacion'],
    'Control de malezas': ['Malezas'],
    'Control de plagas': ['Plagas'],

    # --- Crecimiento y desarrollo ---
    VAR_FLORACION: ['Fecha de primera floración', 'Fecha de primera floración ', 'Fecha floracion'],
    VAR_ESTRES_HIDRICO: ['Tolerancia sequia'],
    VAR_SOL: ['Tolerancia sol'],
    VAR_SANIDAD: ['Sanidad'],
    VAR_UNIFORMIDAD: ['Uniformidad'],
    VAR_CAUSAS: ['Motivo'],

    # --- Labores durante fructificación ---
    'Raleo de frutos': ['Raleo'],
    'Eliminación del brote apical': ['Brote apical'],

    # --- Cosecha y poscosecha ---
    **{f'Fecha de cosecha #{n}': [f'{ordinal} fecha']
       for n, ordinal in enumerate(['1ra', '2da', '3ra', '4ta', '5ta', '6ta'], start=1)},
    'Color del fruto al momento de cosechar': ['Color'], 
    VAR_POSCOSECHA: ['Poscosecha'],
    VAR_PESO: ['Peso de todos los frutos cosechados'],

    # --- Degustación ---
    'Olor a tomate maduro': ['Olor maduro'],
    VAR_GUSTO: ['¿Cuánto te gusta este tomate? (1-9)', 'Gusto'],
    'Olor a tomate verde': ['Olor verde'],

    # --- Comentarios ---
    'Comentarios generales': ['comentarios', 'Comentarios'],
}

# Invertimos a {alias: nombre estándar} porque es lo que necesita .replace()
mapeo_variables = {alias: estandar for estandar, aliases in equivalencias.items() for alias in aliases}

df['variable_limpia'] = df['variable'].replace(mapeo_variables)

Para hacer la distinción de las variables que tienen exactamente el mismo nombre pero provienen de etapas distintas, hacemos un diccionario de mapeo aparte

In [169]:
mapeo_por_etapa = {
    'Trasplante a maceta': {
        'Fecha trasplante': 'Fecha de trasplante (maceta)',
        'Fecha de trasplante': 'Fecha de trasplante (maceta)',

        'Cultivo asociado': 'Cultivo asociado (maceta)',
    },
    'Trasplante a suelo': {
        'Fecha de trasplante': 'Fecha de trasplante (suelo)',
        'Cultivo asociado': 'Cultivo asociado (suelo)',
    },
}

def mapear(row):
    return mapeo_por_etapa.get(row['etapa'], {}).get(row['variable'], mapeo_variables.get(row['variable'], row['variable']))

df['variable_limpia'] = df.apply(mapear, axis=1)

In [170]:
print(f"Variables distintas: Antes: {df['variable'].nunique()} — Después: {df['variable_limpia'].nunique()}")
df.head(2)

Variables distintas: Antes: 123 — Después: 80


,campaña,id_participante,id_sitio,sub_ensayo,variedad_id,variedad_original,etapa,variable,valor,nota,variable_norm,variable_limpia
0,2024-2025,AA023T,AA023T,1,V172,172,Siembra y emergencia,Método de siembra,Siembra en almácigos,NaN,metodo de siembra,Método de siembra
1,2024-2025,AA023T,AA023T,1,V172,172,Siembra y emergencia,Fecha de siembra,2024-09-02,NaN,fecha de siembra,Fecha de siembra


#### 2.2.4 Resultado de los mapeos 

##### Variables únicas y su frecuencia después de estandarizar

In [171]:
print(f"Shape: {df.shape}")
variables_limpias = df['variable_limpia'].value_counts().sort_index()
print(f"Total de variables únicas: {df['variable_limpia'].nunique()}")
for var, cant in variables_limpias.items():
    print(f"{var}: {cant}")

Shape: (10313, 12)
Total de variables únicas: 80
#por contenedor: 62
Aguachento: 49
Cantidad de frutos cosechados: 67
Cantidad de plantas implantadas: 55
Cantidad de plantas trasplantadas a la parcela: 117
Cantidad de plantines implantados: 54
Cantidad de plantines trasplantados: 157
Cantidad de plántulas emergidas: 302
Cantidad de semillas sembradas: 342
Cobertura de suelo: 185
Color del fruto al momento de cosechar: 197
Comentarios generales: 778
Conducción: 202
Control de malezas: 231
Control de plagas: 227
Cultivo antecesor: 56
Cultivo asociado (maceta): 62
Cultivo asociado (suelo): 141
Densidad (plantas/m²): 117
Desbrote: 241
Distancia plantas: 59
Distancia surcos: 51
Dulce: 123
Duración poscosecha (días que dura una muestra de 10 tomates en estado para ser consumido): 100
Eliminación del brote apical: 219
Fecha de cosecha #1: 148
Fecha de cosecha #10: 14
Fecha de cosecha #2: 106
Fecha de cosecha #3: 73
Fecha de cosecha #4: 48
Fecha de cosecha #5: 41
Fecha de cosecha #6: 34
Fecha 

Cantidad de variables por etapa y de observaciones registradas para cada una:

##### Siembra y emergencia: 5
- Método de siembra: 139
- Fecha de siembra: 344
- Cantidad de semillas sembradas: 342
- Cantidad de plántulas emergidas: 302
- Fecha de emergencia (cuando emerge el 70% de las plántulas): 278

##### Trasplante a maceta: 9
- Ubicación del contenedor: 166
- Profundidad del contenedor (cm): 132
- Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.): 252
- Fecha de trasplante (maceta): 146
- Cantidad de plantines trasplantados: 157
- Cantidad de plantines implantados: 54
- Cultivo asociado (maceta): 62
- Volumen: 57
- #por contenedor: 62

##### Trasplante a suelo: 12
- Ubicación de la parcela: 189
- Superficie de la parcela (m²): 141
- Fecha de trasplante (suelo): 118
- Cantidad de plantas trasplantadas a la parcela: 117
- Densidad (plantas/m²): 117
- Cobertura de suelo: 185
- Cultivo asociado (suelo): 141
- Cantidad de plantas implantadas: 55
- Surcos: 53
- Distancia surcos: 51
- Distancia plantas: 59
- Cultivo antecesor: 56

##### Labores: 7
- Desbrote: 241
- Tutorado: 247
- Conducción: 202
- Fertilización: 243
- Control de malezas: 231
- Riego: 256
- Control de plagas: 227

##### Crecimiento y desarrollo: 8
- Fecha de primera floración (día en la que al menos la mitad de las plantas están en su primera floración): 172
- Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante): 216
- Tolerancia a la quemadura del sol (1 = nada tolerante, 5 = muy tolerante): 209
- Sanidad (1 = nada sana, 5 = muy sana): 217
- Muerte por virus rugoso del tomate: 105
- Uniformidad (1 = nada uniforme, 5 = muy uniforme): 184
- ¿A qué crees que se debe la uniformidad observada?: 70
- Plagas observadas: 81

##### Labores durante fructificación: 2
- Raleo de frutos: 182
- Eliminación del brote apical: 219

##### Cosecha y poscosecha: 20
- Fecha de cosecha #1: 148
- Fecha de cosecha #2: 106
- Fecha de cosecha #3: 73
- Fecha de cosecha #4: 48
- Fecha de cosecha #5: 41
- Fecha de cosecha #6: 34
- Fecha de cosecha #7: 23
- Fecha de cosecha #8: 20
- Fecha de cosecha #9: 17
- Color del fruto al momento de cosechar: 197
- Duración poscosecha (días que dura una muestra de 10 tomates en estado para ser consumido): 100
- Cantidad de frutos cosechados: 67
- Peso de todos los frutos cosechados (KG): 57
- Fecha de cosecha #10: 14
- Rendimiento 1: 38
- Rendimiento 2: 28
- Rendimiento 3: 12
- Rendimiento 4: 3
- Rendimiento 5: 3
- Rendimiento 6: 2

##### Degustación: 16
- Dulce: 123
- Ácido: 87
- Jugoso: 154
- Persistente: 80
- Olor a tomate maduro: 121
- Mucha pulpa: 99
- Pulpa blanda: 102
- Mucha semilla: 108
- Piel dura: 104
- ¿Cuánto te gusta este tomate? (1 = me disgusta muchísimo, 5 = ni me gusta ni me disgusta, 9 = me gusta muchísimo): 164
- Olor a tomate verde: 32
- Poca pulpa: 68
- Pulpa dura: 43
- Aguachento: 49
- Pastoso: 34
- Poca semilla: 59

##### Comentarios
- Comentarios generales: 778


**Observaciones totales: 10.313**

#### 2.2.5 Hacemos una copia del dataframe para trabajar los cambios de variables y a la vez poder ir comparando con los datos "originales".

In [172]:
df_limpio = df.copy()

### 2.3 Limpieza y estandarización de los valores que toman las variables

Durante esta etapa se desarrollaron varias acciones diferentes para resolver inconsistencias, faltantes, mal cargados, incompatibles, etc. Algunas de estas actividades fueron:
- Revisar las planillas meticulosamente de carga de los usuarios para:
    - Imputar valores cuando eran inconsistentes.
    - Corregir muchas observaciones que se cargaron con el mes y el día intercambiados porque no había estandarización en el formato de las fechas.
    - Eliminar observaciones que tenían datos erróneos o inconsistentes con otros datos ingresados para el mismo ensayo (no se pudieron recuperar).
    - Mapear valores a mano de un usuario "2025-2026,AB013T", ya que el evaluador escribió fechas incoherentes y el modelo de IA las detectó y las corrigió, pero a la vez modificó fechas de este evaluador que sí eran correctas.
    - Recuperar fechas mal cargadas por el modelo de IA: los evaluadores no habían completado el año y el modelo de IA los infirió incorrectamente.
    - Analizar, estandarizar y sumar las variables de "Rendimiento X" en la campaña de 2022-2023 que generalmente representaban los distintos pesos cosechados (algunos en kg otros en g). Pero también había textos libres y observaciones sobre la cantidad de frutos cosechados y no el peso. Se mapearon según correspondían a "Peso de todos los frutos cosechados (KG)" o "Cantidad total de frutos cosechados". Pero algunos se tuvieron que borrar.

- Imputar fechas que estaban en texto "Fines de Diciembre", "Principios de Octubre", etc. con fechas aproximadas del año correspondiente. Esto tiene un margen de error mediano, pero al no tener tantos datos priorizamos su imputación antes que su eliminación.

- Quedarnos únicamente con la primera y la última fecha de cosecha cada ensayo (las fechas podían ir de 1 a 10 dependiendo la campaña). Si un ensayo solo tiene una fecha de cosecha, esa se elige para ambas variables. 

- Resolver inconsistencias en las variables que usaban escalas numéricas, principalmente en las 4 de "Crecimiento y desarrollo" (por ejemplo, Tolerancia al sol), ya que tenían datos que se habían parseado como fechas, pero que en realidad eran decimales.

En valores faltantes o inconsistentes se priorizó su imputación (mediante aproximaciones, revisiones muy cuidadosas, consultas al equipo o a las planillas originales) antes que su eliminación, asegurando la integridad de los datos y priorizando su conservación para evitar una reducción innecesaria del volumen de información disponible. Por estos motivos, esta etapa consumió un tiempo significativamente mayor al previsto.

Este proceso se realizó viendo por separado cada variable, podría haber sido mejor realizarlo en conjunto, teniendo en cuenta principalmente los tipos de datos de las variables y los valores que tomaban, no obstante, la falta de estandarización y la alta heterogeneidad entre los datos de una misma variable hacia que este proceso resulte muy complicado.

#### 2.3.1 Valores en la columna **nota**, **variedad_id** y **variedad_original**

La columna nota cuenta con valores que dicen *campaña histórica*, que son inventados por el modelo de IA cuando armó el conjunto de datos, así que procedemos a despejarlos para utilizar esta columna más adelante

In [173]:
cantidad_borradas = (df_limpio['nota'] == 'campaña histórica').sum()
print(cantidad_borradas)
df_limpio['nota'] = df_limpio['nota'].replace('campaña histórica', None)

7026


Respecto a la columna variedad_id, abajo se describen algunas inconsistencias detectadas y su solución.

| Caso | Qué pasaba | Qué se hizo |
|---|---|---|
| `VJDP` | Es la misma variedad que `V56`, escrita de otra forma | Se mapea a `V56` |
| `V156` | Tenía solo 3 observaciones que, mirando los datos, pertenecen a `V56` | Se corrigió a mano en `df_limpieza_manual_observaciones_agronomicas` (quedó como `V56`) |
| `VNOID` (AA003T, 2025-2026) | 2 filas sueltas de Degustación por un fallo humano del evaluador al desplazarse entre columnas al completar los datos | Se eliminan |
| H035T (2023-2024) | Mirando su planilla original se observó que completó solo `V7`, pero al armar el dataset se cargaron observaciones "fantasma" de otras variedades | Se eliminan las que no son `V7` |
| H015T (2023-2024) | Mismo problema: se cargó una variedad fantasma `V17` | Se eliminan |

Al final se muestra una tabla de antes y después con las combinaciones campaña–variedad que cambiaron.

In [174]:
def frecuencia_variedades(dataset):
    return dataset.groupby(['campaña', 'variedad_id']).size()


def eliminar_filas(dataset, mascara_fn, descripcion, informar=True):
    """Elimina las filas que cumplen `mascara_fn(dataset)` y resetea el índice."""
    mascara = mascara_fn(dataset)
    if informar:
        print(f"{descripcion}: {mascara.sum()} filas eliminadas")
    return dataset[~mascara].reset_index(drop=True)

In [175]:
antes = frecuencia_variedades(df_limpio)

# Mapeamos VJDP -> V56 y Eliminamos las filas fantasmas
for d in (df, df_limpio):
    es_vjdp = d['variedad_id'].astype(str).str.strip().str.upper().str.contains('VJDP', na=False)
    d.loc[es_vjdp, 'variedad_id'] = 'V56'

filas_a_eliminar = {
    'VNOID fantasma (AA003T, Degustación, 2025-2026)': lambda d: (
        (d['campaña'] == '2025-2026') & (d['id_participante'] == 'AA003T')
        & (d['variedad_id'] == 'VNOID') & (d['etapa'] == 'Degustación')
    ),
    'Variedades fantasma de H035T (2023-2024, todas menos V7)': lambda d: (
        (d['campaña'] == '2023-2024') & (d['id_participante'] == 'H035T')
        & (d['variedad_id'] != 'V7')
    ),
    'V17 fantasma de H015T (2023-2024)': lambda d: (
        (d['campaña'] == '2023-2024') & (d['id_participante'] == 'H015T')
        & (d['variedad_id'] == 'V17')
    ),
}

for descripcion, mascara_fn in filas_a_eliminar.items():
    df_limpio = eliminar_filas(df_limpio, mascara_fn, descripcion)
    df = eliminar_filas(df, mascara_fn, descripcion, informar=False)

# Por último, Comparamos lo que cambio 
despues = frecuencia_variedades(df_limpio)
comparacion = (
    pd.concat([antes.rename('antes'), despues.rename('después')], axis=1)
    .fillna(0).astype(int)
)
comparacion['diferencia'] = comparacion['después'] - comparacion['antes']
display(comparacion[comparacion['diferencia'] != 0])

VNOID fantasma (AA003T, Degustación, 2025-2026): 2 filas eliminadas
Variedades fantasma de H035T (2023-2024, todas menos V7): 8 filas eliminadas
V17 fantasma de H015T (2023-2024): 3 filas eliminadas


antes  después  diferencia
campaña   variedad_id                            
2023-2024 V17              3        0          -3
          V33             72       70          -2
          V56            872      918          46
          V65              2        0          -2
          V67              2        0          -2
          VJDP            48        0         -48
2025-2026 VNOID            2        0          -2

Por último, también se detectó que en `variedad_original`, el mismo ensayo tenía distintas etiquetas (`172` y `172.0`) y eso lo partía en dos "ensayos" incompletos (AA023T y H014T, 2023-2024). Se unifican todos estos valores.

In [176]:
df_limpio['variedad_original'] = (
    df_limpio['variedad_original'].astype(str).str.replace(r'^(\d+)\.0$', r'\1', regex=True)
)

#### 2.3.2 Funciones reutilizables

Definimos las funciones centrales que se reutilizan a lo largo de esta sección:

| Función | Para qué sirve |
|---|---|
| `inspeccionar_valores` | Muestra todos los valores únicos de una variable con su frecuencia (se usa antes y después de limpiar). |
| `limpiar_variable` | **Motor general**: aplica una función a una variable, opcionalmente renombra `variable`, explota respuestas múltiples (`explotar=True`), borra nulos (`borrar_nulos=True`) e inspecciona el resultado. |
| `limpiador_categorico`, `limpiador_entero`, `limpiador_fecha` | Generan la función de limpieza a partir de un diccionario de mapeo (lo único que cambia entre variables). Lo que no se pueda mapear queda como `SIN_MAPEAR:<texto>`. |
| `mapear_variable_texto`, `mapear_variable_numerica`, `limpiar_dimensiones_numericas` | Variantes para categóricas con reemplazo exacto, cuantitativas y dimensiones con unidades (`"30 cm"`). |
| `mapear_fila` / `normalizar_texto` | Separan respuestas múltiples (`"Tierra, compost + turba"`) y las llevan a categorías. |
| `asignar_valor`, `aplicar_correcciones_manuales` | Correcciones puntuales por participante / variedad, con auditoría de lo que se creó o cambió. |

Cada etapa del cultivo tiene su propia sección más abajo: se mira el valor crudo con `inspeccionar_valores(df, ...)`, se define el mapeo y se aplica con `limpiar_variable(df_limpio, ...)`, que muestra cómo quedó. Las funciones que son específicas de una sola variable (por ejemplo `extraer_cultivos` o `extraer_poscosecha`) viven junto a esa variable.

In [177]:
def encabezado_antes():
    print(f"\n{'=' * 70}\n🔹 ANTES (datos originales)\n{'=' * 70}")


def encabezado_despues():
    print(f"\n{'-' * 70}\n🔸 DESPUÉS (datos limpios)\n{'-' * 70}")


def inspeccionar_valores(dataset: pd.DataFrame, variable_limpia: str, momento: str = 'auto') -> None:
    """
    Imprime TODOS los valores únicos de la variable con su frecuencia (incluye NaN), sin truncar.
    Utiliza 'variable_limpia' si existe; de lo contrario, utiliza 'variable'.

    momento: 'antes' | 'despues' | None | 'auto'.
    """
    if momento == 'auto':
        momento = 'antes' if dataset is globals().get('df') else 'despues'
    if momento == 'antes':
        encabezado_antes()
    elif momento == 'despues':
        encabezado_despues()

    columna_variable = 'variable_limpia' if 'variable_limpia' in dataset.columns else 'variable'

    print(f"\n=== {variable_limpia} ===")

    conteo = dataset.loc[dataset[columna_variable] == variable_limpia, 'valor'].value_counts(dropna=False)
    for valor, frecuencia in conteo.items():
        print(f"Valor: {valor}, Frecuencia: {frecuencia}")
    print("Observaciones totales: ", conteo.sum())


def normalizar_texto(texto):
    """Minúsculas, sin tildes, sin espacios repetidos y sin punto final. NaN se devuelve igual."""
    if pd.isna(texto):
        return texto
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'\s+', ' ', texto)
    texto = texto.rstrip('.')
    return texto


def mapear_fila(valor_original, overrides, mapeo_componentes):
    """
    Mapea una celda con respuestas múltiples ("Tierra, compost + turba") a una lista de categorías.
    - overrides: {texto_normalizado_completo: [cat1, cat2]} para casos irregulares.
    - mapeo_componentes: {componente_normalizado: categoría}; lo que no esté queda como 'SIN_MAPEAR:<texto>'.
    Devuelve la lista sin duplicados y respetando el orden.
    """
    if pd.isna(valor_original):
        return []
    norm = normalizar_texto(valor_original)

    if norm in overrides:
        return overrides[norm]

    resultado = []
    for p in re.split(r',|\+', norm):
        p = re.sub(r'^(y|e)\s+', '', p.strip())  # saca conectores al inicio
        if not p:
            continue
        resultado.append(mapeo_componentes.get(p, f'SIN_MAPEAR:{p}'))

    return list(dict.fromkeys(resultado))


def mapear_variable_texto(dataset, variable_limpia, diccionario_mapeo, mostrar=True):
    """Variables categóricas simples: strip + reemplazo exacto (lo que no está en el diccionario queda igual)."""
    df_out = dataset.copy()
    mask = df_out['variable_limpia'] == variable_limpia
    df_out.loc[mask, 'valor'] = df_out.loc[mask, 'valor'].astype(str).str.strip()
    df_out.loc[mask, 'valor'] = df_out.loc[mask, 'valor'].replace(diccionario_mapeo)
    if mostrar:
        inspeccionar_valores(df_out, variable_limpia)
    return df_out


def mapear_variable_numerica(dataset, variable_limpia, diccionario_mapeo=None, descartar_contiene=(),
                             tipo='entero', borrar_nulos=False, mostrar=True):
    """
    Variables cuantitativas. Orden: reemplazos del diccionario -> descarta (None) los textos que contienen
    alguno de `descartar_contiene` -> to_numeric(coerce) -> cast según `tipo`.
    tipo: 'entero' (Int64 redondeado), 'float' o None (deja lo que devuelva to_numeric).
    borrar_nulos=True elimina las filas de ESTA variable que hayan quedado nulas.
    """
    df_out = dataset.copy()
    mask = df_out['variable_limpia'] == variable_limpia

    if diccionario_mapeo:
        df_out.loc[mask, 'valor'] = df_out.loc[mask, 'valor'].replace(diccionario_mapeo)
    for patron in descartar_contiene:
        m = mask & df_out['valor'].astype(str).str.contains(patron, case=False, na=False)
        df_out.loc[m, 'valor'] = None

    serie = pd.to_numeric(df_out.loc[mask, 'valor'], errors='coerce')
    if tipo == 'entero':
        serie = serie.round().astype('Int64')
    elif tipo == 'float':
        serie = serie.astype(float)
    df_out.loc[mask, 'valor'] = serie

    if borrar_nulos:
        df_out = df_out[~(mask & df_out['valor'].isna())].reset_index(drop=True)
    if mostrar:
        inspeccionar_valores(df_out, variable_limpia)
    return df_out


def limpiar_dimensiones_numericas(dataset, variable_limpia, diccionario_mapeo, mostrar=True):
    """Dimensiones con unidades ("30 cm"): aplica el diccionario y extrae la parte numérica (entero o decimal)."""
    df_out = dataset.copy()
    mask = df_out['variable_limpia'] == variable_limpia
    df_out.loc[mask, 'valor'] = df_out.loc[mask, 'valor'].replace(diccionario_mapeo)
    valores_limpios = df_out.loc[mask, 'valor'].astype(str).str.extract(r'(\d+\.?\d*)')[0]
    df_out.loc[mask, 'valor'] = pd.to_numeric(valores_limpios, errors='coerce')
    if mostrar:
        inspeccionar_valores(df_out, variable_limpia)
    return df_out

In [178]:
# Mapeo Sí/No que se repite en varias variables (Tutorado, Desbrote, Raleo, Brote apical)
SI_NO = {'si': 'Sí', 'Si': 'Sí', 'SI': 'Sí', 'sí': 'Sí', 'Sí': 'Sí', 'SÍ': 'Sí', 'true': 'Sí', 'True': 'Sí', 'TRUE': 'Sí',
         'no': 'No', 'No': 'No', 'NO': 'No', 'false': 'No', 'False': 'No', 'FALSE': 'No'}


# ---- Generadores de funciones de limpieza (valor -> valor limpio) ----
def limpiador_categorico(mapeo, no_mapeados='marcar'):
    """
    Devuelve una función que pasa el valor a minúsculas/strip y lo busca en `mapeo` (claves en minúscula).
    NaN -> None. Si no está en el mapeo: 'marcar' -> 'SIN_MAPEAR:<texto>', 'conservar' -> el texto en
    minúscula, o cualquier otro valor literal (ej. 'Otro').
    """
    def f(val):
        if pd.isna(val):
            return None
        v = str(val).strip().lower()
        if v in mapeo:
            return mapeo[v]
        if no_mapeados == 'marcar':
            return f'SIN_MAPEAR:{v}'
        if no_mapeados == 'conservar':
            return v
        return no_mapeados
    return f


def limpiador_entero(overrides=None, invalidos='marcar', minimo=None):
    """
    Devuelve una función que convierte a entero (int(float(x))). `overrides` = {texto en minúscula: valor}.
    invalidos: 'marcar' -> 'SIN_MAPEAR:<texto>' | None -> None. minimo: valores menores pasan a None.
    """
    overrides = overrides or {}
    def f(val):
        if pd.isna(val):
            return None
        v = str(val).strip().lower()
        if v in overrides:
            return overrides[v]
        try:
            n = int(float(v))
        except (ValueError, TypeError):
            return f'SIN_MAPEAR:{v}' if invalidos == 'marcar' else None
        return None if (minimo is not None and n < minimo) else n
    return f


def limpiador_fecha(overrides=None, nulos=(), parsear=False):
    """
    Devuelve una función para fechas. `overrides` = {texto en minúscula: 'YYYY-MM-DD'}; `nulos` = textos que
    pasan a None. Si no está en ninguno: parsear=True -> parsea (dayfirst) y formatea a YYYY-MM-DD
    (si falla, 'SIN_MAPEAR:<texto>'); parsear=False -> deja el texto en mayúsculas.
    """
    overrides = overrides or {}
    def f(val):
        if pd.isna(val):
            return None
        v = str(val).strip().lower()
        if v in nulos:
            return None
        if v in overrides:
            return overrides[v]
        if parsear:
            try:
                return pd.to_datetime(v, dayfirst=True).strftime('%Y-%m-%d')
            except Exception:
                return f'SIN_MAPEAR:{v}'
        return v.upper()
    return f


# ---- Motor: aplica una función a UNA variable (por 'variable_limpia') y devuelve el df actualizado ----
def limpiar_variable(dataset, variable_limpia, funcion, nombre_variable=None, *,
                     por_fila=False, explotar=False, convertir=None,
                     borrar_nulos=False, omitir_variable=(), mostrar=True):
    """
    Reemplaza el patrón "máscara -> renombrar 'variable' -> aplicar función a 'valor' -> inspeccionar".

    Parámetros
    ----------
    funcion : callable
        valor -> valor limpio. Con por_fila=True recibe la fila completa (para usar 'nota', 'id_sitio', etc.).
    nombre_variable : str, opcional
        Nuevo nombre para la columna 'variable' (ej. 'ubicacion_parcela'). Si es None no se toca.
    explotar : False | True | 'tuplas'
        True: la función devuelve una LISTA de categorías y se genera una fila por categoría (lista vacía = se
        elimina la fila). 'tuplas': la lista tiene tuplas (nombre_variable, valor) y cada una pasa a 'variable'/'valor'.
    convertir : callable, opcional
        Conversión de toda la serie resultante (ej. pd.to_datetime).
    borrar_nulos : bool
        Elimina las filas de esta variable cuyo resultado sea nulo.
    omitir_variable : lista de nombres de 'variable'
        Filas ya procesadas (se saltean). Sirve para poder re-ejecutar celdas que explotan una variable en dos.
    mostrar : bool
        Imprime la tabla de frecuencias final y avisa si quedaron valores 'SIN_MAPEAR'.
    """
    out = dataset.copy()

    def _mascara(d):
        m = d['variable_limpia'] == variable_limpia
        if len(omitir_variable):
            m &= ~d['variable'].isin(omitir_variable)
        return m

    def _calcular(sub):
        res = sub.apply(funcion, axis=1) if por_fila else sub['valor'].apply(funcion)
        return convertir(res) if convertir is not None else res

    mask = _mascara(out)
    sub = out[mask]
    if sub.empty:
        print(f"No se encontraron filas para '{variable_limpia}'")
        return out
    nuevos = _calcular(sub)

    if explotar:
        filas = sub.assign(valor=nuevos).explode('valor')
        filas = filas[filas['valor'].notna()]
        if explotar == 'tuplas':
            filas = filas.assign(variable=filas['valor'].map(lambda t: t[0]),
                                 valor=filas['valor'].map(lambda t: t[1]))
        elif nombre_variable:
            filas = filas.assign(variable=nombre_variable)
        out = pd.concat([out[~mask], filas], ignore_index=True)
    else:
        if borrar_nulos and nuevos.isna().any():
            out = out.drop(index=nuevos.index[nuevos.isna()])
            mask = _mascara(out)
            sub = out[mask]
            nuevos = _calcular(sub) if not sub.empty else nuevos.iloc[0:0]
        out.loc[mask, 'valor'] = nuevos
        if nombre_variable:
            out.loc[mask, 'variable'] = nombre_variable
        out = out.reset_index(drop=True)

    if mostrar:
        inspeccionar_valores(out, variable_limpia)
        sin = out.loc[out['variable_limpia'] == variable_limpia, 'valor'].astype(str).str.startswith('SIN_MAPEAR')
        if sin.any():
            print(f"Quedaron {int(sin.sum())} valores SIN_MAPEAR")
    return out


# ---- Correcciones puntuales por ensayo / participante ----
def mascara_ensayo(dataset, filtros):
    """Máscara de las filas que contienen (str.contains) cada valor de `filtros` ({columna: valor})."""
    mask = pd.Series(True, index=dataset.index)
    for col, val in filtros.items():
        if col in dataset.columns:
            mask &= dataset[col].astype(str).str.contains(str(val), na=False)
    return mask


def asignar_valor(dataset, variable_limpia, nuevo_valor, valor_actual=None, **filtros):
    """
    Asigna `nuevo_valor` (IN PLACE) a las filas de `variable_limpia` que cumplan los filtros por igualdad
    (si el filtro es una lista, con isin). `valor_actual` agrega la condición valor == valor_actual.
    Devuelve la cantidad de filas modificadas. Ej.: asignar_valor(df, VAR, 0.2, campaña='2023-2024', id_participante='AA017T')
    """
    mask = dataset['variable_limpia'] == variable_limpia
    for col, val in filtros.items():
        mask &= (dataset[col].isin(val) if isinstance(val, (list, tuple, set)) else dataset[col] == val)
    if valor_actual is not None:
        mask &= dataset['valor'] == valor_actual
    dataset.loc[mask, 'valor'] = nuevo_valor
    return int(mask.sum())


def aplicar_correcciones_manuales(df_limpio, df, casos):
    """
    Aplica `casos` (lista de {'filtros': {...}, 'correcciones': {...}}) sobre df_limpio Y sobre df (el original,
    para que las inspecciones posteriores ya reflejen la corrección) e imprime una auditoría:
    filas creadas desde cero vs. filas existentes que cambiaron de valor. Devuelve (df_limpio, df).
    """
    def filas_intervenidas(d):
        partes = [d[mascara_ensayo(d, c['filtros']) & d['variable_limpia'].isin(list(c['correcciones']))].copy()
                  for c in casos]
        return pd.concat(partes, ignore_index=True)

    # --- Imprimimos el estado inicial ---
    print("--- ESTADO INICIAL (ANTES) ---")
    n_limpio_antes, n_df_antes = len(df_limpio), len(df)
    print(f"Filas en df_limpio : {n_limpio_antes}")
    print(f"Filas en df        : {n_df_antes}\n")
    antes = filas_intervenidas(df_limpio)

    # --- EJECUCIÓN ---
    for caso in casos:
        df_limpio = corregir_observaciones_manuales(
            df_limpio, caso['filtros'], caso['correcciones'],
            col_variable='variable_limpia', modo='reemplazar'
        )
        df = corregir_observaciones_manuales(
            df, caso['filtros'], caso['correcciones'],
            col_variable='variable_limpia', modo='reemplazar'
        )

    # --- Imprimimos el estado final ---
    print("--- AUDITORÍA DE CAMBIOS (DESPUÉS) ---")
    print(f"Filas agregadas netas en df_limpio : {len(df_limpio) - n_limpio_antes}")
    print(f"Filas agregadas netas en df        : {len(df) - n_df_antes}\n")
    despues = filas_intervenidas(df_limpio)

    cols_clave = ['campaña', 'id_participante', 'variedad_id', 'variable_limpia']

    nuevas = despues.merge(antes[cols_clave], on=cols_clave, how='left', indicator=True) \
                    .query('_merge == "left_only"').drop(columns=['_merge'])
    print(f"📌 1. FILAS TOTALMENTE NUEVAS CREADAS ({len(nuevas)}):")
    cols_ver = [c for c in ['campaña', 'id_participante', 'variedad_id', 'etapa', 'variable_limpia', 'valor'] if c in nuevas.columns]
    if not nuevas.empty:
        print(nuevas[cols_ver].to_string(index=False))
    else:
        print("Ninguna fila fue creada desde cero (todas las variables ya existían previamente).")

    print("\n" + "=" * 80 + "\n")

    modificadas = antes.merge(despues[cols_clave + ['valor']], on=cols_clave, suffixes=('_ANTES', '_DESPUES'))
    print(f"2. FILAS EXISTENTES QUE FUERON MODIFICADAS ({len(modificadas)}):")
    cols_mod = [c for c in ['campaña', 'id_participante', 'variedad_id', 'variable_limpia', 'valor_ANTES', 'etapa', 'valor_DESPUES'] if c in modificadas.columns]
    if not modificadas.empty:
        print(modificadas[cols_mod].to_string(index=False))
    else:
        print("No hubo modificaciones en filas preexistentes.")

    return df_limpio, df

In [179]:
def corregir_observaciones_manuales(
    df,
    filtros_base,
    correcciones,
    etapa_default=None,
    *,
    col_variable='variable_limpia',
    modo='reemplazar',
    variables_multivalor=(),
    filtro='contiene'
):
    """
    Corrige observaciones manuales en un DataFrame largo.

    Parámetros
    ----------
    df : DataFrame
        Dataset que se desea corregir.
    filtros_base : dict
        Filtros que identifican el ensayo.
    correcciones : dict
        {nombre_variable: valor} o {nombre_variable: (valor, etapa)}.
    etapa_default : str, opcional
        Etapa usada cuando se crea una variable que no existía y no se indicó
        una etapa particular.
    col_variable : str
        Columna que identifica la variable. Usar 'variable_limpia' en las
        etapas de limpieza y 'variable' en df_final_recuperado.
    modo : {'reemplazar', 'actualizar'}
        'reemplazar': elimina las filas anteriores de esa variable y crea las
        nuevas. Es el comportamiento histórico de las dos funciones.
        'actualizar': actualiza las filas existentes; si no existe, crea una.
        Para variables multivalor, 'reemplazar' es el modo recomendado.
    variables_multivalor : iterable
        Variables para las que una cadena separada por comas representa
        observaciones independientes. También admite listas explícitas.
        Las demás variables conservan las comas dentro del texto.
    filtro : {'contiene', 'exacto'}
        'contiene' conserva el comportamiento histórico con str.contains
        (incluidas expresiones regulares, como r'^56$').
        'exacto' compara por igualdad.

    Notas
    -----
    - Solo utiliza columnas que existen en el DataFrame.
    - Actualiza 'variable', 'variable_limpia' y 'variable_norm' únicamente
      cuando esas columnas están presentes.
    - Para crear una variable nueva necesita conocer la etapa, ya sea en la
      tupla de corrección o mediante etapa_default.
    """

    if col_variable not in df.columns:
        raise ValueError(
            f"No existe la columna '{col_variable}' en el DataFrame. "
            f"Columnas disponibles: {list(df.columns)}"
        )

    if modo not in {'reemplazar', 'actualizar'}:
        raise ValueError("modo debe ser 'reemplazar' o 'actualizar'.")

    if filtro not in {'contiene', 'exacto'}:
        raise ValueError("filtro debe ser 'contiene' o 'exacto'.")

    df_out = df.copy()
    variables_multivalor = set(variables_multivalor)

    def construir_mascara_base(data):
        mask = pd.Series(True, index=data.index)

        for col, val in filtros_base.items():
            if col not in data.columns:
                raise ValueError(
                    f"El filtro '{col}' no existe en el DataFrame. "
                    f"No se puede aplicar con seguridad: {filtros_base}"
                )

            if filtro == 'exacto':
                mask &= data[col].eq(val)
            else:
                # Se conserva regex=True por compatibilidad con filtros como r'^56$'.
                mask &= data[col].astype('string').str.contains(
                    str(val), na=False, regex=True
                )

        return mask

    for var_target, item_corr in correcciones.items():

        if isinstance(item_corr, tuple):
            if len(item_corr) != 2:
                raise ValueError(
                    f"La corrección de '{var_target}' debe ser (valor, etapa)."
                )
            nuevo_val, etapa_var = item_corr
        else:
            nuevo_val, etapa_var = item_corr, None

        mask_base = construir_mascara_base(df_out)

        if not mask_base.any():
            print(
                f"Advertencia: No se encontró ningún registro para los filtros: "
                f"{filtros_base}"
            )
            return df_out

        mask_var = mask_base & df_out[col_variable].eq(var_target)

        # Si la variable ya existe, conservar su etapa cuando no se indicó otra.
        if mask_var.any() and etapa_var is None and 'etapa' in df_out.columns:
            etapas_existentes = df_out.loc[mask_var, 'etapa'].dropna()
            if not etapas_existentes.empty:
                etapa_var = etapas_existentes.iloc[0]

        etapa_final = etapa_var if etapa_var is not None else etapa_default

        # Solo separar comas cuando la variable fue declarada multivalor.
        if isinstance(nuevo_val, list):
            valores_a_insertar = nuevo_val
        elif var_target in variables_multivalor and isinstance(nuevo_val, str):
            valores_a_insertar = [v.strip() for v in nuevo_val.split(',') if v.strip()]
        else:
            valores_a_insertar = [nuevo_val]

        # Actualizar valores existentes sin reconstruir las filas.
        if modo == 'actualizar' and mask_var.any():
            if len(valores_a_insertar) == 1:
                df_out.loc[mask_var, 'valor'] = valores_a_insertar[0]
            else:
                raise ValueError(
                    f"'{var_target}' recibió varios valores en modo='actualizar'. "
                    "Usá modo='reemplazar' para generar una fila por valor."
                )
            continue

        # Para crear una variable nueva, la etapa es obligatoria.
        if not mask_var.any() and etapa_final is None:
            raise ValueError(
                f"No se puede crear '{var_target}' para {filtros_base}: "
                "la variable no existía y no se especificó la etapa. "
                "Indicá (valor, 'Etapa') o etapa_default."
            )

        # Tomar una fila de referencia antes de eliminar las observaciones viejas.
        fila_plantilla = df_out.loc[mask_base].iloc[0].copy()

        if modo == 'reemplazar':
            df_out = df_out.loc[~mask_var].copy()

        nuevas_filas = []

        for val in valores_a_insertar:
            nueva_fila = fila_plantilla.copy()

            # Actualizar solamente las columnas identificadoras que existan.
            if 'variable' in df_out.columns:
                nueva_fila['variable'] = var_target
            if 'variable_limpia' in df_out.columns:
                nueva_fila['variable_limpia'] = var_target
            if 'variable_norm' in df_out.columns:
                nueva_fila['variable_norm'] = str(var_target).lower()

            nueva_fila[col_variable] = var_target
            nueva_fila['valor'] = val

            if etapa_final is not None and 'etapa' in df_out.columns:
                nueva_fila['etapa'] = etapa_final

            if 'nota' in df_out.columns:
                nueva_fila['nota'] = 'Fila creada/corregida manualmente'

            nuevas_filas.append(nueva_fila)

        if nuevas_filas:
            df_out = pd.concat(
                [df_out, pd.DataFrame(nuevas_filas)],
                ignore_index=True
            )

    return df_out


#### 2.3.3 Siembra y emergencia  5/5
- Método de siembra
- Fecha de siembra
- Cantidad de semillas sembradas
- Cantidad de plántulas emergidas
- Fecha de emergencia (cuando emerge el 70% de las plántulas)

In [180]:
# --- Método de siembra ---
# Como en la nueva campaña se tiene que especificar el recipiente, se creó una categoría nueva de almácigo genérico para 
# aquellas respuestas en campañas anteriores que no especifican el recipiente.
inspeccionar_valores(df, 'Método de siembra', momento='antes')
# Normalización de las respuestas
siembra_mapeo = { 
    # Almácigo genérico (no se detalló el contenedor)
    'siembra en almácigos': 'Almácigo (genérico)',
    'almácigo': 'Almácigo (genérico)',
    'almacigo': 'Almácigo (genérico)',
    'almacigos': 'Almácigo (genérico)',
    'almácigos': 'Almácigo (genérico)',
    'siembra en almágicos': 'Almácigo (genérico)',
    'almágics': 'Almácigo (genérico)',
    'plantines': 'Almácigo (genérico)',

    # Almácigo en bandeja
    'almacigo 72': 'Almácigo en bandeja',
    'almacigo de 72': 'Almácigo en bandeja',
    'bandeja 25 celdas': 'Almácigo en bandeja',
    'sembré en plantineras.': 'Almácigo en bandeja',
    'manual en bandejas': 'Almácigo en bandeja',
    'semillero': 'Almácigo en bandeja',

    # Almácigo en otro recipiente
    'ambiente controlado: tuppers y papel absorvente': 'Almácigo en otro recipiente',
    'platito': 'Almácigo en otro recipiente',
    'se sembraron en vasitos de yogures, dentro de casa.': 'Almácigo en otro recipiente',
    'germinación de las semillas en platito. tardaron mas que otras variedades en germinar, 6 dias. una vez germinadas las paso a una maceta del 12 con sustrato growmix profesional. realizo almacigos comunitarios en esas macetas. luego, cuando los plantines tienen unos 12 cm y sus hojas verdaderas hago un repique a bolsitas de cultivo del 8 o del 10 (con el mismo sustrato), de a 2 plantitas y ahi crecen hasta que tienen unos 30 cm y se van a la tierra.': 'Almácigo en otro recipiente',
    'germinación en servilleta': 'Almácigo en otro recipiente',
    'maceta': 'Almácigo en otro recipiente',

    # Siembra directa
    'siembra directa': 'Siembra directa',
    'directa': 'Siembra directa',

    'otro': 'Otro'
}

df_limpio = limpiar_variable(df_limpio, 'Método de siembra', limpiador_categorico(siembra_mapeo, no_mapeados='Otro'))


# --- Fecha de siembra ---
inspeccionar_valores(df, 'Fecha de siembra', momento='antes')
# 'noviembre' -> mediados de noviembre como estimación conservadora con margen de error mediano
df_limpio = limpiar_variable(df_limpio, 'Fecha de siembra', limpiador_fecha({'noviembre': '2022-11-15'}), 'fecha_siembra',
                             convertir=lambda s: pd.to_datetime(s, errors='coerce').dt.date)


# --- Cantidad de semillas sembradas ---
inspeccionar_valores(df, 'Cantidad de semillas sembradas', momento='antes')
mapeo_cant_sem_semb = {
    '10+1': 11,
    '06/10/2025': 10
}
df_limpio = mapear_variable_numerica(df_limpio, 'Cantidad de semillas sembradas', mapeo_cant_sem_semb, borrar_nulos=True)


# --- Cantidad de plántulas emergidas ---
# Se mapearon manualmente dentro de df_limpieza_manual_..., 4 valores del usuario AA017T al cruzarlo con semillas sembradas (0.9 --> 90).
inspeccionar_valores(df, 'Cantidad de plántulas emergidas', momento='antes')
df_limpio = mapear_variable_numerica(df_limpio, 'Cantidad de plántulas emergidas', borrar_nulos=True)


# --- Fecha de emergencia ---
# 1 nan se logró imputar, los otros 5 se borran
inspeccionar_valores(df, VAR_EMERGENCIA, momento='antes')
def limpiar_fecha_emergencia(row):
    val = str(row['valor']).strip().lower() if pd.notna(row['valor']) else ''

    # Buscamos en las posibles columnas donde haya quedado la nota histórica
    nota = str(row.get('nota', '')) + ' ' + str(row.get('comentario', ''))
    nota = nota.strip().lower()

    # Caso imputado: AA022T ("aprox 3 semanas" desde el 2024-11-01)
    if '3 semanas' in val or '3 semanas' in nota:
        return '2024-11-22'

    # Fechas interpretables
    if val.startswith('202') or val.startswith('201'):
        return val.upper()

    # Casos no interpretables 
    return None

df_limpio = limpiar_variable(df_limpio, VAR_EMERGENCIA, limpiar_fecha_emergencia, 'fecha_emergencia',
                             por_fila=True, borrar_nulos=True)


🔹 ANTES (datos originales)

=== Método de siembra ===
Valor: Siembra en almácigos, Frecuencia: 60
Valor: Almácigo, Frecuencia: 15
Valor: almácigo, Frecuencia: 9
Valor: Almacigo 72, Frecuencia: 4
Valor: plantines, Frecuencia: 3
Valor: Ambiente controlado: tuppers y papel absorvente, Frecuencia: 3
Valor: ALMACIGOS, Frecuencia: 3
Valor: Almácigos , Frecuencia: 3
Valor: almacigo, Frecuencia: 3
Valor: platito, Frecuencia: 3
Valor: Se sembraron en vasitos de yogures, dentro de casa., Frecuencia: 3
Valor: Germinación de las semillas en platito. tardaron mas que otras variedades en germinar, 6 dias. una vez germinadas las paso a una maceta del 12 con sustrato Growmix profesional. realizo almacigos comunitarios en esas macetas. Luego, cuando los plantines tienen unos 12 cm y sus hojas verdaderas hago un repique a bolsitas de cultivo del 8 o del 10 (con el mismo sustrato), de a 2 plantitas y ahi crecen hasta que tienen unos 30 cm y se van a la tierra., Frecuencia: 3
Valor: germinación en servil

#### 2.3.3 Trasplante a maceta — 7/9 con variables deprecadas: 9/9
- Ubicación del contenedor
- Profundidad del contenedor (cm)
- Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.)
- Fecha de trasplante (maceta)
- Cantidad de plantines trasplantados
- Cantidad de plantines implantados
- Cultivo asociado (maceta)
- Volumen: 57 --> ¿Variable deprecada? 
- #por contenedor: 62 --> ¿Variable deprecada? 

In [181]:
# --- Ubicación del contenedor ---
inspeccionar_valores(df, 'Ubicación del contenedor')

mapeo_ubicacion_contenedor = {
    'al aire libre': 'Al aire libre',
    'aire libre': 'Al aire libre',
    'Aire Iibre': 'Al aire libre',
    'en invernadero': 'En invernadero',
    'invernadero': 'En invernadero',

    'otro ambiente protegido': 'Otro ambiente protegido'
}
df_limpio = mapear_variable_texto(df_limpio, 'Ubicación del contenedor', mapeo_ubicacion_contenedor)


# --- Profundidad del contenedor (cm) ---
inspeccionar_valores(df, 'Profundidad del contenedor (cm)')

# Un evaluador escribió "Pase a tierra", lo que significa que no completó directamente esta sección de trasplante a maceta -> se elimina la fila
mascara_borrar = (df_limpio['variable_limpia'] == 'Profundidad del contenedor (cm)') & \
                 (df_limpio['valor'].astype(str).str.lower().str.strip() == 'pase a tierra')
df_limpio = df_limpio[~mascara_borrar].copy()

mapeo_profundidad = {
    'Balde de pintura': 35.0,
    'medio barril metalico ': 45.0,
    '32 y 37': 34.5  
}
df_limpio = limpiar_dimensiones_numericas(df_limpio, 'Profundidad del contenedor (cm)', mapeo_profundidad)


🔹 ANTES (datos originales)

=== Ubicación del contenedor ===
Valor: al aire libre, Frecuencia: 60
Valor: aire libre, Frecuencia: 54
Valor: en invernadero, Frecuencia: 25
Valor: invernadero, Frecuencia: 17
Valor: Al aire libre, Frecuencia: 6
Valor: Aire Iibre, Frecuencia: 4
Observaciones totales:  166

----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Ubicación del contenedor ===
Valor: Al aire libre, Frecuencia: 124
Valor: En invernadero, Frecuencia: 42
Observaciones totales:  166

🔹 ANTES (datos originales)

=== Profundidad del contenedor (cm) ===
Valor: 30, Frecuencia: 20
Valor: 10, Frecuencia: 14
Valor: 12, Frecuencia: 13
Valor: 50, Frecuencia: 8
Valor: 20, Frecuencia: 7
Valor: 7cm, Frecuencia: 6
Valor: 9, Frecuencia: 6
Valor: 15, Frecuencia: 6
Valor: 40, Frecuencia: 5
Valor: 26, Frecuencia: 5
Valor: 10 cm, Frecuencia: 5
Valor: 15 cm  macetas negras plasticas, 

In [182]:
# --- Sustrato utilizado (respuesta múltiple: se explota en una fila por componente) ---
# En esta etapa se realizan 2 mapeos distintos, ya que una primera instancia fue un mapeo con las categorías exactas de las respuestas
# de los evaluadores, pero luego tras dialogar con la dirección, se acordaron categorías más amplias. 
# No se borró el primer mapeo para mantener la trazabilidad del detalle exacto del dato en la columna 'nota'.
# Aclaración: Las mezclas sin tierra (ej. compost + perlita) entran igual en las categorías "tierra con...".
SUSTRATO_CATEGORIAS = {
    'Tierra / suelo': ['Tierra / suelo', 'Tierra negra'],
    'Materia orgánica y enmiendas': [   # procesada (compost, humus...) y sin procesar (restos, ceniza...)
        'Compost', 'Humus', 'Bocashi', 'Heces animal', 'Mantillo de monte', 'Ceniza', 'Hojas de árboles',
        'Ramas', 'Borra de café', 'Cáscara de huevo', 'Restos de frutas y verduras',
    ],
    'Minerales y aireantes': ['Perlita', 'Vermiculita', 'Arena', 'Turba'],
    'Sustrato comercial': ['Sustrato comercial'],
    'Otro': ['Otro', 'otro', 'Otros', 'otra', 'Otras'],   
}
SUSTRATO_A_CATEGORIA = {comp: cat for cat, comps in SUSTRATO_CATEGORIAS.items() for comp in comps}

CAT_TIERRA_SOLA = 'Suelo / tierra sola (del lugar o comprada, por ejemplo tierra negra)'
CAT_CON_MO      = 'Suelo / tierra con materia orgánica y enmiendas (procesadas y sin procesar)'
CAT_CON_MIN     = 'Suelo / tierra con minerales y aireantes'
CAT_COMERCIAL   = 'Sustrato comercial'
CAT_OTRO        = 'Otro'

inspeccionar_valores(df, VAR_SUSTRATO)

# Mapeo de espuestas complejas
sustrato_over = {
    'mezcla de tierra, compost y turba en proporciones iguales':
        ['Tierra / suelo', 'Compost', 'Turba'],
    'tierra del patio, ceniza, y tierra negra comprada':
        ['Tierra / suelo', 'Ceniza', 'Tierra negra'],
    'sustrato comercial grow y compost':
        ['Sustrato comercial', 'Compost'],
    'hojas de arboles, ramas':
        ['Hojas de árboles', 'Ramas'],
    'compost, suelo, bosta de vaca, ceniza de madera, mantullo de monte':
        ['Compost', 'Tierra / suelo', 'Heces animal', 'Ceniza', 'Mantillo de monte'],
    'compost, perlita, vermiculita, borra cafe, cascara huevo y banana molida':  
        ['Compost', 'Perlita', 'Vermiculita', 'Borra de café', 'Cáscara de huevo', 'Restos de frutas y verduras'],
    'tierra, compost, suelo, abono animal':
        ['Tierra / suelo', 'Compost', 'Heces animal'],
    'compost, perlita, bosta de caballo seca':
        ['Compost', 'Perlita', 'Heces animal'],
}

# Mapeo de respuestas simples
sustrato_mapeo = {
    'tierra': 'Tierra / suelo',
    'suelo': 'Tierra / suelo',
    'tierra / suelo': 'Tierra / suelo',
    'tierra negra': 'Tierra negra',
    'compost': 'Compost',
    'perlita': 'Perlita',
    'pelita': 'Perlita',        
    'sustrato comercial': 'Sustrato comercial',
    'bocashi': 'Bocashi',
    'arena': 'Arena',
    'vermiculita': 'Vermiculita',
    'turba': 'Turba',
    'ceniza': 'Ceniza',
    'humus': 'Humus',
    'borra de cafe': 'Borra de café',
    'borra cafe': 'Borra de café',
    'cascara de huevo': 'Cáscara de huevo',
    'cascara huevo': 'Cáscara de huevo',
    'banana molida': 'Restos de frutas y verduras',
    'mantillo de monte': 'Mantillo de monte',
    'ramas': 'Ramas',
    'hojas de arboles': 'Hojas de árboles',
    'otro': 'Otro',
    'guano de gallina': 'Heces animal',
    'guano gallina': 'Heces animal',
}

# Columnas que identifican "un mismo sustrato" (un participante/ensayo/etapa). 
# Agrupamos por esto porque en 2024-2025 el dataset ya viene explotado y "tierra sola" solo se puede decidir
# mirando todos los componentes juntos
COLS_GRUPO_SUSTRATO = ['campaña', 'id_participante', 'id_sitio', 'sub_ensayo', 'variedad_id', 'variedad_original', 'etapa']

def estandarizar_sustrato(valor_original):
    """Paso 1: texto original -> lista de componentes estandarizados (sin repetir)."""
    return list(dict.fromkeys(mapear_fila(valor_original, sustrato_over, sustrato_mapeo)))


def categorias_nuevas_sustrato(componentes):
    """Paso 2: componentes estandarizados de UN mismo sustrato -> {categoría nueva: [componentes que la generaron]}.
    En las categorías "con ..." se incluye también la tierra (si la hay) para que la nota quede completa."""
    def bloque(c):
        return c if c.startswith('SIN_MAPEAR') else SUSTRATO_A_CATEGORIA.get(c, f'SIN_MAPEAR:{c}')

    por_bloque = {}
    for c in componentes:
        por_bloque.setdefault(bloque(c), []).append(c)

    tierra = por_bloque.get('Tierra / suelo', [])
    cats = {}
    if 'Materia orgánica y enmiendas' in por_bloque:
        cats[CAT_CON_MO] = tierra + por_bloque['Materia orgánica y enmiendas']
    if 'Minerales y aireantes' in por_bloque:
        cats[CAT_CON_MIN] = tierra + por_bloque['Minerales y aireantes']
    if 'Sustrato comercial' in por_bloque: # Los Tierra + sustrato comercial quedan solamente como Sustrato comercial y no se genera una fila como Suelo / tierra sola
        cats[CAT_COMERCIAL] = por_bloque['Sustrato comercial']
    if 'Otro' in por_bloque:
        cats[CAT_OTRO] = por_bloque['Otro']
    # Tierra sola = tierra / suelo / tierra negra solamente
    if set(por_bloque) == {'Tierra / suelo'}:
        cats[CAT_TIERRA_SOLA] = tierra
    for b in por_bloque:
        if b.startswith('SIN_MAPEAR'):
            cats[b] = por_bloque[b]

    return cats


def reclasificar_sustrato(df, nombre_variable='sustrato'):
    """Agrupa las filas (ya explotadas, una por componente) por sustrato y genera una fila por categoría nueva.
    Deja en 'nota' los valores estandarizados (los de sustrato_mapeo) que dieron esa categoría,
    ej. 'Sustrato original: Tierra / suelo; Compost; Cáscara de huevo'.
    Al agrupar acá no quedan filas repetidas."""
    mask = (df['variable'] == nombre_variable).values
    cols = [c for c in COLS_GRUPO_SUSTRATO if c in df.columns]
    if 'nota' not in df.columns:
        df = df.assign(nota=pd.NA)

    nuevas = []
    for _, g in df[mask].groupby(cols, dropna=False, sort=False):
        componentes = list(dict.fromkeys(g['valor'].astype(str)))
        previas = [str(n).strip() for n in g['nota'].dropna().unique() if str(n).strip()]
        base = g.iloc[0].to_dict()
        for cat, originales in categorias_nuevas_sustrato(componentes).items():
            fila = dict(base)
            fila['valor'] = cat
            fila['nota'] = ' | '.join(previas + ['Sustrato original: ' + '; '.join(originales)])   # no pisa una nota previa
            nuevas.append(fila)

    print(f"Filas de '{nombre_variable}': {int(mask.sum())} (una por componente) → {len(nuevas)} (una por categoría)")
    return pd.concat([df[~mask], pd.DataFrame(nuevas, columns=df.columns)], ignore_index=True)


df_limpio = limpiar_variable(df_limpio, VAR_SUSTRATO, estandarizar_sustrato, 'sustrato',
                             explotar=True, mostrar=False)
df_limpio = reclasificar_sustrato(df_limpio, 'sustrato')

inspeccionar_valores(df_limpio, VAR_SUSTRATO)
sin = df_limpio.loc[df_limpio['variable'] == 'sustrato', 'valor'].astype(str).str.startswith('SIN_MAPEAR')
if sin.any():
    print(f"Quedaron {int(sin.sum())} valores SIN_MAPEAR")


🔹 ANTES (datos originales)

=== Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.) ===
Valor: Tierra, Frecuencia: 99
Valor: tierra, compost, Frecuencia: 19
Valor: tierra, compost, perlita, Frecuencia: 19
Valor: Compost, Frecuencia: 17
Valor: tierra, perlita, Frecuencia: 8
Valor: Perlita, Frecuencia: 7
Valor: tierra, compost, turba, suelo, Frecuencia: 6
Valor: tierra, compost, turba, Frecuencia: 5
Valor: Suelo, Frecuencia: 5
Valor: tierra, compost, perlita, suelo, Frecuencia: 5
Valor: tierra, compost, perlita, humus, Frecuencia: 4
Valor: compost, Frecuencia: 4
Valor: compost, perlita, turba, Frecuencia: 4
Valor: Bocashi, Frecuencia: 3
Valor: tierra, compost, arena, Frecuencia: 3
Valor: compost, suelo, bosta de vaca, ceniza de madera, mantullo de monte, Frecuencia: 3
Valor: tierra, perlita, turba, Frecuencia: 3
Valor: Mezcla de tierra, compost y turba en proporciones iguales., Frecuencia: 3
Valor: Arena, Frecuencia: 3
V

In [183]:
# --- Fecha de trasplante (maceta) ---
inspeccionar_valores(df, 'Fecha de trasplante (maceta)')

fechas_over = {
    'primeros dias de diciembre': '2024-12-04',
    '30/09/2023 y 09/10/2023': '2023-09-30'
}
# parsear=True: lo que no esté en el mapeo se parsea automáticamente a YYYY-MM-DD
df_limpio = limpiar_variable(df_limpio, 'Fecha de trasplante (maceta)', limpiador_fecha(fechas_over, parsear=True),
                             'fecha_trasplante_maceta')


# --- Cantidad de plantines trasplantados ---
inspeccionar_valores(df, 'Cantidad de plantines trasplantados')

plantines_over = {'4 y 3': 7}  
df_limpio = limpiar_variable(df_limpio, 'Cantidad de plantines trasplantados', limpiador_entero(plantines_over),
                             'cantidad_plantines_trasplantados')


# --- Cantidad de plantines implantados ---
inspeccionar_valores(df, 'Cantidad de plantines implantados')

# Se eliminan las filas no numéricas o negativas (los 0 se conservan)
df_limpio = limpiar_variable(df_limpio, 'Cantidad de plantines implantados', limpiador_entero(invalidos=None, minimo=0),
                             'cantidad_plantines_implantados', borrar_nulos=True)


🔹 ANTES (datos originales)

=== Fecha de trasplante (maceta) ===
Valor: 2022-11-30, Frecuencia: 8
Valor: 2022-11-28, Frecuencia: 6
Valor: 2022-11-12, Frecuencia: 5
Valor: 2022-10-29, Frecuencia: 5
Valor: 2023-10-13, Frecuencia: 5
Valor: 2023-10-05, Frecuencia: 5
Valor: 2024-10-18, Frecuencia: 5
Valor: 2022-11-05, Frecuencia: 4
Valor: 2022-11-18, Frecuencia: 4
Valor: 2024-09-23, Frecuencia: 3
Valor: 2023-09-26, Frecuencia: 3
Valor: 2023-09-08, Frecuencia: 3
Valor: 2022-10-13, Frecuencia: 3
Valor: 2022-12-01, Frecuencia: 3
Valor: 2023-09-28, Frecuencia: 3
Valor: 2023-11-17, Frecuencia: 3
Valor: 2022-12-25, Frecuencia: 3
Valor: 2023-09-21, Frecuencia: 3
Valor: 2025-10-25, Frecuencia: 3
Valor: 2023-12-23, Frecuencia: 2
Valor: 2024-11-11, Frecuencia: 2
Valor: 2022-11-15, Frecuencia: 2
Valor: 2022-10-01, Frecuencia: 2
Valor: 2023-09-30, Frecuencia: 2
Valor: 2023-11-07, Frecuencia: 2
Valor: 2023-10-31, Frecuencia: 2
Valor: 2023-09-10, Frecuencia: 2
Valor: 2022-11-10, Frecuencia: 2
Valor: 202

C:\Users\ariel\AppData\Local\Temp\ipykernel_11792\2677611060.py:64: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(v, dayfirst=True).strftime('%Y-%m-%d')


In [184]:
# --- Cultivo asociado (maceta) ---
VAR_CULTIVO_MACETA = 'Cultivo asociado (maceta)'
CULTIVO_CATEGORIAS = {
    'Frutas y hortalizas': [
        'Lechuga', 'Acelga', 'Rúcula', 'Repollo', 'Kale', 'Apio', 'Puerro', 'Cebolla',
        'Cebolla de verdeo / Cebollín', 'Ajo', 'Zapallo', 'Zapallito veronés', 'Pimiento (morrón)',
        'Ají (chile)', 'Berenjena', 'Tomate', 'Remolacha', 'Zanahoria', 'Yacón', 'Habas', 'Arvejas',
        'Chaucha', 'Melón', 'Sandía', 'Frutilla',   
    ],
    'Aromáticas y medicinales': [
        'Albahaca', 'Orégano', 'Romero', 'Menta', 'Menta marina', 'Hierbabuena', 'Cilantro', 'Perejil',
        'Ciboulette', 'Stevia', 'Melisa', 'Eneldo', 'Hinojo', 'Paico (epazote)', 'Borraja', 'Consuelda',
        'Burrito', 'Vetiver', 'Monarda',
        'Aromáticas (genérico)',
    ],
    'Flores y ornamentales': [
        'Caléndula', 'Tagetes (copete)', 'Tagetes / Copete', 'Taco de reina', 'Girasol', 'Cineraria',
        'Petunia', 'Zinnia', 'Conejito (boca de dragón)',
        'Flores (genérico)',
    ],
    'No hubo': ['Ninguno', 'No hubo'],  
    'Otro': ['Otros', 'otro', 'Otras', 'Otra'],
}
CULTIVO_A_CATEGORIA = {cultivo: cat for cat, cultivos in CULTIVO_CATEGORIAS.items() for cultivo in cultivos}

def agrupar_cultivos(nombres):
    """Lista de cultivos estandarizados -> lista de categorías sin repetir (si hay 2 aromáticas queda 1 fila).
    Lo que no esté en CULTIVO_A_CATEGORIA queda como 'SIN_MAPEAR:<nombre>' para revisarlo (no se manda a 'Otros')."""
    categorias = []
    for n in nombres:
        if str(n).startswith('SIN_MAPEAR'):
            categorias.append(n)
        else:
            categorias.append(CULTIVO_A_CATEGORIA.get(n, f'SIN_MAPEAR:{n}'))
    return list(dict.fromkeys(categorias))

CAT_FRUTAS_HORTALIZAS = 'Frutas y hortalizas'
CAT_AROMATICAS_MEDICINALES = 'Aromáticas y medicinales'
CAT_FLORES_ORNAMENTALES = 'Flores y ornamentales'
CAT_NO_HUBO = 'No hubo'
CAT_OTRO_CULTIVO = 'Otro'

inspeccionar_valores(df, VAR_CULTIVO_MACETA)

# Lista de respuestas que equivalen a no haber asociado cultivo
SIN_ASOCIACION = {
    'ninguno', 'no', 'no se asoció en el almácigo', 
    'no hubo asociacion en el almàcigo', 'otras variedades en huerta'
}

# Respuestas compuestas irregulares
CULTIVOS_OVER = {
    'apio albahaca': ['Apio', 'Albahaca'],
    'hierba buena acelga': ['Hierbabuena', 'Acelga'],
    'hierba buena': ['Hierbabuena'],
    'borraja, acelga,, zapallito verones, calendula, puerro': [
        'Borraja', 'Acelga', 'Zapallito veronés', 'Caléndula', 'Puerro'
    ]
}

# Diccionario de equivalencias individuales
CULTIVOS_MAPEO = {
    'albahaca': 'Albahaca',
    'tagetes': 'Tagetes / Copete',
    'copete': 'Tagetes / Copete',
    'caléndulas': 'Caléndula',
    'calendula': 'Caléndula',
    'stevia': 'Stevia',
    'lechuga': 'Lechuga',
    'puerro': 'Puerro',
    'frutillas': 'Frutilla',
    'orégano': 'Orégano',
    'cilantro': 'Cilantro',
    'perejil': 'Perejil',
    'ciboulette': 'Ciboulette',
    'acelga': 'Acelga',
    'borraja': 'Borraja',
    'zapallito verones': 'Zapallito veronés'
}

def procesar_cultivo_asociado(valor):
    if pd.isna(valor):
        return ['Ninguno']
    
    norm = str(valor).strip().lower()
    norm = re.sub(r',+', ',', norm).strip()
    
    # Si se indica que no hubo asociación, asigna 'Ninguno'
    if norm in SIN_ASOCIACION:
        return ['Ninguno']
    
    if norm in CULTIVOS_OVER:
        return CULTIVOS_OVER[norm]
    
    partes = norm.split(',')
    resultado = []
    for p in partes:
        p = p.strip()
        if not p:
            continue
        cat = CULTIVOS_MAPEO.get(p, f'SIN_MAPEAR:{p}')
        resultado.append(cat)
        
    return list(dict.fromkeys(resultado))

# procesar_cultivo_asociado estandariza -> agrupar_cultivos lo pasa a las categorías grandes
df_limpio = limpiar_variable(df_limpio, VAR_CULTIVO_MACETA, lambda v: agrupar_cultivos(procesar_cultivo_asociado(v)),
                             'cultivo_asociado_maceta', explotar=True)
print(f"Filas en 'Otro': {(df_limpio.loc[df_limpio['variable'] == 'cultivo_asociado_maceta', 'valor'] == 'Otro').sum()}")


🔹 ANTES (datos originales)

=== Cultivo asociado (maceta) ===
Valor: albahaca, Frecuencia: 15
Valor: ninguno, Frecuencia: 6
Valor: Albahaca, Frecuencia: 4
Valor: no, Frecuencia: 3
Valor: albahaca, tagetes, Frecuencia: 3
Valor: albahaca, caléndulas, Frecuencia: 3
Valor: stevia, lechuga, Frecuencia: 3
Valor: Ninguno, Frecuencia: 3
Valor: Ninguno , Frecuencia: 3
Valor: puerro, lechuga, Frecuencia: 2
Valor: No se asoció en el almácigo, Frecuencia: 2
Valor: apio albahaca, Frecuencia: 2
Valor: lechuga, frutillas, Frecuencia: 2
Valor: orégano, Frecuencia: 1
Valor: Cilantro, Frecuencia: 1
Valor: albahaca, copete, Frecuencia: 1
Valor: no hubo asociacion en el almàcigo, Frecuencia: 1
Valor: perejil, ciboulette, Frecuencia: 1
Valor: orégano, ciboulette, Frecuencia: 1
Valor: hierba buena acelga, Frecuencia: 1
Valor: hierba buena , Frecuencia: 1
Valor: Otras variedades en huerta, Frecuencia: 1
Valor: No, Frecuencia: 1
Valor: borraja, acelga,, zapallito verones, calendula, puerro, Frecuencia: 1
Obs

#### 2.3.4 Trasplante a suelo — 8/12 con variables deprecadas: 12/12
- Ubicación de la parcela
- Superficie de la parcela (m²) 
- Fecha de trasplante (suelo)
- Cantidad de plantas trasplantadas a la parcela
- Densidad (plantas/m²)
- Cobertura de suelo
- Cultivo asociado (suelo)
- Cantidad de plantas implantadas
- Surcos: 53 --> ¿Variable deprecada? 
- Distancia surcos: 51 --> ¿Variable deprecada? 
- Distancia plantas: 59 --> ¿Variable deprecada? 
- Cultivo antecesor: 56 --> ¿Variable deprecada? 

Para esta etapa vamos a corregir primero datos desfasados o completados en las variables equivocadas.

Estos casos nacen de que el evaluador rompió el excel y el modelo de IA no logró interpretar las respuestas correctamente o simplemente el evaluador completo un campo incorrectamente.

In [185]:
CASOS_A_CORREGIR = [
    # Caso 1: Desfasaje de filas para AA009T (V56) en 2024-2025
    {
        'filtros': {
            'campaña': '2024-2025', 
            'id_participante': 'AA009T', 
            'id_sitio': 'AA009T',
            'variedad_id': 'V56'},
        'correcciones': {
            'Cantidad de plantas implantadas': (3, 'Trasplante a suelo'),
            VAR_DENSIDAD: 0.125,
            'Cobertura de suelo': 'sin cobertura',
            'Cultivo asociado (suelo)': 'lechugas y remolachas'
        }
    },
    # Caso 2: Desfasaje de filas para AA009T (V172) en 2024-2025
    {
        'filtros': {'campaña': '2024-2025', 'id_participante': 'AA009T', 'variedad_id': 'V172'},
        'correcciones': {
            'Cantidad de plantas implantadas': 2,
            VAR_DENSIDAD: 0.083,
            'Cobertura de suelo': 'sin cobertura',
            'Cultivo asociado (suelo)': 'lechugas y remolachas'
        }
    },
    # Caso 3: Corrección por nota de mortandad y espacio físico para AA022T (V9)
    {
        'filtros': {'campaña': '2024-2025', 'id_participante': 'AA022T', 'variedad_id': 'V9'},
        'correcciones': {
            'Cantidad de plantas implantadas': 2,
            'Superficie de la parcela (m²)': 24.0
        }
    },
    # Caso 4: Superficie del recinto global para AA022T (V55 y V56) observaciones agregadas
    {
        'filtros': {'campaña': '2024-2025', 'id_participante': 'AA022T', 'variedad_id': 'V55'},
        'correcciones': {'Superficie de la parcela (m²)': (24.0, 'Trasplante a suelo')}
    },
    {
        'filtros': {'campaña': '2024-2025', 'id_participante': 'AA022T', 'variedad_id': 'V56'},
        'correcciones': {'Superficie de la parcela (m²)': (24.0, 'Trasplante a suelo')}
    }
]

df_limpio, df = aplicar_correcciones_manuales(df_limpio, df, CASOS_A_CORREGIR)

--- ESTADO INICIAL (ANTES) ---
Filas en df_limpio : 10345
Filas en df        : 10300

--- AUDITORÍA DE CAMBIOS (DESPUÉS) ---
Filas agregadas netas en df_limpio : 2
Filas agregadas netas en df        : 2

📌 1. FILAS TOTALMENTE NUEVAS CREADAS (2):
  campaña id_participante variedad_id              etapa               variable_limpia valor
2024-2025          AA022T         V55 Trasplante a suelo Superficie de la parcela (m²)  24.0
2024-2025          AA022T         V56 Trasplante a suelo Superficie de la parcela (m²)  24.0


2. FILAS EXISTENTES QUE FUERON MODIFICADAS (10):
  campaña id_participante variedad_id                 variable_limpia                                                                                     valor_ANTES              etapa         valor_DESPUES
2024-2025          AA009T         V56           Densidad (plantas/m²)                                                                                               4 Trasplante a suelo                 0.125
2024-2025 

In [186]:
# --- Ubicación de la parcela ---
inspeccionar_valores(df, 'Ubicación de la parcela')

ubicacion_mapeo = {
    'al aire libre': 'Al aire libre',
    'aire libre': 'Al aire libre',
    'en invernadero': 'En invernadero',
    'invernadero': 'En invernadero',

    'otro ambiente protegido': 'Otro ambiente protegido'
}
df_limpio = limpiar_variable(df_limpio, 'Ubicación de la parcela', limpiador_categorico(ubicacion_mapeo), 'ubicacion_parcela')


# --- Superficie de la parcela (m²) ---
inspeccionar_valores(df, 'Superficie de la parcela (m²)')
mapeo_superficie_parcial = {
    'bancales de 4 x1 mt': 4.0,
    '01/02/2025': 1.2,
    '30 cm': np.nan,  # No se pudo recuperar --> Se borra
    '5m2': 5.0,
    '6 x 5 mts': 30.0,
    '4m2': 4.0,
    '2m2': 2.0
}
df_limpio = mapear_variable_numerica(df_limpio, 'Superficie de la parcela (m²)', mapeo_superficie_parcial,
                                     tipo='float', borrar_nulos=True)


# --- Fecha de trasplante (suelo) ---
inspeccionar_valores(df, 'Fecha de trasplante (suelo)')
# Solo se estandariza el nombre
df_limpio = limpiar_variable(df_limpio, 'Fecha de trasplante (suelo)', lambda v: str(v).strip(), 'fecha_trasplante_suelo')


# --- Cantidad de plantas trasplantadas a la parcela ---
inspeccionar_valores(df, 'Cantidad de plantas trasplantadas a la parcela')
plantas_over = {'0.1': 1}  # error de tipeo del evaluador
df_limpio = limpiar_variable(df_limpio, 'Cantidad de plantas trasplantadas a la parcela', limpiador_entero(plantas_over),
                             'cantidad_plantas_trasplantadas_parcela')


# --- Cobertura de suelo ---
inspeccionar_valores(df, 'Cobertura de suelo')
mapeo_cobertura = {
    'cobertura vegetal': 'Cobertura vegetal (hojas secas o restos vegetales)',
    'vegetal': 'Cobertura vegetal (hojas secas o restos vegetales)',
    'sin cobertura': 'Sin cobertura',
    'sin': 'Sin cobertura',
    'plástica': 'Cobertura plástica, cartón u otra similar',
    'otro': 'Otra'
}
df_limpio = mapear_variable_texto(df_limpio, 'Cobertura de suelo', mapeo_cobertura)


# --- Cantidad de plantas implantadas ---
inspeccionar_valores(df, 'Cantidad de plantas implantadas')
mapeo_implantadas = {
    '1 de cada una': 1,
    '1 en cada balde': 2,  # Actualizado gracias a cruzar con las trasplantadas
    '350': 1,              # El error del 350 lo bajamos a 1, coincidiendo con las trasplantadas
    350: 1
}
df_limpio = mapear_variable_numerica(df_limpio, 'Cantidad de plantas implantadas', mapeo_implantadas)


🔹 ANTES (datos originales)

=== Ubicación de la parcela ===
Valor: al aire libre, Frecuencia: 105
Valor: aire libre, Frecuencia: 56
Valor: en invernadero, Frecuencia: 16
Valor: invernadero, Frecuencia: 11
Observaciones totales:  188

----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Ubicación de la parcela ===
Valor: Al aire libre, Frecuencia: 161
Valor: En invernadero, Frecuencia: 27
Observaciones totales:  188

🔹 ANTES (datos originales)

=== Superficie de la parcela (m²) ===
Valor: 2, Frecuencia: 17
Valor: 1, Frecuencia: 16
Valor: 20, Frecuencia: 11
Valor: 5, Frecuencia: 9
Valor: 15, Frecuencia: 6
Valor: 24, Frecuencia: 6
Valor: 280, Frecuencia: 5
Valor: 50, Frecuencia: 5
Valor: 10, Frecuencia: 5
Valor: 6, Frecuencia: 4
Valor: bancales de 4 x1 mt, Frecuencia: 3
Valor: 24.0, Frecuencia: 3
Valor: 01/02/2025, Frecuencia: 3
Valor: 4.5, Frecuencia: 3
Valor: 370, Fre

##### Notas de la revisión de la variable Densidad (plantas/m²)

Se revisaron y auditaron las inconsistencias de densidad registradas por participante y campaña. 
Algunos datos se pudieron recuperar, otros se eliminaron (#VALUE!) y se recalcularon casos ambiguos manteniendo la coherencia entre la superficie declarada y las plantas reales.

1. Campaña 2023-2024
- Participante AA052T: Se recuperó una observación con #VALUE! calculando su densidad real en $0.5\text{ plantas/m}^2$ ($1\text{ planta} / 2\text{ m}^2$).
- Participante H014T: Se borraron $3$ registros con error #VALUE! debido a inconsistencias en la superficie declarada.
- Participante AA017T: Se ajustaron mediciones a $0.20\text{ plantas/m}^2$ ($10\text{ plantas} / 50\text{ m}^2$).

2. Campaña 2024-2025Participante 
- AA017T: Se corrigió el valor fijo original ($4$) recalculando la densidad matemática real a $6.75$ ($27 / 4\text{ m}^2$) para la variedad V172, y a $7.20$ ($36 / 5\text{ m}^2$) para las variedades V56, V7 y VFCAYF.
- Participante AA006T: Se corrigió el valor de planilla ($4$) recalculando a $0.25\text{ plantas/m}^2$ ($1\text{ planta} / 4\text{ m}^2$).
- Participante AA008T: Se recalcularon las densidades reales por superficie de bancal ($4\text{ m}^2$): V7 en $3.25$ ($13 / 4\text{ m}^2$), V56 en $4.25$ ($17 / 4\text{ m}^2$) y V172 en $1.25$ ($5 / 4\text{ m}^2$). 

3. Campaña 2025-2026
- Participante AB013T: Se unificó y recalculó la densidad real en $3.33\text{ plantas/m}^2$ (tenía registrado erróneamente $0.3$).
- Participantes AA005T y AB007T: Se ajustaron densidades muy bajas por parcelas de gran superficie ($280\text{ m}^2$ y $600\text{ m}^2$), conservando precisión entre $0.00357$ y $0.018\text{ plantas/m}^2$.
- Participante AA023T: Se corrigió la densidad de V56 a $0.90\text{ plantas/m}^2$ ($9\text{ plantas} / 10\text{ m}^2$).
- Participante AA006T: Se recalculó el valor a $0.20\text{ plantas/m}^2$ ($1\text{ planta} / 5\text{ m}^2$).
- Participantes en maceta / espacio reducido (ej. AA021T): Se desestimaron o ajustaron por no corresponder a un marco de plantación a campo.

*Criterio de Formateo Global:* Se estandarizaron los resultados generales a $2$ decimales, preservando hasta $3$ o $5$ decimales únicamente en casos con valores exactos (.125, .083) o parcelas de gran tamaño.

In [187]:
# --- Densidad (plantas/m²) ---
inspeccionar_valores(df, VAR_DENSIDAD)

def parsear_densidad_smart(val):
    """
    Formatea y redondea valores flotantes de densidad:
    - #VALUE! o NaN -> np.nan
    - Menores a 0.01 -> conserva hasta 5 decimales (ej. parcelas gigantes de 280m² o 600m²)
    - Valores con 3 decimales exactos (.125, .083, .375, .625, .875) -> conserva 3 decimales
    - Resto general -> redondea a 2 decimales
    """
    if pd.isna(val):
        return np.nan
    
    val_str = str(val).strip()
    if '#VALUE!' in val_str or 'NaN' in val_str:
        return np.nan
        
    try:
        num = float(val_str.replace(',', '.'))
        
        # Densidades muy bajas por parcelas gigantes (ej. 1/280m² o 1/600m²)
        if 0 < num < 0.01:
            return round(num, 5)
            
        # Resguardar precisión de tres decimales exactos
        if round(num, 3) in [0.125, 0.083, 0.375, 0.625, 0.875]:
            return round(num, 3)
            
        # Formateo estándar
        return round(num, 2)
    except ValueError:
        return np.nan


# Correcciones / recálculos manuales por participante (campaña, participante, variedades o None = todas, densidad)
CORRECCIONES_DENSIDAD = [
    # --- 2023-2024 ---
    ('2023-2024', 'AA017T', None, 0.2),                         # 10 plantas / 50 m²
    ('2023-2024', 'AA052T', None, 0.5),                         # recuperación de #VALUE!: 1 planta / 2.0 m²
    # --- 2024-2025 ---
    ('2024-2025', 'AA017T', ['V172'], 6.75),                    # 27 plantas / 4 m²
    ('2024-2025', 'AA017T', ['V56', 'V7', 'VFCAYF'], 7.2),      # 36 plantas / 5 m²
    ('2024-2025', 'AA006T', ['V7'], 0.25),                      # 1 planta / 4 m²
    ('2024-2025', 'AA008T', ['V7'], 3.25),                      # recálculos por bancal de 4 m²
    ('2024-2025', 'AA008T', ['V56'], 4.25),
    ('2024-2025', 'AA008T', ['V172'], 1.25),
    # --- 2025-2026 ---
    ('2025-2026', 'AB013T', None, 3.33),                        # 7/2.1 = 1/0.3 = 6/1.8 = 3.33
    ('2025-2026', 'AA005T', ['V113'], 0.00357),                 # parcela de 280 m²
    ('2025-2026', 'AA005T', ['V56'], 0.00357),
    ('2025-2026', 'AA005T', ['VCARC'], 0.018),
    ('2025-2026', 'AB007T', ['VCARC'], 0.018),                  # parcela de 600 m²
    ('2025-2026', 'AB007T', ['V113'], 0.0083),
    ('2025-2026', 'AB007T', ['V56'], 0.0067),
    ('2025-2026', 'AA023T', ['V56'], 0.9),                      # 9 plantas / 10 m²
    ('2025-2026', 'AA006T', ['VCARC', 'V113'], 0.2),            # 1 planta / 5 m²
]
for campana, participante, variedades, densidad in CORRECCIONES_DENSIDAD:
    filtros = {'campaña': campana, 'id_participante': participante}
    if variedades:
        filtros['variedad_id'] = variedades
    asignar_valor(df_limpio, VAR_DENSIDAD, densidad, **filtros)

df_limpio = limpiar_variable(df_limpio, VAR_DENSIDAD, parsear_densidad_smart, borrar_nulos=True)


🔹 ANTES (datos originales)

=== Densidad (plantas/m²) ===
Valor: 3, Frecuencia: 13
Valor: 4, Frecuencia: 11
Valor: 2, Frecuencia: 10
Valor: 5, Frecuencia: 8
Valor: 8.33, Frecuencia: 6
Valor: 6.25, Frecuencia: 4
Valor: 1, Frecuencia: 4
Valor: #VALUE!, Frecuencia: 4
Valor: 6.67, Frecuencia: 3
Valor: 2.78, Frecuencia: 3
Valor: 4.55, Frecuencia: 3
Valor: 50, Frecuencia: 3
Valor: 0.2, Frecuencia: 3
Valor: 0.1666666667, Frecuencia: 3
Valor: 0.3, Frecuencia: 3
Valor: 16.67, Frecuencia: 3
Valor: 133.33, Frecuencia: 3
Valor: 10, Frecuencia: 2
Valor: 0.6666666667, Frecuencia: 2
Valor: 33.33, Frecuencia: 2
Valor: 1.25, Frecuencia: 2
Valor: 0.03, Frecuencia: 2
Valor: 0.0035714285714285713, Frecuencia: 2
Valor: 2.5, Frecuencia: 1
Valor: 0.13, Frecuencia: 1
Valor: 1.5, Frecuencia: 1
Valor: 12, Frecuencia: 1
Valor: 1.555555556, Frecuencia: 1
Valor: 7.12, Frecuencia: 1
Valor: 8.68, Frecuencia: 1
Valor: 3.111111111, Frecuencia: 1
Valor: 2.22, Frecuencia: 1
Valor: 3.57, Frecuencia: 1
Valor: 2.6, Frecue

In [188]:
# --- Cultivo asociado (suelo) ---
# Utiliza las mismas categorías definidas en cultivo asociado (maceta)
VAR_CULTIVO_SUELO = 'Cultivo asociado (suelo)'
inspeccionar_valores(df, VAR_CULTIVO_SUELO)

def extraer_cultivos(row):
    val = row['valor']
    sitio = str(row.get('id_sitio', '')).strip()
    
    if pd.isna(val):
        return []
        
    val_str = str(val).strip().lower()
    
    # 1. Descartar explícitos y excepciones
    if val_str in ['-', 'ninguno', 'no', 'paja']:
        return ['No hubo']
        
    if val_str == 'idem': # 1 solo participante lo uso, así que se puede reemplazar directamente
        val_str = 'caléndula, albahaca, aromáticas'
        
    if val_str == '2': # Se logró recuperar de la planilla original porque este evaluador tuvo un desfasaje en sus respuestas
        if sitio == 'AB013T-Manzanares':
            val_str = 'rúcula, caléndula'
        else:
            return [] # Se elimina la fila de UNSAM
            
    if 'balde de pintura' in val_str:
        return []   # describe el sustrato (puesto por error en esta variable), no dice nada de cultivos asociados: se elimina la fila

    cultivos = set()
    
    # --- Zapallo vs Zapallito veronés ---
    if 'zapallito' in val_str:
        cultivos.add('Zapallito veronés')
    val_sin_zapallito = val_str.replace('zapallito', '')
    if 'zapallo' in val_sin_zapallito:
        cultivos.add('Zapallo')
        
    # --- Menta marina vs Menta ---
    if 'menta marina' in val_str:
        cultivos.add('Menta marina')
    val_sin_menta = val_str.replace('menta marina', '')
    if 'menta' in val_sin_menta:
        cultivos.add('Menta')
        
    # --- Cebolla de verdeo / Cebollín vs Cebolla ---
    if 'verdeo' in val_str or 'cebollín' in val_str or 'cebollin' in val_str:
        cultivos.add('Cebolla de verdeo / Cebollín')
    val_sin_verdeo = (
        val_str.replace('cebolla de verdeo', '')
               .replace('verdeo', '')
               .replace('cebollín', '')
               .replace('cebollin', '')
    )
    if 'cebolla' in val_sin_verdeo or 'cebollas' in val_sin_verdeo:
        cultivos.add('Cebolla')

    # --- Pimiento (morrón) vs Ají (chile) ---
    if 'pimiento' in val_str or 'morron' in val_str or 'morrón' in val_str:
        cultivos.add('Pimiento (morrón)')
    if 'ají' in val_str or 'aji' in val_str or 'ajies' in val_str:
        cultivos.add('Ají (chile)')

    # --- Hortalizas de hoja, fruto y raíz ---
    if 'albahaca' in val_str: cultivos.add('Albahaca')
    if 'lechuga' in val_str: cultivos.add('Lechuga')
    if 'acelga' in val_str: cultivos.add('Acelga')
    if 'rucula' in val_str or 'rúcula' in val_str: cultivos.add('Rúcula')
    if 'puerro' in val_str: cultivos.add('Puerro')
    if 'oregano' in val_str or 'orégano' in val_str: cultivos.add('Orégano')
    if 'romero' in val_str: cultivos.add('Romero')
    if 'cilantro' in val_str: cultivos.add('Cilantro')
    if 'hinojo' in val_str: cultivos.add('Hinojo')
    if 'burrito' in val_str: cultivos.add('Burrito')
    if 'vetiver' in val_str: cultivos.add('Vetiver')
    if 'prejil' in val_str or 'perejil' in val_str: cultivos.add('Perejil')
    if 'paico' in val_str: cultivos.add('Paico (epazote)')
    if 'eneldo' in val_str: cultivos.add('Eneldo')
    if 'apio' in val_str: cultivos.add('Apio')
    if 'melisa' in val_str: cultivos.add('Melisa')
    if 'kale' in val_str: cultivos.add('Kale')
    if 'borraja' in val_str: cultivos.add('Borraja')
    if 'repollo' in val_str: cultivos.add('Repollo')
    if 'berenjena' in val_str: cultivos.add('Berenjena')
    if 'melon' in val_str or 'melón' in val_str: cultivos.add('Melón')
    if 'sandia' in val_str or 'sandía' in val_str: cultivos.add('Sandía')
    if 'habas' in val_str: cultivos.add('Habas')
    if 'arveja' in val_str or 'arvejas' in val_str: cultivos.add('Arvejas')
    if 'chaucha' in val_str: cultivos.add('Chaucha')
    if 'ajo' in val_str: cultivos.add('Ajo')
    if 'remolacha' in val_str: cultivos.add('Remolacha')
    if 'yacon' in val_str or 'yacón' in val_str: cultivos.add('Yacón')
    if 'zanahoria' in val_str: cultivos.add('Zanahoria')
    if 'tomate' in val_str: cultivos.add('Tomate')
    if 'ciboulette' in val_str: cultivos.add('Ciboulette')
    if 'hierbabuena' in val_str: cultivos.add('Hierbabuena')
    if 'frutilla' in val_str: cultivos.add('Frutilla')

    # --- Flores / Ornamentales ---
    if 'calendula' in val_str or 'caléndula' in val_str: cultivos.add('Caléndula')
    if 'copete' in val_str or 'tagete' in val_str: cultivos.add('Tagetes (copete)')
    if 'taco' in val_str or 'taconde' in val_str: cultivos.add('Taco de reina')
    if 'consuelda' in val_str: cultivos.add('Consuelda')
    if 'girasol' in val_str: cultivos.add('Girasol')
    if 'cineraria' in val_str: cultivos.add('Cineraria')
    if 'petuña' in val_str or 'petunia' in val_str: cultivos.add('Petunia')
    if 'monarda' in val_str: cultivos.add('Monarda')
    if 'zinnia' in val_str: cultivos.add('Zinnia')
    if 'conejito' in val_str or 'boca de dragón' in val_str or 'boca de dragon' in val_str:
        cultivos.add('Conejito (boca de dragón)')
    if 'stevia' in val_str: cultivos.add('Stevia')

    # Menciones genéricas ("aromáticas", "flores", "florales ornamentales"): se suman aunque haya cultivos específicos.
    if re.search(r'\baromatic|\baromátic', val_str): cultivos.add('Aromáticas (genérico)')
    if re.search(r'\bflor(es|ales)?\b|\bornamental', val_str): cultivos.add('Flores (genérico)')

    # Lo que no se reconoce no se manda a 'No hubo' en silencio: queda marcado para revisarlo.
    if not cultivos:
        cultivos.add(f'SIN_MAPEAR:{val_str}')

    return sorted(cultivos)

df_limpio = limpiar_variable(df_limpio, VAR_CULTIVO_SUELO, lambda row: agrupar_cultivos(extraer_cultivos(row)),
                             'cultivo_asociado_suelo', por_fila=True, explotar=True)
print(f"Filas en 'Otro': {(df_limpio.loc[df_limpio['variable'] == 'cultivo_asociado_suelo', 'valor'] == 'Otro').sum()}")



🔹 ANTES (datos originales)

=== Cultivo asociado (suelo) ===
Valor: albahaca, Frecuencia: 19
Valor: 2, Frecuencia: 6
Valor: albahaca , Frecuencia: 6
Valor: -, Frecuencia: 5
Valor: zapallo, Frecuencia: 4
Valor: albahaca y copetes, Frecuencia: 3
Valor: paja, Frecuencia: 3
Valor: Ninguno, Frecuencia: 3
Valor: lechuga, Frecuencia: 3
Valor: flores y albahaca, Frecuencia: 3
Valor: lechugas y remolachas, Frecuencia: 3
Valor: ají, Frecuencia: 3
Valor: albahaca, menta, Frecuencia: 3
Valor: Ninguno , Frecuencia: 3
Valor: copetes, zinnias, Frecuencia: 2
Valor: oregano, romero, flores (petuña, conejitos y calendulas), Frecuencia: 2
Valor: Cebolla, Ajo, Frecuencia: 2
Valor: oregano y calendula, Frecuencia: 2
Valor: Sandias, melones, rucula, consuelda, Frecuencia: 2
Valor: ninguno, Frecuencia: 2
Valor: pimiento, Frecuencia: 2
Valor: lechuga, berenjena, albahaca, florales ornamentales, menta, cineraria, zapallo, girasol, Frecuencia: 2
Valor: aromáticas, Frecuencia: 2
Valor: albahaca, lechuga, Frecue

#### 2.3.5 Labores — 7/7
- Desbrote
- Tutorado
- Conducción
- Fertilización 
- Control de malezas
- Riego
- Control de plagas

In [189]:
# --- Desbrote ---
inspeccionar_valores(df, 'Desbrote')
# 1 solo participante uso 'idem', así que se puede parsear directamente a sus otras respuestas (Si)
df_limpio = limpiar_variable(df_limpio, 'Desbrote', limpiador_categorico({**SI_NO, 'idem': 'Sí'}), 'desbrote') 


# --- Tutorado ---
inspeccionar_valores(df, 'Tutorado')
df_limpio = limpiar_variable(df_limpio, 'Tutorado', limpiador_categorico(SI_NO), 'tutorado')


# --- Conducción ---
inspeccionar_valores(df, 'Conducción')
conduccion_mapeo = {
    'sin conducción': 'Sin conducción',
    'un tallo': 'A un tallo',
    'a un tallo': 'A un tallo',
    'a dos tallos': 'A dos tallos',
    'dos tallos': 'A dos tallos',
    'preparados biodinámicos': None,  # Error de carga del evaluador
    'otra': 'Otra'
}
df_limpio = limpiar_variable(df_limpio, 'Conducción', limpiador_categorico(conduccion_mapeo), 'conduccion', mostrar=False)
df_limpio = df_limpio.dropna(subset=['valor']).reset_index(drop=True)
inspeccionar_valores(df_limpio, 'Conducción')


# --- Fertilización (respuesta múltiple) ---
inspeccionar_valores(df, 'Fertilización')
# Mapeo de respuestas compuestas (se explotan) y simples
override_fertilizacion = {
    'compost y biopreparados': [
        'Compost, lombricompuesto, humus o bokashi', 
        'Abonos líquidos o biopreparados'
    ],
    'compost, humus, biopreparados': [
        'Compost, lombricompuesto, humus o bokashi', 
        'Abonos líquidos o biopreparados'
    ]
}
mapeo_comp_fertilizacion = {
    'compost': 'Compost, lombricompuesto, humus o bokashi',
    'bocashi': 'Compost, lombricompuesto, humus o bokashi',
    'biopreparados': 'Abonos líquidos o biopreparados',
    'preparados biodinamicos': 'Abonos líquidos o biopreparados',
    'sin fertilizacion': 'No utilicé',
    'abono verde': 'Abono verde',
    'fertilizantes sinteticos': 'Fertilizantes químicos o comerciales',

    'combine más de una opción': 'Combine más de una opción',
    'otro': 'Otra',
    'otra': 'Otra'
}
df_limpio = limpiar_variable(df_limpio, 'Fertilización',
                             partial(mapear_fila, overrides=override_fertilizacion, mapeo_componentes=mapeo_comp_fertilizacion),
                             'Fertilización', explotar=True)


# --- Control de malezas ---
inspeccionar_valores(df, 'Control de malezas')
malezas_mapeo = {
    'sin control': 'No realicé control',
    'control mecánico': 'Retiro manual o con herramientas',
    'control manual': 'Retiro manual o con herramientas',
    'manual': 'Retiro manual o con herramientas',
    'Control Mecánico': 'Retiro manual o con herramientas',
    'herbicidas sintéticos': 'Herbicida químico o comercial',

    'cobertura u otro método físico': 'Cobertura u otro método físico',
    'otro': 'Otro'
}
df_limpio = limpiar_variable(df_limpio, 'Control de malezas', limpiador_categorico(malezas_mapeo), 'control_malezas')


# --- Riego ---
inspeccionar_valores(df, 'Riego')
riego_mapeo = {
    'manguera/regadera': 'Manual con regadera o manguera',
    'con manguera/regadera': 'Manual con regadera o manguera',
    'manguera/regadora': 'Manual con regadera o manguera',
    'con manguera y una botella con agua enterrada': 'Manual con regadera o manguera',
    'por goteo': 'Goteo',
    'goteo': 'Goteo',
    'por surco': 'Surco',
    'surco': 'Surco',
    'por microaspersión': 'Aspersión / microaspersión',
    'microaspersión': 'Aspersión / microaspersión',
    'por aspersión': 'Aspersión / microaspersión',

    'secano / solo lluvia': 'Secano / solo lluvia',
    'otro': 'Otro'
}
df_limpio = limpiar_variable(df_limpio, 'Riego', limpiador_categorico(riego_mapeo), 'riego')


# --- Control de plagas ---
inspeccionar_valores(df, 'Control de plagas')
plagas_mapeo = {
    'sin control': 'No realicé control',
    'control mecánico': 'Retiro manual',
    'manualmente': 'Retiro manual',
    'biopreparados': 'Biopreparados',
    'preparados biodinámicos': 'Biopreparados',
    'plaguicidas sintéticos': 'Producto comercial de síntesis',

    'trampas o barreras físicas': 'Trampas o barreras físicas',
    'control biológico': 'Control biológico',
    'combine más de una opción': 'Combine más de una opción',
    'otro': 'Otro',
}
df_limpio = limpiar_variable(df_limpio, 'Control de plagas', limpiador_categorico(plagas_mapeo), 'control_plagas')


🔹 ANTES (datos originales)

=== Desbrote ===
Valor: sí, Frecuencia: 180
Valor: no, Frecuencia: 54
Valor: Sí, Frecuencia: 4
Valor: idem, Frecuencia: 2
Valor: Si, Frecuencia: 1
Observaciones totales:  241

----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Desbrote ===
Valor: Sí, Frecuencia: 187
Valor: No, Frecuencia: 54
Observaciones totales:  241

🔹 ANTES (datos originales)

=== Tutorado ===
Valor: sí, Frecuencia: 228
Valor: no, Frecuencia: 15
Valor: Sí, Frecuencia: 4
Observaciones totales:  247

----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Tutorado ===
Valor: Sí, Frecuencia: 232
Valor: No, Frecuencia: 15
Observaciones totales:  247

🔹 ANTES (datos originales)

=== Conducción ===
Valor: sin conducción, Frecuencia: 54
Valor: un tallo, Frecu

#### 2.3.6 Crecimiento y desarrollo — 7/8 con variables deprecadas: 8/8
- Fecha de primera floración (día en la que al menos la mitad de las plantas están en su primera floración)
- Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante)
- Tolerancia a la quemadura del sol (1 = nada tolerante, 5 = muy tolerante)
- Sanidad (1 = nada sana, 5 = muy sana)
- Muerte por virus rugoso del tomate: 106 --> ¿Variable deprecada? 
- Uniformidad (1 = nada uniforme, 5 = muy uniforme)
- ¿A qué crees que se debe la uniformidad observada?
- Plagas observadas

In [190]:
# --- Fecha de primera floración ---
inspeccionar_valores(df, VAR_FLORACION)
# Mapeo de fechas aproximadas
fechas_aproximadas = {
    'fines de octubre': '2024-10-28',
    'principios de noviembre': '2023-11-05',
    'diciembre': '2022-12-15',
    'ultimos dias de diciembre 2024': '2024-12-28',
    'primeros dias de enero 2025': '2025-01-05'
}
df_limpio = limpiar_variable(df_limpio, VAR_FLORACION,
                             limpiador_fecha(fechas_aproximadas, nulos=['nan', 'none', '', 'no registré', 'np registré']),
                             'fecha_primera_floracion', borrar_nulos=True)


🔹 ANTES (datos originales)

=== Fecha de primera floración (día en la que al menos la mitad de las plantas están en su primera floración) ===
Valor: 2023-01-06, Frecuencia: 6
Valor: 2023-11-12, Frecuencia: 6
Valor: 2022-12-20, Frecuencia: 6
Valor: nan, Frecuencia: 4
Valor: 2022-11-19, Frecuencia: 4
Valor: 2022-12-10, Frecuencia: 4
Valor: fines de octubre, Frecuencia: 4
Valor: 2023-01-20, Frecuencia: 4
Valor: 2023-01-15, Frecuencia: 3
Valor: 2022-12-01, Frecuencia: 3
Valor: 2022-11-20, Frecuencia: 3
Valor: 2024-11-20, Frecuencia: 3
Valor: 2024-11-15, Frecuencia: 3
Valor: 2022-12-26, Frecuencia: 3
Valor: 2023-11-15, Frecuencia: 3
Valor: 2023-12-15, Frecuencia: 3
Valor: 2025-12-30, Frecuencia: 3
Valor: 2025-10-20, Frecuencia: 3
Valor: 2025-11-18, Frecuencia: 3
Valor: 2026-01-05, Frecuencia: 2
Valor: 2023-02-02, Frecuencia: 2
Valor: 2022-11-22, Frecuencia: 2
Valor: 2024-10-27, Frecuencia: 2
Valor: 2022-12-25, Frecuencia: 2
Valor: 2023-11-30, Frecuencia: 2
Valor: 2022-11-15, Frecuencia: 2


In [191]:
# --- Variables en escala 1 a 5: Tolerancia al estrés hídrico / a la quemadura del sol, Sanidad, Uniformidad ---
variables_escala = [VAR_ESTRES_HIDRICO, VAR_SOL, VAR_SANIDAD, VAR_UNIFORMIDAD]

def corregir_valor_escala(val):
    if pd.isna(val):
        return val
        
    val_str = str(val).strip()
    
    # Recuperamos decimales .5 parseados erróneamente como fechas
    match = re.match(r'^0?([1-4])/0?5/\d{2,4}$', val_str)
    if match:
        return float(f"{match.group(1)}.5")
    
    try:
        return float(val_str.replace(',', '.'))
    except ValueError:
        return val

for var in variables_escala:
    inspeccionar_valores(df, var)
    df_limpio = limpiar_variable(df_limpio, var, corregir_valor_escala)


🔹 ANTES (datos originales)

=== Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante) ===
Valor: 4, Frecuencia: 64
Valor: 3, Frecuencia: 51
Valor: 5, Frecuencia: 43
Valor: 2.5, Frecuencia: 16
Valor: 3.5, Frecuencia: 12
Valor: 2, Frecuencia: 6
Valor: 1, Frecuencia: 6
Valor: 03/05/2024, Frecuencia: 4
Valor: 4.5, Frecuencia: 3
Valor: 02/05/2024, Frecuencia: 3
Valor: 04/05/2024, Frecuencia: 2
Valor: 02/05/2023, Frecuencia: 2
Valor: 04/05/2023, Frecuencia: 1
Valor: 03/05/2023, Frecuencia: 1
Valor: 1.5, Frecuencia: 1
Valor: 01/05/2024, Frecuencia: 1
Observaciones totales:  216

----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante) ===
Valor: 4.0, Frecuencia: 64
Valor: 3.0, Frecuencia: 51
Valor: 5.0, Frecuencia: 43
Valor: 2.5, Frecuencia: 21
Valor: 3.5, Frecuencia: 17
Valor: 4.5, Frecuencia: 

In [192]:
# --- ¿CUÁL CREES QUE ES LA CAUSA DE LA DESUNIFORMIDAD OBSERVADA? ---
inspeccionar_valores(df, VAR_CAUSAS)

# Se realizan los mapeos en base a la causa que nombra la persona cruzandolo con el puntaje en la escala numérica de uniformidad (1-5).
# Si el puntaje es mayor a 3 se considera que habla de uniformidad y se establece como Null el valor (para tener causas realmente de desuniformidad).
#   - algo que hizo distinto entre plantas (riego, poda, tutorado, nutrición)      → CAT_DIF_MANEJO
#   - algo del lugar que no es manejo (sol/sombra, suelo, maceta)                  → CAT_DIF_AMBIENTE
#   - desuniformidad real, sin causa de manejo ni del lugar (semillas, "mismas condiciones") → CAT_DESUNIF_MISMAS_COND
#   - el texto explica UNIFORMIDAD o no explica nada                               → None
CAT_NO_DESUNIF = 'No hubo desuniformidad'
CAT_DIF_MANEJO = 'Diferencias en el manejo (riego, labores, nutrición)'  
CAT_DIF_AMBIENTE = 'Diferencias en el terreno, el contenedor o la luz' 
CAT_DESUNIF_MISMAS_COND = 'Hubo desuniformidad, aunque todas tuvieron las mismas condiciones'
CAT_POCAS_PLANTAS = 'Pocas plantas para evaluarlo (1 o 2)'

mapeo_uniformidad_detallado = {
    # Respuestas que no son válidas para las campañas. Pasan a Null.
    'genética': None,  # (Freq: 5)
    'Genética': None,  # (Freq: 1)
    'mejoramiento': None,  # (Freq: 3)
    'Características logradas y conservadas en la variedad, es evidente que son plantas muy vitales y resistentes y eso influye en su uniformidad.': None,  # (Freq: 3)
    'Salvo la diferencia de plantas con cherry chico y cherry más grande, crecieron todas con características vegetativas similares, produjeron de manera consistente y constante a lo largo de la temporada.': None,  # (Freq: 1)
    'Parece ser el tipo de tomate, Los frutos que crecen dentro de la bolsa de tul (asegurando autopolinización) son sumamente uniformes.': None,  # (Freq: 2)

    # Respuestas válidas, pero que explican UNIFORMIDAD y no la nueva pregunta (campaña 2026). Pasan a Null.
    'por su naturaleza y alimentación': None,  # (Freq: 3), puso 5 en uniformidad
    'Similares características y lugar de cultivo': None,  # (Freq: 3), puso 4
    'las temperaturas son mas amigables ,por ahora,  esperemos a Enero hay mas humedad ambiente, tengo abundante melisa, hierba buena ,menta marina, capuchina y otras flores.': None,  # (Freq: 5), puso 5
    'A las condiciones similares': None,  # (Freq: 2), puso 5
    'Las plantas se asolean bien parejo': None,  # (Freq: 1), puso 5
    'Tuvimos lluvias muy fuertes y días de mucho calor y las plantas están perfectas y los tomates tienen una forma linda y muy consistente, en todos los casos': None,  # (Freq: 1), puso 5
    'Quizás porque toleró mejor el estrés general, ya sea hidrico o del ambiente': None,  # (Freq: 1), puso 5
    'A las tareas de enturado, eliminacion de brote apical y control de maleza': None,  # (Freq: 1), puso 4.5, explica uniformidad
    'Constancia en riego y jabón potásico semanal, más el recorte de las ramas no principales': None,  # (Freq: 1), puso 4, explica uniformidad
    'buen manejo del cultivo': None,  # (Freq: 1), puso 4.5, explica uniformidad
    'vitalidad y sanidad de la planta': None,  # (Freq: 1), puso 4
    'fueron frutos chicos de entre 140 a 150 gramos, plantas debiles y enfermas': None,  # (Freq: 1), puso 5
    'en su mayoria todas tienen un mismo crecimiento,en cuanto a tamaño y sanidad.': None,  # (Freq: 1), puso 5
    'a que estan en la misma maceta las 5': None,  # (Freq: 1). puso 3.5, es ambiguo así que queda como None.

    # CAT_DIF_MANEJO
    'A que esta vez no estuve muy pendiente de poda ni a sacar chupones.': CAT_DIF_MANEJO,  # (Freq: 3)
    'quizás no estaba bien tutorada': CAT_DIF_MANEJO,  # (Freq: 3)
    'podría ser al cultivo': CAT_DIF_MANEJO,  # (Freq: 1)
    'deficiencia nutricional': CAT_DIF_MANEJO,  # (Freq: 1)

    # CAT_DIF_AMBIENTE (incluye luz/orientación/sombreo y contenedor)
    'Por la orientación algunas plantas reciben menos sol directo que otras.': CAT_DIF_AMBIENTE,  # (Freq: 2)
    'Algunas plantas reciben menos sol, por sombreo de otros cultivos': CAT_DIF_AMBIENTE,  # (Freq: 1)
    'Tal vez el contenedor no fue el adecuado': CAT_DIF_AMBIENTE,  # (Freq: 3)

    # CAT_POCAS_PLANTAS
    '(tuve poca cantidad para poder evaluar su uniformidad)': CAT_POCAS_PLANTAS,  # (Freq: 1)

    # CAT_NO_DESUNIF: no hay respuestas de campañas anteriores que correspondan

    # CAT_DESUNIF_MISMAS_COND: desuniformidad real cuya causa no es una diferencia de manejo ni del ambiente
    'No puedo definirlo, una planta detuvo su crecimiento, no prosperan todas las flores, sus frutos crecen hermosos pero pocos. estan bajo las mismas condiciones.': CAT_DESUNIF_MISMAS_COND,  # (Freq: 1)
    'podría deberse a un lote poco uniforme de semillas ya que desde el comienzo el crecimiento fue disparejo': CAT_DESUNIF_MISMAS_COND,  # (Freq: 1) 
    'Misma Zona, igual riego': CAT_DESUNIF_MISMAS_COND,  # (Freq: 1), puso 3.

    # Sin causa declarada o comentarios que no son correctos para esta variable → Null 
    'Desconozco': None,  # (Freq: 3)
    'no lo sé': None,  # (Freq: 2)
    'frutos muy desparejos en cuanto tamaño y deformes, como si tuvieran otro tomatito en la punta.': None,  # síntoma de desuniformidad sin causa → Null
    'una más frondosa que la otra': None,  # síntoma de desuniformidad sin causa → Null
    'algunas plantas son fuertes y grandes pero sin flores ni frutos, otras pequeñas, como determinadas pero llenas de frutos': None,  # síntoma de desuniformidad sin causa → Null 
    'frutos muy chiquitos en general.': None,  # describe tamaño de frutos, ni causa ni desuniformidad → Null
    'Frutos de entre 6 a 7 gramos': None,  # describe tamaño de frutos, ni causa ni desuniformidad → Null
    'Tamaño de frutos medianos para la variedad 150 gramos de promedio': None,  # describe tamaño de frutos, ni causa ni desuniformidad → Null
    'las chinches': None,  # (Freq: 1) 
    'a las enfermedades  hongos principalemente despues arañuelas': None,  # (Freq: 2)
}

mask_causas = df_limpio['variable_limpia'] == VAR_CAUSAS
df_limpio.loc[mask_causas, 'nota'] = 'Texto original: ' + df_limpio.loc[mask_causas, 'valor'].astype(str).str.strip()
df_limpio = mapear_variable_texto(df_limpio, VAR_CAUSAS, mapeo_uniformidad_detallado)

mask_nulos = mask_causas & (
    df_limpio['valor'].isna() | df_limpio['valor'].isin(['None', 'nan']) 
)
df_limpio = df_limpio[~mask_nulos].reset_index(drop=True)
print(f"Filas en Null descartadas: {int(mask_nulos.sum())}")


🔹 ANTES (datos originales)

=== ¿A qué crees que se debe la uniformidad observada? ===
Valor: genética, Frecuencia: 5
Valor: las temperaturas son mas amigables ,por ahora,  esperemos a Enero hay mas humedad ambiente, tengo abundante melisa, hierba buena ,menta marina, capuchina y otras flores., Frecuencia: 5
Valor: Tal vez el contenedor no fue el adecuado , Frecuencia: 3
Valor: A que esta vez no estuve muy pendiente de poda ni a sacar chupones., Frecuencia: 3
Valor: Características logradas y conservadas en la variedad, es evidente que son plantas muy vitales y resistentes y eso influye en su uniformidad., Frecuencia: 3
Valor: por su naturaleza y alimentación, Frecuencia: 3
Valor: quizás no estaba bien tutorada, Frecuencia: 3
Valor: Desconozco , Frecuencia: 3
Valor: mejoramiento, Frecuencia: 3
Valor: Similares características y lugar de cultivo, Frecuencia: 3
Valor: Por la orientación algunas plantas reciben menos sol directo que otras.	, Frecuencia: 2
Valor: a las enfermedades  hongo

In [193]:
# --- Plagas observadas → sanidad_observada ---
inspeccionar_valores(df, 'Plagas observadas')

# --- PLAGAS OBSERVADAS → SE TRANSFORMA EN SANIDAD_OBSERVADA ---
CAT_PERFECTO_ESTADO = 'No se observó ningún problema'
CAT_BICHOS_CHICOS   = 'Bichitos chicos (pulgones, arañuelas, chinches, etc.)'
CAT_BICHOS_GRANDES  = 'Bichos grandes (orugas, babosas, langostas, etc.)'
CAT_MANCHAS         = 'Manchas en hojas o tallos'
CAT_AMARILLAS       = 'Hojas amarillas, enruladas o deformadas'
CAT_MORDIDAS_HOJAS  = 'Hojas con caminitos, agujeros o mordidas'
CAT_MARCHITAS       = 'Plantas marchitas aunque tengan agua'
CAT_POLVILLO        = 'Polvillo o manchas blancas sobre las hojas'
CAT_FRUTOS          = 'Frutos con manchas, podridos o con la punta negra'
CAT_HERBIVOROS      = 'Mordeduras de mamíferos o picoteos de aves en frutos u hojas'

# Sumar CAT_MORDIDAS_HOJAS solo cuando además nombran los daños
KW_DANIO_HOJA = ['mordi', 'agujero', 'comid', 'comieron', 'devor', 'daño', 'danio', 'caminito']

# "Ninguna plaga" en Plagas observadas NO implica "ningún problema de sanidad", por eso va a null
SIN_DATO = [None]

def normalizar_plagas(val):
    """Devuelve una lista de tuplas (categoria_nueva, valor_viejo).
    - categoria_nueva: va a 'valor'.
    - valor_viejo: etiqueta(s) del código original (Chinches, Hormigas, etc.), va a 'nota'.
    """
    if pd.isna(val):
        return SIN_DATO

    val_str = str(val).strip().lower()

    ninguna_exacto = [
        'ninguna', 'no', 'ninguna de momento',
        'sin plaga de insectos. no se observaron mordiduras en los cherry. desconozco si alguno fue comido por estos mamíferos, pero no observé ausencia de tomates en los racimos bajos de la planta.'
    ]
    if val_str in ninguna_exacto:
        return SIN_DATO

    # categoria_nueva -> set de etiquetas viejas que la generaron
    hallazgos = {}
    def agregar(nueva, vieja):
        hallazgos.setdefault(nueva, set()).add(vieja)

    # Bichos chicos (por tamaño)
    if 'chinche' in val_str or 'chiche' in val_str:
        agregar(CAT_BICHOS_CHICOS, 'Chinches')

    if 'arañuel' in val_str and 'sospecha' not in val_str and 'deficiencia' not in val_str:
        agregar(CAT_BICHOS_CHICOS, 'Arañuela / Ácaros')

    if any(k in val_str for k in ['pulgón', 'pulgon', 'bichitos amarillos']):
        agregar(CAT_BICHOS_CHICOS, 'Pulgones')

    if 'cochinilla' in val_str or 'bicho bolita' in val_str:
        agregar(CAT_BICHOS_CHICOS, 'Cochinilla')

    if 'mosca blanca' in val_str:
        agregar(CAT_BICHOS_CHICOS, 'Mosca blanca')

    if 'hormiga' in val_str:
        agregar(CAT_BICHOS_CHICOS, 'Hormigas')

    # Bichos grandes
    grandes = []
    if 'gusano' in val_str:
        grandes.append('Gusanos')
    if 'babosa' in val_str or 'caracol' in val_str:
        grandes.append('Babosas y caracoles')
    if 'langosta' in val_str or 'tucura' in val_str:
        grandes.append('Langostas / Tucuras')

    for vieja in grandes:
        agregar(CAT_BICHOS_GRANDES, vieja)
        # Solo si el texto habla de daño en hojas se suma también la categoría de mordidas
        if any(k in val_str for k in KW_DANIO_HOJA):
            agregar(CAT_MORDIDAS_HOJAS, vieja)

    # Hongos / enfermedades: se reparte según el síntoma que se pueda inferir del texto
    sospechas = ['sospecha de hongos que podría haber sido deficiencia', 'posible cancro bacteriano']
    sospecha_descartada = any(sospecha in val_str for sospecha in sospechas)
    if not sospecha_descartada:
        if 'pelusa blanca' in val_str:
            agregar(CAT_POLVILLO, 'Hongos / Enfermedades')
        if 'cancro' in val_str:
            agregar(CAT_MANCHAS, 'Hongos / Enfermedades')
        if 'hongo' in val_str:
            agregar(CAT_POLVILLO, 'Hongos / Enfermedades')

    # Enfermedad virósica → hojas amarillas / enruladas / deformadas
    if 'virósica' in val_str or 'virosica' in val_str:
        agregar(CAT_AMARILLAS, 'Enfermedad virósica')

    # Herbívoros (mamíferos).
    if any(k in val_str for k in ['cuyes', 'liebre', 'herbivoría']):
        agregar(CAT_HERBIVOROS, 'Herbívoros (mamíferos)')

    # Sin coincidencias (incluye deficiencias o sospechas descartadas) → NULL
    if not hallazgos:
        return SIN_DATO

    return [(nueva, '; '.join(sorted(viejas))) for nueva, viejas in sorted(hallazgos.items())]

df_limpio = limpiar_variable(df_limpio, 'Plagas observadas', normalizar_plagas,
                             explotar=True, mostrar=False)

# Separamos la tupla: categoría nueva → 'valor'; valor viejo → 'nota' (sin pisar una nota previa)
mask_plagas = df_limpio['variable_limpia'] == 'Plagas observadas'

nuevas    = df_limpio.loc[mask_plagas, 'valor'].map(lambda t: t[0])
originales = df_limpio.loc[mask_plagas, 'valor'].map(lambda t: t[1])
previas   = df_limpio.loc[mask_plagas, 'nota']

df_limpio.loc[mask_plagas, 'nota'] = [
    (f"{p} | " if pd.notna(p) and str(p).strip() else '') + f"Plaga original: {o}"
    for p, o in zip(previas, originales)
]
df_limpio.loc[mask_plagas, 'valor'] = nuevas

inspeccionar_valores(df_limpio, 'Plagas observadas')



🔹 ANTES (datos originales)

=== Plagas observadas ===
Valor: ninguna, Frecuencia: 8
Valor: Babosas, Frecuencia: 5
Valor: caracoles, Frecuencia: 4
Valor: Ninguna de momento, Frecuencia: 3
Valor: Chinches (similar a Leptoglossus zonatus?), Frecuencia: 3
Valor: hormigas/chinches, Frecuencia: 3
Valor: hormigas, Frecuencia: 3
Valor: chinche, Frecuencia: 3
Valor: cochinilla, chinches, arañuela, hongos, Frecuencia: 3
Valor: Arañuela, chinches, Frecuencia: 3
Valor: Babosas, gusanos, Frecuencia: 3
Valor: Arañuela y gusano, Frecuencia: 3
Valor: Arañuelas, Frecuencia: 2
Valor: Arañuela Roja, Frecuencia: 2
Valor: Hongos, Frecuencia: 2
Valor: chinche del tomate y chinche común, Frecuencia: 2
Valor: Mosca blanca, pero no plaga, Frecuencia: 2
Valor: pulgon moro, Frecuencia: 2
Valor: no, Frecuencia: 2
Valor: Sospecha de hongos que podría haber sido deficiencia nutricional (cuando agregamos compost mejoraron y no hubo contagio a pesar de haber muchas plantas "hongueadas"). Además hubo mucha chinche ve

#### 2.3.7 Labores durante fructificación — 2/2
- - - Raleo de frutos: 182
- - - Eliminación del brote apical: 219

In [194]:
# --- Raleo de frutos ---
inspeccionar_valores(df, 'Raleo de frutos')
df_limpio = limpiar_variable(df_limpio, 'Raleo de frutos', limpiador_categorico(SI_NO), 'raleo_frutos')


# --- Eliminación del brote apical ---
inspeccionar_valores(df, 'Eliminación del brote apical')
df_limpio = limpiar_variable(df_limpio, 'Eliminación del brote apical', limpiador_categorico(SI_NO), 'eliminacion_brote_apical')


🔹 ANTES (datos originales)

=== Raleo de frutos ===
Valor: no, Frecuencia: 161
Valor: sí, Frecuencia: 21
Observaciones totales:  182

----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Raleo de frutos ===
Valor: No, Frecuencia: 161
Valor: Sí, Frecuencia: 21
Observaciones totales:  182

🔹 ANTES (datos originales)

=== Eliminación del brote apical ===
Valor: no, Frecuencia: 138
Valor: sí, Frecuencia: 78
Valor: si, Frecuencia: 3
Observaciones totales:  219

----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Eliminación del brote apical ===
Valor: No, Frecuencia: 138
Valor: Sí, Frecuencia: 81
Observaciones totales:  219


#### 2.3.8 Cosecha y poscosecha — 20/20
- Fecha de cosecha #1 a #10: 148, 106, 73, 48, 41, 34, 23, 20, 17, 14 --> Se dejó solo fecha_primera_cosecha y fecha_ultima_cosecha. Usuario AA017T se mapeo manualmente una de sus última fechas de cosecha de la campaña 2024-2025 que escribió erróneamente.
    - Fechas de cosecha 4, 5, 6, 9 y 10 --> Limpias desde el original
    - Fechas de cosecha 1, 2, 3, 7 y 8 --> Se eliminaron 3, 2, 2, 1, 1 observaciones (500g, enero , enero y febrero, nan)

- Color del fruto al momento de cosechar: 197 --> Se definió que no se van a tomar colores como tal, sino fracciones representando el estado de maduración al momento de cosechar.
- Duración poscosecha (días que dura una muestra de 10 tomates en estado para ser consumido): 100 --> 193, se explotaron a 2 variables distintas (los días y si se mantuvo en heladera, afuera o no se especificó). 
- Cantidad de frutos cosechados: 67 --> 81, se eliminaron 4 observaciones (x frutos por planta y nan), se imputo 1 y se agregaron 13 observaciones de un participante que lo había puesto en las variables de "Rendimiento X".
- Peso de todos los frutos cosechados (KG): 57 --> 83, se eliminaron observaciones que decían peso promedio por planta (sin saber cuantas plantas tuvieron no se pudó obtener el total real) y otras 2 observaciones que completaron pero en los comentarios pusieron que no hubo cosecha porque estaban verdes los tomates. Se agregaron 5 observaciones de un participante que se rescataron de otra variable. Se sumaron 28 observaciones que se rescataron de las variables "Rendimiento X" de la campaña 2022-2023.
- Rendimiento 1,2,3,4,5,6 --> Se analizaron sus valores y algunas observaciones fueron pasadas a "Peso de todos los frutos cosechados (KG)" otras a "Cantidad de frutos cosechados" y otras se eliminaron (1)


In [195]:
# --- Fechas de cosecha #1 a #10 -> solo se guardan la primera y la última de cada ensayo ---
inspeccionar_valores(df, 'Fecha de cosecha #1')

vars_cosecha = [f'Fecha de cosecha #{i}' for i in range(1, 11)]
mask_fechas = df_limpio['variable_limpia'].isin(vars_cosecha)

df_fechas = df_limpio[mask_fechas].copy()
df_fechas['valor_dt'] = pd.to_datetime(df_fechas['valor'], errors='coerce')
df_fechas = df_fechas.dropna(subset=['valor_dt'])

# Cálculo de la fecha mínima y máxima por ensayo y variedad
cols_agrupacion = ['campaña', 'id_participante', 'id_sitio','sub_ensayo', 'variedad_id']
resumen_fechas = (
    df_fechas
    .groupby(cols_agrupacion)['valor_dt']
    .agg(min='min', max='max')
    .reset_index()
)

# Selección de todas las demás columnas (etapa, nota, variable_norm, etc.) 
cols_metadatos = [c for c in df_limpio.columns
                  if c not in cols_agrupacion + ['variable', 'variable_limpia', 'valor']]

# Tomamos el primer registro existente de cada ensayo para arrastrar sus metadatos reales
metadatos = (
    df_fechas[cols_agrupacion + cols_metadatos]
    .drop_duplicates(subset=cols_agrupacion)
)

resumen_fechas = pd.merge(resumen_fechas, metadatos, on=cols_agrupacion, how='left')

def armar_filas_consolidadas(df_base, col_fecha, nombre_variable):
    df_sub = df_base.copy()
    
    df_sub['valor'] = df_sub[col_fecha].dt.strftime('%Y-%m-%d')
    
    df_sub['variable'] = nombre_variable
    df_sub['variable_limpia'] = nombre_variable
    
    return df_sub[df_limpio.columns]

# Generación de las filas para primera y última cosecha
df_primera = armar_filas_consolidadas(resumen_fechas, 'min', 'fecha_primera_cosecha')
df_ultima = armar_filas_consolidadas(resumen_fechas, 'max', 'fecha_ultima_cosecha')

df_fechas_consolidadas = pd.concat([df_primera, df_ultima], ignore_index=True)

# Eliminación de las 10 fechas viejas e inserción de las 2 consolidadas
df_limpio = df_limpio[~mask_fechas].copy()
df_limpio = pd.concat([df_limpio, df_fechas_consolidadas], ignore_index=True)


🔹 ANTES (datos originales)

=== Fecha de cosecha #1 ===
Valor: 2023-02-09, Frecuencia: 5
Valor: 2023-01-25, Frecuencia: 4
Valor: 2023-02-01, Frecuencia: 3
Valor: 2023-01-08, Frecuencia: 3
Valor: 2026-02-01, Frecuencia: 3
Valor: 2022-12-18, Frecuencia: 3
Valor: 2026-01-27, Frecuencia: 2
Valor: 2026-01-28, Frecuencia: 2
Valor: 2026-01-20, Frecuencia: 2
Valor: 2026-01-19, Frecuencia: 2
Valor: 2026-01-29, Frecuencia: 2
Valor: 2025-01-10, Frecuencia: 2
Valor: 2023-03-12, Frecuencia: 2
Valor: 2023-02-10, Frecuencia: 2
Valor: 2022-12-23, Frecuencia: 2
Valor: 2024-01-08, Frecuencia: 2
Valor: 2023-12-06, Frecuencia: 2
Valor: 2026-01-08, Frecuencia: 2
Valor: 2024-02-23, Frecuencia: 2
Valor: 2024-02-09, Frecuencia: 2
Valor: 2023-02-08, Frecuencia: 2
Valor: 2023-02-15, Frecuencia: 2
Valor: 2023-02-28, Frecuencia: 2
Valor: 2023-01-31, Frecuencia: 2
Valor: 2023-01-27, Frecuencia: 2
Valor: 2024-12-23, Frecuencia: 2
Valor: 2025-01-18, Frecuencia: 2
Valor: 2026-01-02, Frecuencia: 2
Valor: 2024-02-27, 

In [196]:
# --- Verificación de la consolidación de fechas ---
# Conteo de registros por tipo de fecha consolidada
print("=== 1. Frecuencias de variables consolidadas ===")
print(df_limpio['variable_limpia'].value_counts().loc[['fecha_primera_cosecha', 'fecha_ultima_cosecha']])

fechas_viejas = [f'Fecha de cosecha #{i}' for i in range(1, 11)]
resto_viejas = df_limpio['variable_limpia'].isin(fechas_viejas).sum()
print(f"\n=== 2. Fechas intermedias originales restantes (debe ser 0): {resto_viejas} ===")

print("\n=== 3. Muestra de filas consolidadas ===")
cols_revision = ['campaña', 'id_participante', 'variedad_id', 'variable', 'variable_limpia', 'valor', 'variable_norm', 'nota']
print(
    df_limpio[df_limpio['variable_limpia'].isin(['fecha_primera_cosecha', 'fecha_ultima_cosecha'])]
    [cols_revision]
    .head(10)
    .to_string()
)

=== 1. Frecuencias de variables consolidadas ===
variable_limpia
fecha_primera_cosecha    146
fecha_ultima_cosecha     146
Name: count, dtype: int64

=== 2. Fechas intermedias originales restantes (debe ser 0): 0 ===

=== 3. Muestra de filas consolidadas ===
        campaña id_participante variedad_id               variable        variable_limpia       valor variable_norm  nota
9816  2022-2023          AA009T        V115  fecha_primera_cosecha  fecha_primera_cosecha  2023-03-03     1ra fecha  None
9817  2022-2023          AA009T         V16  fecha_primera_cosecha  fecha_primera_cosecha  2023-02-19     1ra fecha  None
9818  2022-2023          AA009T        V172  fecha_primera_cosecha  fecha_primera_cosecha  2023-02-14     1ra fecha  None
9819  2022-2023          AA009T         V18  fecha_primera_cosecha  fecha_primera_cosecha  2023-02-08     1ra fecha  None
9820  2022-2023          AA009T         V63  fecha_primera_cosecha  fecha_primera_cosecha  2023-02-09     1ra fecha  None
9821  202

In [197]:
# --- Color del fruto al momento de cosechar ---
inspeccionar_valores(df, 'Color del fruto al momento de cosechar')

mapeo_color = {
    # Categoría: Verde
    'completamente verde': 'Completamente verde',
    '0': 'Completamente verde',
    '0.0': 'Completamente verde',

    # Categorías: 0.5 y 0.75
    'a mitad de su coloracion': 'A mitad de su coloración',
    'a mitad de su coloración': 'A mitad de su coloración',
    '0.5': 'A mitad de su coloración',
    'a 3/4 de su coloración': 'A 3/4 de su coloración',
    '0.75': 'A 3/4 de su coloración',
    
    # Categoría: 100% 
    'a la totalidad de su coloración': 'A la totalidad de su coloración',
    '1': 'A la totalidad de su coloración',
    '1.0': 'A la totalidad de su coloración'
}

df_limpio = limpiar_variable(df_limpio, 'Color del fruto al momento de cosechar',
                             limpiador_categorico(mapeo_color, no_mapeados='conservar'), 'color_fruto_cosecha')


🔹 ANTES (datos originales)

=== Color del fruto al momento de cosechar ===
Valor: a la totalidad de su coloración, Frecuencia: 67
Valor: 1, Frecuencia: 66
Valor: 0.75, Frecuencia: 21
Valor: a 3/4 de su coloración, Frecuencia: 20
Valor: a mitad de su coloracion, Frecuencia: 14
Valor: 0.5, Frecuencia: 6
Valor: completamente verde, Frecuencia: 3
Observaciones totales:  197

----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Color del fruto al momento de cosechar ===
Valor: A la totalidad de su coloración, Frecuencia: 133
Valor: A 3/4 de su coloración, Frecuencia: 41
Valor: A mitad de su coloración, Frecuencia: 20
Valor: Completamente verde, Frecuencia: 3
Observaciones totales:  197


In [198]:
# --- Duración poscosecha (se explota en 2 variables: días y condición de conservación) ---
inspeccionar_valores(df, VAR_POSCOSECHA)

def extraer_poscosecha(row):
    val = row['valor']
    if pd.isna(val):
        return []
        
    val_str = str(val).lower().strip()
    
    nulos = ['no lo registr', 'no ko registr', 'todavía no coseche', 
             'no se hizo', 'no sé efectuó', 'idem a la anterior']
    if any(x in val_str for x in nulos):
        return [('condicion_poscosecha', 'No se evaluó')]
        
    condicion = 'No especificado'
    if 'heladera' in val_str:
        if 'sin heladera' in val_str or 'fuera de heladera' in val_str:
            condicion = 'A temperatura ambiente'
        else:
            condicion = 'En heladera'
            
    dias = None
    if val_str == 'una semana': 
        dias = 7
    elif any(x in val_str for x in ['mas de una semana', 'más de una semana', 'mas de 7', 'más de 7']): 
        dias = 8
    elif '12 a 15' in val_str: 
        dias = 14
    elif 'duraron 8' in val_str: 
        dias = 8
    else:
        match = re.search(r'(\d+)', val_str)
        if match:
            num = int(match.group(1))
            if num > 0:
                dias = num
                
    resultados = []
    if dias is not None:
        resultados.append(('duracion_poscosecha_dias', dias))
        resultados.append(('condicion_poscosecha', condicion))
        
    return resultados

NUEVAS_POSCOSECHA = ['duracion_poscosecha_dias', 'condicion_poscosecha']

# explotar='tuplas': cada resultado es (nombre_variable, valor).
df_limpio = limpiar_variable(df_limpio, VAR_POSCOSECHA, extraer_poscosecha, por_fila=True, explotar='tuplas',
                             omitir_variable=NUEVAS_POSCOSECHA, mostrar=False)

es_nueva = df_limpio['variable'].isin(NUEVAS_POSCOSECHA)
df_limpio.loc[es_nueva, 'variable_limpia'] = df_limpio.loc[es_nueva, 'variable']

for var in NUEVAS_POSCOSECHA:
    inspeccionar_valores(df_limpio, var)


🔹 ANTES (datos originales)

=== Duración poscosecha (días que dura una muestra de 10 tomates en estado para ser consumido) ===
Valor: 7, Frecuencia: 20
Valor: 5, Frecuencia: 11
Valor: 30, Frecuencia: 6
Valor: 8, Frecuencia: 6
Valor: 7 días en heladera, Frecuencia: 5
Valor: 4, Frecuencia: 5
Valor: 15, Frecuencia: 5
Valor: mas de una semana, Frecuencia: 3
Valor: 1, Frecuencia: 3
Valor: 10, Frecuencia: 2
Valor: 25, Frecuencia: 2
Valor: 16, Frecuencia: 2
Valor: 2, Frecuencia: 2
Valor: 12 a 15, Frecuencia: 2
Valor: 9, Frecuencia: 2
Valor: una semana, Frecuencia: 2
Valor: 3, Frecuencia: 2
Valor: 12, Frecuencia: 2
Valor: 5 dias (fuera de heladera), Frecuencia: 1
Valor: 4 dias (fuera de heladera), Frecuencia: 1
Valor: 20, Frecuencia: 1
Valor: más de 7, Frecuencia: 1
Valor: No llegaron a haber 10 tomates para hacer la prueba pero en promedio duraron 8 días en poscosecha (sin heladera)., Frecuencia: 1
Valor: 21, Frecuencia: 1
Valor: 14, Frecuencia: 1
Valor: 0 no lo registre, Frecuencia: 1
Valor

In [199]:
# --- Cantidad de frutos cosechados ---
inspeccionar_valores(df, 'Cantidad de frutos cosechados')

# Agregamos los datos de un participante (H051T) que estaban en su planilla pero no se pusieron en el dataset
NOTA_H051T = 'Valor reconstruido desde registros de fechas'
H051T = [  # (variedad_id, variedad_original, cantidad de frutos, peso en kg)
    ('V7',   '7.0',  '30',  '3.465'),
    ('V56',  'JDP',  '39',  '5.88'),   # VJDP == V56 
    ('V43',  '43.2', '116', '2.97'),
    ('V80',  '80.0', '38',  '0.64'),
    ('V33',  '33.0', '41',  '2.48'),
]
nuevos_registros = [
    {'campaña': '2023-2024', 'id_participante': 'H051T', 'id_sitio': 'H051T', 'sub_ensayo': 1,
     'variedad_id': vid, 'variedad_original': vorig, 'etapa': 'Cosecha y poscosecha',
     'variable': var, 'variable_limpia': var_limpia, 'valor': val, 'variable_norm': val, 'nota': NOTA_H051T}
    for vid, vorig, frutos, peso in H051T
    for var, var_limpia, val in [
        ('Cantidad de frutos cosechados', 'Cantidad de frutos cosechados', frutos),
        ('Peso de todos los frutos cosechados', VAR_PESO, peso),
    ]
]
df_h051t = pd.DataFrame(nuevos_registros)

mask_h051t_existente = (
    (df_limpio['id_participante'] == 'H051T') &
    (df_limpio['variable_limpia'].isin(['Cantidad de frutos cosechados', VAR_PESO]))
)
df_limpio = pd.concat([df_limpio[~mask_h051t_existente], df_h051t], ignore_index=True)

reemplazos_directos = {
    'Hay 3 tomates en una planta. Las otras no tienen fruto.': '3',
    '168 hasta 24/3': '168',
    'aprox.por planta': None
}
df_limpio = mapear_variable_numerica(df_limpio, 'Cantidad de frutos cosechados', reemplazos_directos,
                                     descartar_contiene=['por planta'], tipo=None, borrar_nulos=True)


🔹 ANTES (datos originales)

=== Cantidad de frutos cosechados ===
Valor: 3, Frecuencia: 8
Valor: 9, Frecuencia: 5
Valor: 4, Frecuencia: 5
Valor: 2, Frecuencia: 4
Valor: 5, Frecuencia: 4
Valor: 7, Frecuencia: 3
Valor: 19, Frecuencia: 3
Valor: 14, Frecuencia: 2
Valor: 30, Frecuencia: 2
Valor: 8, Frecuencia: 2
Valor: 10, Frecuencia: 2
Valor: 13, Frecuencia: 2
Valor: 15, Frecuencia: 1
Valor: 137, Frecuencia: 1
Valor: 12, Frecuencia: 1
Valor: aprox.por planta, Frecuencia: 1
Valor: 127, Frecuencia: 1
Valor: 91, Frecuencia: 1
Valor: 20 frutos por planta, Frecuencia: 1
Valor: 60 frutos por planta, Frecuencia: 1
Valor: 15 frutos por planta, Frecuencia: 1
Valor: 200, Frecuencia: 1
Valor: 362, Frecuencia: 1
Valor: 2375, Frecuencia: 1
Valor: 17, Frecuencia: 1
Valor: 118, Frecuencia: 1
Valor: 27, Frecuencia: 1
Valor: 32, Frecuencia: 1
Valor: 237, Frecuencia: 1
Valor: 38, Frecuencia: 1
Valor: 18, Frecuencia: 1
Valor: 6, Frecuencia: 1
Valor: Hay 3 tomates en una planta. Las otras no tienen fruto., F

In [200]:
# --- Peso de todos los frutos cosechados (KG) ---
# Se tuvo que revisar los registros muy cuidadosamente, porque había errores de carga, de interpretación, respuestas en gramos y en kilogramos, entre otros.
inspeccionar_valores(df, VAR_PESO)

CORRECCIONES_PESO = [
    ({'campaña': '2024-2025', 'id_participante': 'AA009T', 'variedad_id': 'V172'}, '20 gr promedio',  '0.6'),
    ({'campaña': '2023-2024', 'id_participante': 'AA122T', 'variedad_id': 'V7'},   '80',              '0.08'),   # 80 g
    ({'campaña': '2023-2024', 'id_participante': 'AA122T', 'variedad_id': 'V172'}, '25',              '0.025'),  # 25 g
    ({'id_participante': 'AA009T', 'variedad_id': 'V7'},                            '90 gr promedio',  '0.81'),
    ({'id_participante': 'AA009T', 'variedad_id': 'V56'},                           '200 gr promedio', '3.0'),
]
for filtros, texto_original, kg in CORRECCIONES_PESO:
    asignar_valor(df_limpio, VAR_PESO, kg, valor_actual=texto_original, **filtros)

mask_peso = df_limpio['variable_limpia'] == VAR_PESO

df_limpio.loc[mask_peso & df_limpio['valor'].astype(str).str.contains('21,498', na=False), 'valor'] = '21.498' # '21,498 kgs hasta 24/3' -> 21.498

# Observaciones contradictorias: completaron un peso pero en los comentarios dicen que no hubo cosecha (frutos verdes) -> se anulan
invalidas_verdes = mask_peso & df_limpio['valor'].isin(['5', '102']) & (
    df_limpio['nota'].astype(str).str.contains('frutos verdes', case=False, na=False) |
    df_limpio['id_participante'].isin(['AA020T']))  
df_limpio.loc[invalidas_verdes, 'valor'] = None

# 4. Gramos -> kg. Se descartan los "por planta" (sin saber cuántas plantas son no se puede obtener el total).
reemplazos_gramos = {
    '320 gramos': '0.32',
    '280 gramos': '0.28',
    'aprox. 9 gramos cada uno': '0.36',
    '1700 gramos': '1.7',
    'entre 20 y 40 gramos cada uno': '1.35',
    '800grs': '0.8',
    '430 gramos': '0.43',
    '420 gramos': '0.42',
    '32 gramos': '0.032',
    '800': '0.8',
    '400': '0.4',
    '300': '0.3',
    '3kg': '3',
    '195': '0.195',
    '324': '0.324',
}
df_limpio = mapear_variable_numerica(df_limpio, VAR_PESO, reemplazos_gramos,
                                     descartar_contiene=['por planta'], tipo=None, borrar_nulos=True)


🔹 ANTES (datos originales)

=== Peso de todos los frutos cosechados (KG) ===
Valor: 1, Frecuencia: 3
Valor: 200 gr promedio, Frecuencia: 1
Valor: 90 gr promedio, Frecuencia: 1
Valor: 324, Frecuencia: 1
Valor: peso total producido por planta, un promedio de 1,2 kg, Frecuencia: 1
Valor: peso total producido por planta, un promedio de 1,6 kg, Frecuencia: 1
Valor: 20 gr promedio, Frecuencia: 1
Valor: peso total producido por planta, un promedio de 1,7 kg, Frecuencia: 1
Valor: 0.773, Frecuencia: 1
Valor: 3.424, Frecuencia: 1
Valor: 9.966, Frecuencia: 1
Valor: 0.013, Frecuencia: 1
Valor: 0.12, Frecuencia: 1
Valor: 0.14, Frecuencia: 1
Valor: 15, Frecuencia: 1
Valor: 30.875, Frecuencia: 1
Valor: 45.609, Frecuencia: 1
Valor: 0.781, Frecuencia: 1
Valor: 0.097, Frecuencia: 1
Valor: 0.405, Frecuencia: 1
Valor: 1.44, Frecuencia: 1
Valor: 2.495, Frecuencia: 1
Valor: peso total producido por planta, un promedio de 1,4 kg, Frecuencia: 1
Valor: 0.483, Frecuencia: 1
Valor: 1.71, Frecuencia: 1
Valor: 3.

##### Variable Rendimiento:

Se armo un csv aparte con únicamente las observaciones de estas variables de "Rendimiento X" para poder validarlas contra las planillas originales. El resultado se pegó abajo.

Rendimiento 1 a 6 (solo campaña 2022-2023, 86 filas en el original) se repartió así:
- **64 filas** → se suman por ensayo en "Peso de todos los frutos cosechados (KG)" (28 registros). Son las que están en `csv_rendimientos_validos`, con valores idénticos a los del original.
- **21 filas** → "Cantidad de frutos cosechados" (13 registros, lista `cantidades_rescatadas`): 13 con texto ("5 frutos", "2 tomatitos") y 8 enteros sin balanza, de ensayos que anotaron cantidad en vez de kg.
- **1 fila** → eliminada: H053T V90, "son muy chicos" (sin dato numérico).

In [201]:
# 1. Dataset de origen con las observaciones numéricas de peso validadas
csv_rendimientos_validos = """campaña,id_participante,id_sitio,sub_ensayo,variedad_id,variedad_original,etapa,variable,valor,nota,variable_norm,variable_limpia
2022-2023,H038T,H038T,1,V67,67,Cosecha y poscosecha,Rendimiento 1,0.662,,rendimiento 1,Rendimiento 1
2022-2023,H038T,H038T,1,V67,67,Cosecha y poscosecha,Rendimiento 2,1.626,,rendimiento 2,Rendimiento 2
2022-2023,H038T,H038T,1,V94,94,Cosecha y poscosecha,Rendimiento 1,0.17,,rendimiento 1,Rendimiento 1
2022-2023,H038T,H038T,1,V94,94,Cosecha y poscosecha,Rendimiento 2,0.4,,rendimiento 2,Rendimiento 2
2022-2023,H003T,H003T,1,V83,83,Cosecha y poscosecha,Rendimiento 1,0.885,,rendimiento 1,Rendimiento 1
2022-2023,H003T,H003T,1,V83,83,Cosecha y poscosecha,Rendimiento 2,1.466,,rendimiento 2,Rendimiento 2
2022-2023,H003T,H003T,1,V83,83,Cosecha y poscosecha,Rendimiento 3,1,,rendimiento 3,Rendimiento 3
2022-2023,H003T,H003T,1,V89,89,Cosecha y poscosecha,Rendimiento 1,0.1,,rendimiento 1,Rendimiento 1
2022-2023,H003T,H003T,1,V89,89,Cosecha y poscosecha,Rendimiento 2,0.4,,rendimiento 2,Rendimiento 2
2022-2023,H003T,H003T,1,V89,89,Cosecha y poscosecha,Rendimiento 3,0.4,,rendimiento 3,Rendimiento 3
2022-2023,H005T,H005T,1,V80,80,Cosecha y poscosecha,Rendimiento 1,0.05,,rendimiento 1,Rendimiento 1
2022-2023,H005T,H005T,1,V80,80,Cosecha y poscosecha,Rendimiento 2,0.29,,rendimiento 2,Rendimiento 2
2022-2023,H005T,H005T,1,V80,80,Cosecha y poscosecha,Rendimiento 3,0.1,,rendimiento 3,Rendimiento 3
2022-2023,H005T,H005T,1,V80,80,Cosecha y poscosecha,Rendimiento 4,0.15,,rendimiento 4,Rendimiento 4
2022-2023,H005T,H005T,1,V80,80,Cosecha y poscosecha,Rendimiento 5,0.05,,rendimiento 5,Rendimiento 5
2022-2023,H005T,H005T,1,V16,16,Cosecha y poscosecha,Rendimiento 1,0.1,,rendimiento 1,Rendimiento 1
2022-2023,H005T,H005T,1,V16,16,Cosecha y poscosecha,Rendimiento 2,0.04,,rendimiento 2,Rendimiento 2
2022-2023,H005T,H005T,1,V16,16,Cosecha y poscosecha,Rendimiento 3,0.52,,rendimiento 3,Rendimiento 3
2022-2023,H005T,H005T,1,V16,16,Cosecha y poscosecha,Rendimiento 4,0.05,,rendimiento 4,Rendimiento 4
2022-2023,H005T,H005T,1,V16,16,Cosecha y poscosecha,Rendimiento 5,0.26,,rendimiento 5,Rendimiento 5
2022-2023,H005T,H005T,1,V16,16,Cosecha y poscosecha,Rendimiento 6,0.15,,rendimiento 6,Rendimiento 6
2022-2023,H010T,H010T,1,V80,80,Cosecha y poscosecha,Rendimiento 1,0.15,,rendimiento 1,Rendimiento 1
2022-2023,H020T,H020T,1,V83,83,Cosecha y poscosecha,Rendimiento 1,0.1,,rendimiento 1,Rendimiento 1
2022-2023,H025T,H025T,1,V97,97,Cosecha y poscosecha,Rendimiento 1,0.25,,rendimiento 1,Rendimiento 1
2022-2023,H030T,H030T,1,V117,117,Cosecha y poscosecha,Rendimiento 1,0.2,,rendimiento 1,Rendimiento 1
2022-2023,H032T,H032T,1,V43,43,Cosecha y poscosecha,Rendimiento 1,1,,rendimiento 1,Rendimiento 1
2022-2023,H032T,H032T,1,V43,43,Cosecha y poscosecha,Rendimiento 2,1.1,,rendimiento 2,Rendimiento 2
2022-2023,H032T,H032T,1,V43,43,Cosecha y poscosecha,Rendimiento 3,1.3,,rendimiento 3,Rendimiento 3
2022-2023,H032T,H032T,1,V43,43,Cosecha y poscosecha,Rendimiento 4,1.1,,rendimiento 4,Rendimiento 4
2022-2023,H032T,H032T,1,V43,43,Cosecha y poscosecha,Rendimiento 5,0.7,,rendimiento 5,Rendimiento 5
2022-2023,H032T,H032T,1,V43,43,Cosecha y poscosecha,Rendimiento 6,1.2,,rendimiento 6,Rendimiento 6
2022-2023,H033T,H033T,1,VMATU,Matusalem,Cosecha y poscosecha,Rendimiento 1,0.1,,rendimiento 1,Rendimiento 1
2022-2023,H033T,H033T,1,VMATU,Matusalem,Cosecha y poscosecha,Rendimiento 2,0.6,,rendimiento 2,Rendimiento 2
2022-2023,H033T,H033T,1,VGEMA,Gema,Cosecha y poscosecha,Rendimiento 1,0.025,,rendimiento 1,Rendimiento 1
2022-2023,H033T,H033T,1,VGEMA,Gema,Cosecha y poscosecha,Rendimiento 2,0.1,,rendimiento 2,Rendimiento 2
2022-2023,H033T,H033T,1,VGEMA,Gema,Cosecha y poscosecha,Rendimiento 3,0.1,,rendimiento 3,Rendimiento 3
2022-2023,H033T,H033T,1,V67,67,Cosecha y poscosecha,Rendimiento 1,0.6,,rendimiento 1,Rendimiento 1
2022-2023,H033T,H033T,1,V67,67,Cosecha y poscosecha,Rendimiento 2,0.25,,rendimiento 2,Rendimiento 2
2022-2023,H033T,H033T,1,V67,67,Cosecha y poscosecha,Rendimiento 3,0.052,,rendimiento 3,Rendimiento 3
2022-2023,H033T,H033T,1,V115,115,Cosecha y poscosecha,Rendimiento 1,0.102,,rendimiento 1,Rendimiento 1
2022-2023,H033T,H033T,1,V115,115,Cosecha y poscosecha,Rendimiento 2,0.64,,rendimiento 2,Rendimiento 2
2022-2023,H033T,H033T,1,V115,115,Cosecha y poscosecha,Rendimiento 3,0.06,,rendimiento 3,Rendimiento 3
2022-2023,H033T,H033T,1,V172,172,Cosecha y poscosecha,Rendimiento 1,0.3,,rendimiento 1,Rendimiento 1
2022-2023,H033T,H033T,1,V172,172,Cosecha y poscosecha,Rendimiento 2,0.4,,rendimiento 2,Rendimiento 2
2022-2023,H033T,H033T,1,V172,172,Cosecha y poscosecha,Rendimiento 3,0.1,,rendimiento 3,Rendimiento 3
2022-2023,AA021T,AA021T,1,V174,174,Cosecha y poscosecha,Rendimiento 1,0.05,,rendimiento 1,Rendimiento 1
2022-2023,AA138T,AA138T,1,V90,90,Cosecha y poscosecha,Rendimiento 1,0.025,,rendimiento 1,Rendimiento 1
2022-2023,AA138T,AA138T,1,V90,90,Cosecha y poscosecha,Rendimiento 2,0.025,,rendimiento 2,Rendimiento 2
2022-2023,H048T,H048T,1,V67,67,Cosecha y poscosecha,Rendimiento 1,0.4,,rendimiento 1,Rendimiento 1
2022-2023,H048T,H048T,1,V115,115,Cosecha y poscosecha,Rendimiento 1,0.5,,rendimiento 1,Rendimiento 1
2022-2023,AA148T,AA148T,1,V34,34,Cosecha y poscosecha,Rendimiento 1,0.05,,rendimiento 1,Rendimiento 1
2022-2023,AA148T,AA148T,1,V34,34,Cosecha y poscosecha,Rendimiento 2,0.01,,rendimiento 2,Rendimiento 2
2022-2023,AA148T,AA148T,1,V97,97,Cosecha y poscosecha,Rendimiento 2,0.1,,rendimiento 2,Rendimiento 2
2022-2023,AA148T,AA148T,1,V104,104,Cosecha y poscosecha,Rendimiento 2,0.05,,rendimiento 2,Rendimiento 2
2022-2023,AA023T,AA023T,1,V65,65,Cosecha y poscosecha,Rendimiento 1,0.3,,rendimiento 1,Rendimiento 1
2022-2023,AA023T,AA023T,1,V65,65,Cosecha y poscosecha,Rendimiento 2,1.2,,rendimiento 2,Rendimiento 2
2022-2023,AA023T,AA023T,1,V65,65,Cosecha y poscosecha,Rendimiento 3,0.6,,rendimiento 3,Rendimiento 3
2022-2023,AA152T,AA152T,1,V90,90,Cosecha y poscosecha,Rendimiento 1,0.1,,rendimiento 1,Rendimiento 1
2022-2023,H009T,H009T,1,V34,34,Cosecha y poscosecha,Rendimiento 1,0.2,,rendimiento 1,Rendimiento 1
2022-2023,H009T,H009T,1,V34,34,Cosecha y poscosecha,Rendimiento 2,1.8,,rendimiento 2,Rendimiento 2
2022-2023,H027T,H027T,1,V115,115,Cosecha y poscosecha,Rendimiento 1,0.2,,rendimiento 1,Rendimiento 1
2022-2023,H027T,H027T,1,V115,115,Cosecha y poscosecha,Rendimiento 2,0.3,,rendimiento 2,Rendimiento 2
2022-2023,H027T,H027T,1,V66,66,Cosecha y poscosecha,Rendimiento 1,0.3,,rendimiento 1,Rendimiento 1
2022-2023,H027T,H027T,1,V66,66,Cosecha y poscosecha,Rendimiento 2,0.1,,rendimiento 2,Rendimiento 2
"""

df_rend = pd.read_csv(io.StringIO(csv_rendimientos_validos))
df_rend['valor'] = df_rend['valor'].astype(float)
cols_agrup = ['campaña', 'id_participante', 'id_sitio', 'sub_ensayo', 'variedad_id', 'variedad_original']
df_totales_peso = (
    df_rend.groupby(cols_agrup, as_index=False)
    .agg(
        etapa=('etapa', 'first'),
        variable=('variable', lambda x: 'Peso de todos los frutos cosechados'),
        valor=('valor', lambda x: round(x.sum(), 3)),
        nota=('nota', 'first'),
        variable_norm=('variable_norm', lambda x: 'peso de todos los frutos cosechados'),
        variable_limpia=('variable_limpia', lambda x: VAR_PESO)
    )
)

# Borramos las variables de "Renimiento X" y ejecuciones previas para poder volver a ejecutar sin problemas
vars_rendimiento = [f'Rendimiento {i}' for i in range(1, 7)]
mask_borrar_rend = (
    df_limpio['variable_limpia'].isin(vars_rendimiento) | 
    df_limpio['variable'].isin(vars_rendimiento)
)
participantes_tanda = df_totales_peso['id_participante'].unique()
mask_borrar_previos = (
    (df_limpio['variable_limpia'] == VAR_PESO) & 
    (df_limpio['campaña'] == '2022-2023') & 
    (df_limpio['id_participante'].isin(participantes_tanda))
)
df_limpio = df_limpio[~(mask_borrar_rend | mask_borrar_previos)]
df_limpio = pd.concat([df_limpio, df_totales_peso], ignore_index=True)

inspeccionar_valores(df_limpio, VAR_PESO)


----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Peso de todos los frutos cosechados (KG) ===
Valor: 0.4, Frecuencia: 4
Valor: 1.0, Frecuencia: 3
Valor: 0.8, Frecuencia: 3
Valor: 0.5, Frecuencia: 3
Valor: 0.05, Frecuencia: 3
Valor: 0.1, Frecuencia: 3
Valor: 0.64, Frecuencia: 2
Valor: 3.0, Frecuencia: 2
Valor: 0.324, Frecuencia: 1
Valor: 0.6, Frecuencia: 1
Valor: 0.81, Frecuencia: 1
Valor: 0.773, Frecuencia: 1
Valor: 45.609, Frecuencia: 1
Valor: 0.781, Frecuencia: 1
Valor: 0.097, Frecuencia: 1
Valor: 0.405, Frecuencia: 1
Valor: 0.12, Frecuencia: 1
Valor: 3.424, Frecuencia: 1
Valor: 9.966, Frecuencia: 1
Valor: 0.013, Frecuencia: 1
Valor: 1.71, Frecuencia: 1
Valor: 0.483, Frecuencia: 1
Valor: 2.495, Frecuencia: 1
Valor: 3.77, Frecuencia: 1
Valor: 1.795, Frecuencia: 1
Valor: 0.43, Frecuencia: 1
Valor: 1.167, Frecuencia: 1
Valor: 2.414, Frecuencia: 1
Valor: 0.42, Frec

In [202]:
# Registro de cantidades rescatadas de los textos
cantidades_rescatadas = [
    {'campaña': '2022-2023', 'id_participante': 'H041T', 'id_sitio': 'H041T', 'sub_ensayo': 1, 'variedad_id': 'V80', 'variedad_original': '80', 'cant': '2'},
    {'campaña': '2022-2023', 'id_participante': 'H046T', 'id_sitio': 'H046T', 'sub_ensayo': 1, 'variedad_id': 'V177', 'variedad_original': '177', 'cant': '4'},
    {'campaña': '2022-2023', 'id_participante': 'H046T', 'id_sitio': 'H046T', 'sub_ensayo': 1, 'variedad_id': 'V97', 'variedad_original': '97', 'cant': '35'},
    {'campaña': '2022-2023', 'id_participante': 'H046T', 'id_sitio': 'H046T', 'sub_ensayo': 1, 'variedad_id': 'V34', 'variedad_original': '34', 'cant': '4'},
    {'campaña': '2022-2023', 'id_participante': 'H051T', 'id_sitio': 'H051T', 'sub_ensayo': 1, 'variedad_id': 'V80', 'variedad_original': '80', 'cant': '9'},
    {'campaña': '2022-2023', 'id_participante': 'H043T', 'id_sitio': 'H043T', 'sub_ensayo': 1, 'variedad_id': 'V80', 'variedad_original': '80', 'cant': '5'},
    {'campaña': '2022-2023', 'id_participante': 'H043T', 'id_sitio': 'H043T', 'sub_ensayo': 1, 'variedad_id': 'V89', 'variedad_original': '89', 'cant': '7'},
    {'campaña': '2022-2023', 'id_participante': 'H049T', 'id_sitio': 'H049T', 'sub_ensayo': 1, 'variedad_id': 'V67', 'variedad_original': '67', 'cant': '15'},
    {'campaña': '2022-2023', 'id_participante': 'H049T', 'id_sitio': 'H049T', 'sub_ensayo': 1, 'variedad_id': 'V90', 'variedad_original': '90', 'cant': '15'},
    {'campaña': '2022-2023', 'id_participante': 'H049T', 'id_sitio': 'H049T', 'sub_ensayo': 1, 'variedad_id': 'V115', 'variedad_original': '115', 'cant': '2'},
    {'campaña': '2022-2023', 'id_participante': 'H048T', 'id_sitio': 'H048T', 'sub_ensayo': 1, 'variedad_id': 'V90', 'variedad_original': '90', 'cant': '2'},
    {'campaña': '2022-2023', 'id_participante': 'H012T', 'id_sitio': 'H012T', 'sub_ensayo': 1, 'variedad_id': 'V115', 'variedad_original': '115', 'cant': '1'},
    {'campaña': '2022-2023', 'id_participante': 'H012T', 'id_sitio': 'H012T', 'sub_ensayo': 1, 'variedad_id': 'V83', 'variedad_original': '83', 'cant': '2'},
]

filas_nuevas = []
for reg in cantidades_rescatadas:
    filas_nuevas.append({
        'campaña': reg['campaña'],
        'id_participante': reg['id_participante'],
        'id_sitio': reg['id_sitio'],
        'sub_ensayo': reg['sub_ensayo'],
        'variedad_id': reg['variedad_id'],
        'variedad_original': reg['variedad_original'],
        'etapa': 'Cosecha y poscosecha',
        'variable': 'Cantidad de frutos cosechados',
        'variable_limpia': 'Cantidad de frutos cosechados',
        'valor': float(reg['cant']),  # Forzado a float numérico
        'variable_norm': 'cantidad de frutos cosechados',
        'nota': 'Reclasificado de Rendimiento a Cantidad (enteros sin balanza)'
    })
df_rescatados = pd.DataFrame(filas_nuevas)


# Borramos las variables de "Renimiento X" y ejecuciones previas para poder volver a ejecutar sin problemas
participantes_afectados = ['H041T', 'H046T', 'H051T', 'H043T', 'H049T', 'H048T', 'H012T', 'H053T']
mask_rend_texto = (
    df_limpio['id_participante'].isin(participantes_afectados) & 
    df_limpio['variable_limpia'].str.contains('Rendimiento', case=False, na=False)
)
mask_previos_rescatados = (
    (df_limpio['variable_limpia'] == 'Cantidad de frutos cosechados') &
    (df_limpio['campaña'] == '2022-2023') &
    (df_limpio['id_participante'].isin(participantes_afectados)) &
    (df_limpio['nota'].astype(str).str.contains('Reclasificado de Rendimiento', case=False, na=False))
)
df_limpio = df_limpio[~(mask_rend_texto | mask_previos_rescatados)]
df_limpio = pd.concat([df_limpio, df_rescatados], ignore_index=True)

inspeccionar_valores(df_limpio, 'Cantidad de frutos cosechados')


----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Cantidad de frutos cosechados ===
Valor: 3.0, Frecuencia: 9
Valor: 2.0, Frecuencia: 8
Valor: 4.0, Frecuencia: 7
Valor: 9.0, Frecuencia: 6
Valor: 5.0, Frecuencia: 5
Valor: 7.0, Frecuencia: 4
Valor: 30.0, Frecuencia: 3
Valor: 15.0, Frecuencia: 3
Valor: 19.0, Frecuencia: 3
Valor: 13.0, Frecuencia: 2
Valor: 14.0, Frecuencia: 2
Valor: 38.0, Frecuencia: 2
Valor: 1.0, Frecuencia: 2
Valor: 10.0, Frecuencia: 2
Valor: 8.0, Frecuencia: 2
Valor: 91.0, Frecuencia: 1
Valor: 137.0, Frecuencia: 1
Valor: 127.0, Frecuencia: 1
Valor: 2375.0, Frecuencia: 1
Valor: 12.0, Frecuencia: 1
Valor: 17.0, Frecuencia: 1
Valor: 237.0, Frecuencia: 1
Valor: 200.0, Frecuencia: 1
Valor: 362.0, Frecuencia: 1
Valor: 32.0, Frecuencia: 1
Valor: 27.0, Frecuencia: 1
Valor: 18.0, Frecuencia: 1
Valor: 118.0, Frecuencia: 1
Valor: 6.0, Frecuencia: 1
Valor: 40.

#### 2.3.9 Degustación — 16/16
- ¿Cuánto te gusta este tomate? (...): 164
- Dulce: 123
- Ácido: 87 
- Jugoso: 154 
- Persistente: 80
- Olor a tomate maduro: 121 
- Olor a tomate verde: 32
- Mucha pulpa: 99 
- Poca pulpa: 68 
- Pulpa dura: 43 
- Pulpa blanda: 102
- Mucha semilla: 108 
- Poca semilla: 59
- Aguachento: 49 
- Pastoso: 34
- Piel dura: 104

In [203]:
# --- ¿Cuánto te gusta este tomate? ---
VAR_GUSTO = '¿Cuánto te gusta este tomate? (1 = me disgusta muchísimo, 5 = ni me gusta ni me disgusta, 9 = me gusta muchísimo)'
inspeccionar_valores(df, VAR_GUSTO)

def a_float(val):
    if pd.isna(val):
        return None
    try:
        return float(val)
    except Exception:
        return None

df_limpio = limpiar_variable(df_limpio, VAR_GUSTO, a_float, 'gusto_general')


🔹 ANTES (datos originales)

=== ¿Cuánto te gusta este tomate? (1 = me disgusta muchísimo, 5 = ni me gusta ni me disgusta, 9 = me gusta muchísimo) ===
Valor: 9, Frecuencia: 52
Valor: 8, Frecuencia: 43
Valor: 7, Frecuencia: 28
Valor: 5, Frecuencia: 15
Valor: 6, Frecuencia: 15
Valor: 4, Frecuencia: 3
Valor: 3, Frecuencia: 3
Valor: 1, Frecuencia: 2
Valor: 8.5, Frecuencia: 1
Valor: 2, Frecuencia: 1
Observaciones totales:  163

----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== ¿Cuánto te gusta este tomate? (1 = me disgusta muchísimo, 5 = ni me gusta ni me disgusta, 9 = me gusta muchísimo) ===
Valor: 9.0, Frecuencia: 52
Valor: 8.0, Frecuencia: 43
Valor: 7.0, Frecuencia: 28
Valor: 5.0, Frecuencia: 15
Valor: 6.0, Frecuencia: 15
Valor: 4.0, Frecuencia: 3
Valor: 3.0, Frecuencia: 3
Valor: 1.0, Frecuencia: 2
Valor: 8.5, Frecuencia: 1
Valor: 2.0, Frecuencia: 1
Observaciones tot

*Aclaración*: Las siguientes variables tienen 2 escalas:
- Booleana: el evaluador indica si percibió o le llamó la atención ese atributo durante la degustación.
- Numérica (1-9): el evaluador indica que tanto sintió la presencia de ese atributo durante la degustación.

In [204]:
evaluaciones_degustacion = [
    'Dulce', 'Ácido', 'Jugoso', 'Persistente', 
    'Olor a tomate maduro', 'Olor a tomate verde', 
    'Mucha pulpa', 'Poca pulpa', 'Pulpa dura', 
    'Pulpa blanda', 'Mucha semilla', 'Poca semilla', 
    'Aguachento', 'Pastoso', 'Piel dura'
]

# Mapeo en bucle sobre los valores booleanos usando SI_NO
for var_degustacion in evaluaciones_degustacion:
    df_limpio = mapear_variable_texto(
        dataset=df_limpio, 
        variable_limpia=var_degustacion, 
        diccionario_mapeo=SI_NO,
        mostrar=False
    )

# Corrección específica para la campaña 2022-2023:
# Mapear los valores True (que se cargaron como 1) a 'Sí'. 
mask_2022_degustacion = (
    (df_limpio['campaña'] == '2022-2023') &
    (df_limpio['variable_limpia'].isin(evaluaciones_degustacion)) &
    (df_limpio['valor'].astype(str).str.strip().isin(['1', '1.0']))
)
df_limpio.loc[mask_2022_degustacion, 'valor'] = 'Sí'

for var_degustacion in evaluaciones_degustacion:
    inspeccionar_valores(df_limpio, var_degustacion)


----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Dulce ===
Valor: Sí, Frecuencia: 96
Valor: 7, Frecuencia: 9
Valor: 6, Frecuencia: 6
Valor: 5, Frecuencia: 4
Valor: 8, Frecuencia: 4
Valor: No, Frecuencia: 2
Valor: 2, Frecuencia: 1
Valor: 3, Frecuencia: 1
Observaciones totales:  123

----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== Ácido ===
Valor: Sí, Frecuencia: 57
Valor: 4, Frecuencia: 8
Valor: No, Frecuencia: 4
Valor: 3, Frecuencia: 4
Valor: 7, Frecuencia: 4
Valor: 2, Frecuencia: 4
Valor: 5, Frecuencia: 4
Valor: 6, Frecuencia: 1
Valor: 8, Frecuencia: 1
Observaciones totales:  87

----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
-----------------------------------------------------------------

La campaña del 2024-2025 contaba con valores numéricos que **no eran reales**. Como en esta campaña no se específico el tipo de dato, entonces había observaciones booleanas, textos y un evaluador utilizó una escala numérica del 1-9. El modelo de IA (Claude) alucinó y convirtió todo a una escala numérica fictia de 1-5.

Como en la campaña de 2025-2026 sí se utilizó una escala del 1-9, se decidió mantener las observaciones del evaluador que realmente uso la escala numérica y mapear los textos reales a booleanos.

Observacions reales por usuario:
- AA014T $\rightarrow$ respondió "Sí" y "No". Se dejo igual.

- AA023T $\rightarrow$ respondió "Sí" y "No". Se dejo igual.

- AA017T $\rightarrow$ respondió con checks. Se mapean a Sí.

- AA009T $\rightarrow$ respondió con palabras tipo "si, y "muy", "no". Se mapeo a Sí/No.

- AA001T $\rightarrow$ respondió con palabras tipo "si, "poco", "bastante", "muy", "más o menos", "un poco", "no". Se mapeo a Sí/No.

- AA005T $\rightarrow$ respondió con palabras tipo "si, "poco", "fuerte", "no". Se mapeo a Sí/No.

- AA021T $\rightarrow$ respondió en escala numérica de 1-9 (dejando vacíos la mayoría de atributos). Se dejo igual.


Decisiones de mapeo:
- "muy", "fuerte", "bastante", "un poco", "más o menos", "si" $\rightarrow$ "Sí"
- "no", "poco", "ni" $\rightarrow$ "No"
- Para las planillas donde se registraron únicamente respuestas afirmativas (Sí/True), los atributos vacíos (no medidos o no percibidos) se imputan con 'No' en la columna valor, ya que el modelo de IA no los había incluido al armar el conjunto.




In [205]:
COL_USUARIO   = "id_participante"
COL_CAMPANA   = "campaña"
COLS_UNIDAD_EXTRA = ['id_sitio', 'sub_ensayo', 'variedad_id', 'variedad_original'] 

COL_ETAPA     = "etapa"
COL_VARIABLE  = "variable_limpia"
COL_VALOR     = "valor"
COL_VAR_ORIG = "variedad_original"
ETAPA_DEGUSTACION = "Degustación"
CAMPANAS = ["2022-2023", "2023-2024", "2024-2025"]


def completar_nos_degustacion(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    claves = [COL_USUARIO, COL_CAMPANA] + COLS_UNIDAD_EXTRA

    deg = df[df[COL_ETAPA] == ETAPA_DEGUSTACION]
    variables = evaluaciones_degustacion

    # Filas de degustación en las campañas objetivo con "Sí"
    es_si = deg[COL_VALOR].astype(str).str.strip().str.lower().isin(["sí", "si"])
    deg_obj = deg[deg[COL_CAMPANA].isin(CAMPANAS) & es_si]

    # Unidades (usuario, campaña, extras) que llegaron a degustación con algún Sí y los No faltantes
    unidades = deg_obj[claves].drop_duplicates()
    print(f"Usuarios detectados: {unidades[COL_USUARIO].nunique()} | unidades: {len(unidades)}")

    grilla = unidades.merge(pd.DataFrame({COL_VARIABLE: variables}), how="cross")
    cargadas = deg[claves + [COL_VARIABLE]].drop_duplicates()
    faltantes = grilla.merge(cargadas, on=claves + [COL_VARIABLE],
                             how="left", indicator=True)
    faltantes = faltantes[faltantes["_merge"] == "left_only"].drop(columns="_merge")
    faltantes[COL_ETAPA] = ETAPA_DEGUSTACION
    faltantes[COL_VALOR] = "No"
    faltantes["variable"] = faltantes[COL_VARIABLE]
    faltantes["nota"] = "imputado: vacío → No"
    print(f"Filas 'No' agregadas: {len(faltantes)}")

    faltantes = faltantes.reindex(columns=df.columns)

    return pd.concat([df, faltantes], ignore_index=True)

df_limpio = completar_nos_degustacion(df_limpio)
df_limpio.shape

Usuarios detectados: 60 | unidades: 166
Filas 'No' agregadas: 1532


(11687, 12)

#### 2.3.10 Eliminamos las variables deprecadas (se tomaron en una sola campaña y/o no tienen ningún valor de estudio)

Estas variables no fueron trabajadas ni limpiadas, se analizaron y se llegó a la conclusión de que quedaron discontinuadas y no se van a subir a la base de datos:

- **#por contenedor**: 62 $\rightarrow$ Plantines trasplantados por contenedor. Solo se registró en 2022-2023.

- **Volumen**: 57 $\rightarrow$ Volumen en litros del contenedor del trasplante a maceta. Solo se registró en 2022-2023.

- **Surcos**: 53 $\rightarrow$ Cantidad de surcos. Solo se registró en 2022-2023.

- **Distancia surcos**: 51 $\rightarrow$ Distancia entre surcos. Solo se registró en 2022-2023.

- **Distancia plantas**: 59 $\rightarrow$ Distancia entre plantas de un mismo surco. Solo se registró en 2022-2023.

- **Cultivo antecesor**: 56 $\rightarrow$ Cultivo anterior a la plantación del tomate. Solo se registró en 2022-2023.

- **Muerte por virus rugoso del tomate**: 100 $\rightarrow$ Fue utilizada en diversas campañas, pero todas las observaciones pusieron que "No", por lo que pierde todo su impacto al ser todas respuestas negativas y no aporta nada a la base de datos (puede servir únicamente como información para decir que no se reportaron muertes por este virus). Además se transformo a "¿A qué atribuís las muertes de plantas?".

In [206]:
variables_deprecadas = [
    '#por contenedor',
    'Volumen',
    'Surcos',
    'Distancia surcos',
    'Distancia plantas',
    'Cultivo antecesor',
    'Muerte por virus rugoso del tomate',
]
es_deprecada = df_limpio['variable_limpia'].isin(variables_deprecadas)
print(df_limpio.loc[es_deprecada, 'variable_limpia'].value_counts().reindex(variables_deprecadas, fill_value=0).to_string())

df_limpio = df_limpio[~es_deprecada].reset_index(drop=True)
print(f"\nSe eliminaron {es_deprecada.sum()} observaciones | quedan {df_limpio.shape[0]} observaciones en total.")

variable_limpia
#por contenedor                        62
Volumen                                57
Surcos                                 53
Distancia surcos                       51
Distancia plantas                      59
Cultivo antecesor                      56
Muerte por virus rugoso del tomate    100

Se eliminaron 438 observaciones | quedan 11249 observaciones en total.


#### 2.3.11 Slugs (identificadores simplificados) de los nombres de las variables

- Optimizan las consultas en la base de datos
- Evitamos errores al simplificar la escritura del código

*Aclaración*: fue un fallo hacerlos con toda la limpieza hecha, se deberían haber hecho desde el principio.

In [207]:
# --- Slugs para los nombres de variable ---
# a_slug: función automática (sin tildes, minúsculas, guiones bajos, sin conectores tipo "de", "la", "a")
# SLUGS_MANUALES: los nombres que se eligen a mano (los largos o con aclaraciones entre paréntesis)
PALABRAS_VACIAS = {'de', 'del', 'la', 'las', 'el', 'los', 'a', 'al', 'en', 'y', 'un', 'una'}


def a_slug(texto):
    """'Olor a tomate maduro' -> 'olor_tomate_maduro' | 'Superficie de la parcela (m²)' -> 'superficie_parcela_m2'"""
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode().lower()
    palabras = [p for p in re.split(r'[^a-z0-9]+', texto) if p and p not in PALABRAS_VACIAS]
    return '_'.join(palabras)


SLUGS_MANUALES = {
    VAR_EMERGENCIA: 'fecha_emergencia',
    VAR_SUSTRATO: 'sustrato',
    VAR_FLORACION: 'fecha_primera_floracion',
    VAR_ESTRES_HIDRICO: 'tolerancia_estres_hidrico',
    VAR_SOL: 'tolerancia_quemadura_sol',
    VAR_SANIDAD: 'sanidad',
    VAR_UNIFORMIDAD: 'uniformidad',
    VAR_CAUSAS: 'causa_desuniformidad', # Cambia el nombre porque se definió otra pregunta. Sus valores se mapearon en base a esta nueva definición
    'Color del fruto al momento de cosechar': 'color_fruto_cosecha',
    'Cantidad de plantas trasplantadas a la parcela': 'cantidad_plantas_trasplantadas_parcela',
    VAR_PESO: 'peso_total_cosechado_kg',
    VAR_GUSTO: 'gusto_general',
    'Plagas observadas': 'sanidad_observada', # Cambia de nombre porque se definió otra pregunta. Sus valores se mapearon en base a esta nueva definición
    'Cantidad de frutos cosechados': 'total_frutos_cosechados'
}

variables_actuales = sorted(df_limpio['variable_limpia'].dropna().unique())
slugs_variables = {v: SLUGS_MANUALES.get(v, a_slug(v)) for v in variables_actuales}

# --- Validaciones
repetidos = pd.Series(slugs_variables).loc[lambda s: s.duplicated(keep=False)]
largos = {v: s for v, s in slugs_variables.items() if len(s) > 35}
sin_usar = [v for v in SLUGS_MANUALES if v not in slugs_variables]
print(f"{len(slugs_variables)} variables | slugs repetidos: {len(repetidos)} | de más de 35 caracteres: {len(largos)} | "
      f"manuales que no aparecen en df_limpio: {len(sin_usar)}")
if len(repetidos): display(repetidos.sort_values())
if largos: display(pd.Series(largos))

df_limpio['variable_norm'] = df_limpio['variable_limpia'].map(slugs_variables)
assert df_limpio['variable_norm'].notna().all(), "Hay variables sin slug"

revision = df_limpio.groupby(['etapa', 'variable_norm', 'variable_limpia']).size().rename('n').reset_index()
display(revision)

tabla_variables = (df_limpio[['etapa', 'variable_norm', 'variable_limpia']].drop_duplicates()
                   .rename(columns={'variable_norm': 'variable', 'variable_limpia': 'etiqueta'})
                   .sort_values(['etapa', 'variable']).reset_index(drop=True))

# Guardamos en 'variable' el slug y eliminamos las columnas utilizadas como auxiliares durante el desarrollo.
df_final = (df_limpio.assign(variable=df_limpio['variable_norm'])
            .drop(columns=['variable_limpia', 'variable_norm'])
            .reset_index(drop=True))
print(f"df_final: {df_final.shape[0]} filas | variables: {df_final['variable'].nunique()} | columnas: {list(df_final.columns)}")

60 variables | slugs repetidos: 0 | de más de 35 caracteres: 1 | manuales que no aparecen en df_limpio: 0


Cantidad de plantas trasplantadas a la parcela    cantidad_plantas_trasplantadas_parcela
dtype: object

,etapa,variable_norm,variable_limpia,n
0,Cosecha y poscosecha,color_fruto_cosecha,Color del fruto al momento de cosechar,197
1,Cosecha y poscosecha,comentarios_generales,Comentarios generales,56
2,Cosecha y poscosecha,condicion_poscosecha,condicion_poscosecha,100
3,Cosecha y poscosecha,duracion_poscosecha_dias,duracion_poscosecha_dias,93
4,Cosecha y poscosecha,fecha_primera_cosecha,fecha_primera_cosecha,146
...,...,...,...,...
62,Trasplante a suelo,cultivo_asociado_suelo,Cultivo asociado (suelo),200
63,Trasplante a suelo,densidad_plantas_m2,Densidad (plantas/m²),114
64,Trasplante a suelo,fecha_trasplante_suelo,Fecha de trasplante (suelo),118
65,Trasplante a suelo,superficie_parcela_m2,Superficie de la parcela (m²),140


df_final: 11249 filas | variables: 60 | columnas: ['campaña', 'id_participante', 'id_sitio', 'sub_ensayo', 'variedad_id', 'variedad_original', 'etapa', 'variable', 'valor', 'nota']


In [208]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(
        df_final[['etapa', 'variable']]
        .drop_duplicates()
        .sort_values(by=['etapa', 'variable'])
        .reset_index(drop=True)
    )

,etapa,variable
0,Cosecha y poscosecha,color_fruto_cosecha
1,Cosecha y poscosecha,comentarios_generales
2,Cosecha y poscosecha,condicion_poscosecha
3,Cosecha y poscosecha,duracion_poscosecha_dias
4,Cosecha y poscosecha,fecha_primera_cosecha
5,Cosecha y poscosecha,fecha_ultima_cosecha
6,Cosecha y poscosecha,peso_total_cosechado_kg
7,Cosecha y poscosecha,total_frutos_cosechados
8,Crecimiento y desarrollo,causa_desuniformidad
9,Crecimiento y desarrollo,comentarios_generales


#### 2.3.12 Comentarios generales
Para leer los `comentarios_generales` que son texto libre y son muchos para realizar revisiones manuales y a fondo, se creó **mediante Claude** una automatización (`scripts/tratamiento_comentarios/procesar_comentarios.py`), la cual usa reglas editables (`scripts/tratamiento_comentarios/reglas_comentarios.csv`: "si el comentario dice X → variable V = valor Y"). Si el comentario aporta un dato que falta y no hay contradicción, se agrega la observación (queda con la nota `Recuperado de comentario [regla] | valor original y comentario del que se recuperó`, así se puede auditar y revertir).

- Todos los comentarios (y los patrones de las reglas) se comparan en **minúsculas y sin tildes**.
- **Nunca** se pisa un dato ya cargado: si hay contradicción, o el comentario es dudoso (*creo, capaz, por ahora, lo ideal…*), o menciona variedades, o dice que corrige respuestas anteriores, queda para revisión manual.
- Las **negaciones** se interpretan: "no hice raleo" propone `raleo_frutos = No` (y no `Sí`); "no tuvo pulgones" no agrega nada.
- Para resolver los pendientes: abrir `scripts/tratamiento_comentarios/pendientes_comentarios.csv`, escribir en la columna `decision` **agregar**, **reemplazar** o **ignorar** y volver a correr la celda. Las decisiones se archivan en `scripts/tratamiento_comentarios/decisiones_comentarios.csv` y se reutilizan en ejecuciones siguientes.
- Para sumar reglas nuevas: mirar `df_sin_regla` (comentarios que ninguna regla reconoce) y agregar una fila en `reglas_comentarios.csv` que la capture.
- Con `python scripts/tratamiento_comentarios/test_procesar_comentarios.py` se puede comprobar que no se rompió nada.


*Aclaración importante*: se revisó, iteró y aseguró el correcto funcionamiento y la consistencia de los resultados del archivo `scripts/tratamiento_comentarios/procesar_comentarios.py` sobre las reglas definidas. Pero no se indagó en como se procesaban estos comentarios, la revisión humana fue más por el lado de garantizar la integridad y coherencia de los resultados y las observaciones agregadas. 

Además, la gran mayoría de los comentarios que no entran por ninguna regla (df_sin_regla) son principalmente porque no aportan un valor directo que no este presente ya.

In [209]:
from comentarios.procesar_comentarios import (
    procesar_comentarios,
    exportar_pendientes,
    resumen,
    comentarios_sin_regla,
    auditar_cambios,
)

In [210]:
# Decisiones manuales: el archivo persistente + lo que se acaba de completar en el CSV de pendientes
DECISIONES = [
    'scripts/tratamiento_comentarios/decisiones_comentarios.csv',
    'scripts/tratamiento_comentarios/pendientes_comentarios.csv',
]

# 1. Recuperación automática de datos desde los comentarios (el dataset largo + reglas + decisiones)
df_final_recuperado, informe = procesar_comentarios(
    df_final,
    reglas='scripts/tratamiento_comentarios/reglas_comentarios.csv',
    decisiones=DECISIONES,
)

# 2. Resumen de lo auto-aplicado, descartado o pendiente
resumen(informe)

# 3. Auditoría: qué filas se agregaron/quitaron por variable y chequeos de coherencia
cambios, problemas = auditar_cambios(df_final, df_final_recuperado)

# 4. Comentarios que requieren revisión manual -> CSV (las decisiones ya escritas se archivan antes de regenerarlo)
exportar_pendientes(informe, ruta='scripts/tratamiento_comentarios/pendientes_comentarios.csv',
                    archivo_decisiones='scripts/tratamiento_comentarios/decisiones_comentarios.csv')

# 5. (Opcional) Comentarios para los que ninguna regla aplica aún, para decidir qué reglas escribir
df_sin_regla = comentarios_sin_regla(df_final_recuperado, 'scripts/tratamiento_comentarios/reglas_comentarios.csv')

procesar_comentarios versión 2026-10-30.3

estado
REDUNDANTE                   105
APLICADA_POR_DECISION         58
AUTO_APLICADA                 52
DESCARTADA_INCIERTA           28
IGNORADA_POR_DECISION          1
NEGADO_SIN_ACCION              1
DESCARTADA_FUERA_CATALOGO      1

Por variable (solo lo aplicado):
variable                  valor_propuesto                                             
condicion_poscosecha      A temperatura ambiente                                           4
                          No se evaluó                                                     3
control_malezas           Retiro manual o con herramientas                                 5
control_plagas            Biopreparados                                                   18
desbrote                  Sí                                                               3
eliminacion_brote_apical  No                                                               1
fertilizacion             Abonos líquido

In [211]:
recuperadas = df_final_recuperado[df_final_recuperado['nota'].fillna('').str.startswith('Recuperado de comentario')]
display(recuperadas.groupby(['variable', 'valor']).size().rename('filas').to_frame())
recuperadas[['campaña', 'id_participante', 'sub_ensayo', 'variedad_id', 'variable', 'valor', 'nota']].tail(5)

filas
variable                 valor                                                    
condicion_poscosecha     A temperatura ambiente                                  4
                         No se evaluó                                            3
control_malezas          Retiro manual o con herramientas                        5
control_plagas           Biopreparados                                          15
desbrote                 Sí                                                      3
eliminacion_brote_apical No                                                      1
fertilizacion            Abonos líquidos o biopreparados                         8
                         Compost, lombricompuesto, humus o bokashi               8
                         Fertilizantes químicos o comerciales                    3
metodo_siembra           Almácigo en bandeja                                     8
raleo_frutos             No                                                      1
                         Sí                                                      2
sanidad_observada        Bichitos chicos (pulgones, arañuelas, chinches,...     29
                         Bichos grandes (orugas, babosas, langostas, etc.)       1
                         Hojas amarillas, enruladas o deformadas                 1
                         Mordeduras de mamíferos o picoteos de aves en f...      2
                         Polvillo o manchas blancas sobre las hojas              7
sustrato                 Sustrato comercial                                      3
ubicacion_parcela        En invernadero                                          3

,campaña,id_participante,sub_ensayo,variedad_id,variable,valor,nota
11327,2023-2024,AA023T,1,V172,sanidad_observada,"Bichos grandes (orugas, babosas, langostas, etc.)",Recuperado de comentario [P06] (comentario en ...
11328,2023-2024,AA023T,1,V56,sanidad_observada,"Bichitos chicos (pulgones, arañuelas, chinches...",Recuperado de comentario [P08] (comentario en ...
11329,2023-2024,AA023T,1,V56,sanidad_observada,Polvillo o manchas blancas sobre las hojas,Recuperado de comentario [P11] (comentario en ...
11330,2025-2026,AA023T,1,V56,sanidad_observada,"Bichitos chicos (pulgones, arañuelas, chinches...",Recuperado de comentario [P07] (comentario en ...
11331,2025-2026,AA023T,1,V56,sanidad_observada,Mordeduras de mamíferos o picoteos de aves en ...,Recuperado de comentario [P10] (comentario en ...


In [212]:
df_sin_regla.head(5)

,campaña,etapa,comentario,comentario_normalizado,frecuencia
0,2022-2023,Trasplante a suelo,no hago surcos,no hago surcos,9
1,2024-2025,Labores durante fructificación,deshoje parte baja de la planta,deshoje parte baja de la planta,4
2,2023-2024,Crecimiento y desarrollo,Crecimiento normal,crecimiento normal,4
3,2023-2024,Trasplante a maceta,crecimiento normal,crecimiento normal,4
4,2024-2025,Trasplante a suelo,"implantadas por daño en el cultivo, ocasionado...","implantadas por dano en el cultivo, ocasionado...",4


#### 2.3.13 Validación cruzada de rangos y fechas que toman los datos de las variables

- Verificamos que se cumplan las condiciones/restricciones naturales del ensayo, por ejemplo:

   - Cantidad de semillas sembradas >= Cantidad de semillas emergidas
   - Fecha de siembra <= Fecha de trasplante

Para el arreglo y limpieza de estos casos, se analizaron las planillas de forma manual y se realizó lo siguiente:

- Se borraron algunas observaciones que presentaban datos incorrectos, mal generados por el modelo o que el evaluador cargó, pero en los comentarios indicaba que realizó otro proceso.

- Se imputaron observaciones, arreglando en muchos casos los problemas de validación cruzada obtenidos en el reporte. Principalmente eran casos en los que el evaluador no escribió el año en la fecha y el modelo generador de los datos infirió un año incorrecto.

- Algunas fechas que aparecen en el reporte son correctas y no requieren modificación. La validación se centra en capturar todos los casos excepcionales también, que no tienen porque ser un error.

In [213]:
def auditar_incongruencias(df, col_variable='variable', col_valor='valor'):
    columnas_clave = ['campaña', 'id_participante', 'id_sitio', 'sub_ensayo', 'variedad_id', 'variedad_original']
    claves_presentes = [c for c in columnas_clave if c in df.columns]
    
    df_pivot = df.pivot_table( # Pasa a formato ancho
        index=claves_presentes,
        columns=col_variable,
        values=col_valor,
        aggfunc='first'
    ).reset_index()
    vars_numericas = [
        'cantidad_semillas_sembradas',
        'cantidad_plantulas_emergidas',
        'cantidad_plantines_trasplantados',
        'cantidad_plantines_implantados',
        'cantidad_plantas_trasplantadas_parcela',
        'cantidad_plantas_implantadas'
    ]
    vars_fechas = [
        'fecha_siembra',
        'fecha_emergencia',
        'fecha_trasplante_maceta',
        'fecha_trasplante_suelo',
        'fecha_primera_floracion',
        'fecha_primera_cosecha',
        'fecha_ultima_cosecha'
    ]
    # Forzar tipos de datos, si rompen queda null
    for col in vars_numericas:
        if col in df_pivot.columns:
            df_pivot[col] = pd.to_numeric(df_pivot[col], errors='coerce')
    for col in vars_fechas:
        if col in df_pivot.columns:
            df_pivot[col] = pd.to_datetime(df_pivot[col], errors='coerce')
            
    alertas = []
    
    # REVISIÓN DE RANGOS (en las cantidades)
    def check_mayor(col_menor, col_mayor, mensaje):
        if col_menor in df_pivot.columns and col_mayor in df_pivot.columns:
            mask = df_pivot[col_mayor] > df_pivot[col_menor]
            if mask.any():
                temp = df_pivot[mask][claves_presentes + [col_menor, col_mayor]].copy()
                temp['Tipo_Error'] = mensaje
                alertas.append(temp)

    check_mayor('cantidad_semillas_sembradas', 'cantidad_plantulas_emergidas', 'Emergidas > Sembradas')
    check_mayor('cantidad_plantines_trasplantados', 'cantidad_plantines_implantados', 'Implantadas > Trasplantadas (Maceta)')
    check_mayor('cantidad_plantas_trasplantadas_parcela', 'cantidad_plantas_implantadas', 'Implantadas > Trasplantadas (Suelo)')
    # REVISIÓN DEL ORDEN CRONOLÓGICO (fechas)
    def check_fecha(col_antes, col_despues, mensaje):
        if col_antes in df_pivot.columns and col_despues in df_pivot.columns:
            mask = df_pivot[col_despues] < df_pivot[col_antes]
            if mask.any():
                temp = df_pivot[mask][claves_presentes + [col_antes, col_despues]].copy()
                temp['Tipo_Error'] = mensaje
                alertas.append(temp)
                
    check_fecha('fecha_siembra', 'fecha_emergencia', 'Emergencia ANTERIOR a Siembra')
    check_fecha('fecha_emergencia', 'fecha_trasplante_maceta', 'Trasplante (Maceta) ANTERIOR a Emergencia')
    check_fecha('fecha_emergencia', 'fecha_trasplante_suelo', 'Trasplante (Suelo) ANTERIOR a Emergencia')
    check_fecha('fecha_trasplante_maceta', 'fecha_trasplante_suelo', 'Trasplante (Suelo) ANTERIOR a Trasplante (Maceta)')
    check_fecha('fecha_trasplante_suelo', 'fecha_primera_floracion', 'Floración ANTERIOR a Trasplante (Suelo)')
    check_fecha('fecha_primera_floracion', 'fecha_primera_cosecha', 'Cosecha ANTERIOR a Floración')
    check_fecha('fecha_primera_cosecha', 'fecha_ultima_cosecha', 'Última Cosecha ANTERIOR a Primera Cosecha')

    if alertas:
        df_alertas = pd.concat(alertas, ignore_index=True)
        cols_finales = claves_presentes + ['Tipo_Error'] + [c for c in df_alertas.columns if c not in claves_presentes + ['Tipo_Error']]
        return df_alertas[cols_finales]
    else:
        print("No se encontraron incongruencias lógicas o cronológicas en los datos.")
        return pd.DataFrame()

df_reporte_errores = auditar_incongruencias(df_final_recuperado, col_variable='variable')
df_reporte_errores.to_csv('data/interim/reporte_inconsistencias_variables.csv', index=False)

Se utilizó el archivo_reporte_inconsistencias_variables.csv para descubrir estas fechas mal parseadas por el modelo de IA. Sin embargo, este reporte no pudo encontrar todos estos casos mal parseados, debido a que generalmente si el modelo fallaba en inferir una fecha (por la falta de especificación del año del evaluador o porque invertía el mes con el día) lo hacía para todas las variables de ese participante. 
Por ende requirió un análisis muy cuidadoso y manual de las planillas originales para captar todos estos casos.

In [214]:
# Estrategia unificada de correcciones manuales
CASOS_A_CORREGIR = [
    
    # ACCIÓN 1: MODIFICACIÓN Y CORRECCIÓN DE VALORES EXISTENTES
    
    # --- 1.1 Inconsistencias biológicas / numéricas ---
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'AA152T', 'sub_ensayo': 1, 'variedad_id': 'V37'}, 'correcciones': {'cantidad_semillas_sembradas': (17, 'Siembra y emergencia')}}, # Corrección de conteo de semillas
    {'filtros': {'campaña': '2024-2025', 'id_participante': 'AA001T', 'sub_ensayo': 1, 'variedad_id': 'V7'},  'correcciones': {'cantidad_plantas_implantadas': (0, 'Trasplante a suelo')}},   # Ajuste a 0 según comentarios del evaluador

    # --- 1.2 Fechas invertidas: Emergencia anterior a Siembra ---
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'AA067T', 'sub_ensayo': 1, 'variedad_id': 'V172'}, 'correcciones': {'fecha_emergencia': ('2023-09-11', 'Siembra y emergencia')}}, # Corrección de mes
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'AA067T', 'sub_ensayo': 1, 'variedad_id': 'V56'},  'correcciones': {'fecha_emergencia': ('2023-09-17', 'Siembra y emergencia')}}, # Corrección de mes
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'AA067T', 'sub_ensayo': 1, 'variedad_id': 'V7'},   'correcciones': {'fecha_emergencia': ('2023-09-11', 'Siembra y emergencia')}}, # Corrección de mes
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H035T',  'sub_ensayo': 1, 'variedad_id': 'V7'},   'correcciones': {'fecha_siembra': ('2023-06-17', 'Siembra y emergencia')}},    # Corrección de fecha de carga

    # --- 1.3 Fechas mal interpretadas o sin año explícito (Claude no logró cargarlas bien al conjunto de datos) ---
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'AA048T', 'sub_ensayo': 1, 'variedad_id': 'V7'},   'correcciones': {'fecha_siembra': ('2023-09-16', 'Siembra y emergencia'), 'fecha_emergencia': ('2023-09-21', 'Siembra y emergencia'), 'fecha_trasplante_suelo': ('2023-11-08', 'Trasplante a suelo')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'AA048T', 'sub_ensayo': 1, 'variedad_id': 'V56'},  'correcciones': {'fecha_siembra': ('2023-09-16', 'Siembra y emergencia'), 'fecha_emergencia': ('2023-09-21', 'Siembra y emergencia'), 'fecha_trasplante_maceta': ('2023-10-10', 'Trasplante a maceta'), 'fecha_trasplante_suelo': ('2023-11-08', 'Trasplante a suelo')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'AA048T', 'sub_ensayo': 1, 'variedad_id': 'V172'}, 'correcciones': {'fecha_siembra': ('2023-09-16', 'Siembra y emergencia'), 'fecha_emergencia': ('2023-09-21', 'Siembra y emergencia'), 'fecha_trasplante_maceta': ('2023-10-10', 'Trasplante a maceta'), 'fecha_trasplante_suelo': ('2023-11-08', 'Trasplante a suelo')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'AA052T', 'sub_ensayo': 1, 'variedad_id': 'V56'},  'correcciones': {'fecha_trasplante_suelo': ('2023-12-31', 'Trasplante a suelo')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H043T',  'sub_ensayo': 1, 'variedad_id': 'V56'},  'correcciones': {'fecha_trasplante_suelo': ('2023-10-04', 'Trasplante a suelo')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H043T',  'sub_ensayo': 1, 'variedad_id': 'V7'},   'correcciones': {'fecha_trasplante_suelo': ('2023-10-04', 'Trasplante a suelo')}},

    # --- 1.4 Fechas invertidas: Trasplante a maceta anterior a Emergencia ---
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'AA017T', 'sub_ensayo': 1, 'variedad_id': 'V65'},    'correcciones': {'fecha_trasplante_maceta': ('2022-10-10', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'AA017T', 'sub_ensayo': 1, 'variedad_id': 'VUCO18'}, 'correcciones': {'fecha_trasplante_maceta': ('2022-10-10', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'AA023T', 'sub_ensayo': 1, 'variedad_id': 'V65'},    'correcciones': {'fecha_trasplante_maceta': ('2022-10-03', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'AA148T', 'sub_ensayo': 1, 'variedad_id': 'V104'},   'correcciones': {'fecha_trasplante_maceta': ('2022-11-05', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'AA148T', 'sub_ensayo': 1, 'variedad_id': 'V34'},    'correcciones': {'fecha_trasplante_maceta': ('2022-11-05', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'AA148T', 'sub_ensayo': 1, 'variedad_id': 'V97'},    'correcciones': {'fecha_trasplante_maceta': ('2022-11-05', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'H004T',  'sub_ensayo': 1, 'variedad_id': 'V89'},    'correcciones': {'fecha_trasplante_maceta': ('2022-11-01', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'H007T',  'sub_ensayo': 1, 'variedad_id': 'V88'},    'correcciones': {'fecha_trasplante_maceta': ('2022-10-09', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'H010T',  'sub_ensayo': 1, 'variedad_id': 'V80'},    'correcciones': {'fecha_trasplante_maceta': ('2022-12-05', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'H026T',  'sub_ensayo': 1, 'variedad_id': 'V90'},    'correcciones': {'fecha_trasplante_maceta': ('2022-11-05', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'H027T',  'sub_ensayo': 1, 'variedad_id': 'V115'},   'correcciones': {'fecha_trasplante_maceta': ('2022-11-06', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'H027T',  'sub_ensayo': 1, 'variedad_id': 'V66'},    'correcciones': {'fecha_trasplante_maceta': ('2022-11-06', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'H037T',  'sub_ensayo': 1, 'variedad_id': 'V11'},    'correcciones': {'fecha_trasplante_maceta': ('2022-12-01', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'H037T',  'sub_ensayo': 1, 'variedad_id': 'V91'},    'correcciones': {'fecha_trasplante_maceta': ('2022-12-01', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'H037T',  'sub_ensayo': 1, 'variedad_id': 'V97'},    'correcciones': {'fecha_trasplante_maceta': ('2022-12-01', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'AA020T', 'sub_ensayo': 1, 'variedad_id': 'V56'},    'correcciones': {'fecha_trasplante_maceta': ('2023-09-04', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'AA138T', 'sub_ensayo': 2, 'variedad_id': 'V90'},    'correcciones': {'fecha_trasplante_maceta': ('2023-10-09', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H018T',  'sub_ensayo': 1, 'variedad_id': 'V172'},   'correcciones': {'fecha_trasplante_maceta': ('2023-09-08', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H018T',  'sub_ensayo': 1, 'variedad_id': 'V56'},    'correcciones': {'fecha_trasplante_maceta': ('2023-09-08', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H018T',  'sub_ensayo': 1, 'variedad_id': 'V7'},     'correcciones': {'fecha_trasplante_maceta': ('2023-09-08', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H024T',  'sub_ensayo': 1, 'variedad_id': 'V172'},   'correcciones': {'fecha_trasplante_maceta': ('2023-11-07', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H024T',  'sub_ensayo': 1, 'variedad_id': 'V56'},    'correcciones': {'fecha_trasplante_maceta': ('2023-11-07', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H024T',  'sub_ensayo': 1, 'variedad_id': 'V7'},     'correcciones': {'fecha_trasplante_maceta': ('2023-11-05', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H027T',  'sub_ensayo': 1, 'variedad_id': 'V66'},    'correcciones': {'fecha_trasplante_maceta': ('2023-08-01', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H051T',  'sub_ensayo': 1, 'variedad_id': 'V33'},    'correcciones': {'fecha_trasplante_maceta': ('2023-10-05', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H051T',  'sub_ensayo': 1, 'variedad_id': 'V43'},    'correcciones': {'fecha_trasplante_maceta': ('2023-10-05', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H051T',  'sub_ensayo': 1, 'variedad_id': 'V56'},    'correcciones': {'fecha_trasplante_maceta': ('2023-10-05', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H051T',  'sub_ensayo': 1, 'variedad_id': 'V7'},     'correcciones': {'fecha_trasplante_maceta': ('2023-10-05', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H051T',  'sub_ensayo': 1, 'variedad_id': 'V80'},    'correcciones': {'fecha_trasplante_maceta': ('2023-10-05', 'Trasplante a maceta')}},
    {'filtros': {'campaña': '2024-2025', 'id_participante': 'AA014T', 'sub_ensayo': 1, 'variedad_id': 'V56'},    'correcciones': {'fecha_trasplante_maceta': ('2024-12-10', 'Trasplante a maceta')}},
    
    # --- 1.5 Corrección con ancla Regex para evitar ambigüedad en variedad_original ---
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AA023T', 'sub_ensayo': 1, 'variedad_id': 'V56', 'variedad_original': r'^56$'}, 'correcciones': {'fecha_trasplante_maceta': ('2025-10-01', 'Trasplante a maceta')}},

    # --- 1.6 Fechas invertidas: Cosechas anteriores a Floración ---
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H037T', 'sub_ensayo': 1, 'variedad_id': 'V172'}, 'correcciones': {'fecha_primera_cosecha': ('2024-01-27', 'Cosecha y poscosecha')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H037T', 'sub_ensayo': 1, 'variedad_id': 'V172'}, 'correcciones': {'fecha_ultima_cosecha': ('2024-02-08', 'Cosecha y poscosecha')}},

    # --- 1.7 Reclasificación de ensayo (AA013T: Falso suelo (Tomo un tacho como trasplante a suelo) reasignado a maceta) ---
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AA013T', 'variedad_id': 'VCARC', 'variedad_original': 'Carcione'},
     'correcciones': {
         'ubicacion_contenedor': ('Al aire libre', 'Trasplante a maceta'),
         'cantidad_plantines_trasplantados': (3, 'Trasplante a maceta'),
         'fecha_trasplante_maceta': ('2025-10-31', 'Trasplante a maceta'),
         'comentarios_generales': ('Directo a tierra. NOTA DEL EDA: EL EVALUADOR TOMO UN TACHO COMO TIERRA DIRECTA', 'Trasplante a maceta')
        }
    },

    # --- 1.8 Fechas invertidas que no pudo detectar la función de auditar_incongruencias ---
    # 2025-2026
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AA005T', 'sub_ensayo': 1, 'variedad_id': 'V113'}, 'correcciones': {'fecha_siembra': '2025-10-03'}},
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AA005T', 'sub_ensayo': 1, 'variedad_id': 'V56'}, 'correcciones': {'fecha_siembra': '2025-10-03', 'fecha_emergencia': '2025-10-11'}},
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AA005T', 'sub_ensayo': 1, 'variedad_id': 'VCARC'}, 'correcciones': {'fecha_siembra': '2025-10-03', 'fecha_emergencia': '2025-10-09'}},
    
    # 2023-2024
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H014T', 'sub_ensayo': 1, 'variedad_id': 'V56'}, 'correcciones': {'fecha_trasplante_maceta': '2024-10-02'}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H014T', 'sub_ensayo': 1, 'variedad_id': 'V7'}, 'correcciones': {'fecha_trasplante_maceta': '2024-10-02'}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H051T', 'sub_ensayo': 1, 'variedad_id': 'V43'}, 'correcciones': {'fecha_ultima_cosecha': '2024-02-01'}},


    # ACCIÓN 2: BORRADO DE REGISTROS (SETEAMOS A NA)
    
    # --- 2.1 Fechas inconsistentes irrecuperables ---
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AA013T', 'sub_ensayo': 1, 'variedad_id': 'V113'}, 'correcciones': {'fecha_emergencia': (pd.NA, 'Siembra y emergencia')}},
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AA013T', 'sub_ensayo': 1, 'variedad_id': 'V56'},  'correcciones': {'fecha_emergencia': (pd.NA, 'Siembra y emergencia')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'H037T',  'sub_ensayo': 1, 'variedad_id': 'V11'},  'correcciones': {'fecha_primera_cosecha': (pd.NA, 'Cosecha y poscosecha')}},
    {'filtros': {'campaña': '2022-2023', 'id_participante': 'H037T',  'sub_ensayo': 1, 'variedad_id': 'V11'},  'correcciones': {'fecha_ultima_cosecha': (pd.NA, 'Cosecha y poscosecha')}},

    # --- 2.2 Anulación de campos de suelo erróneamente cargados ---
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AA013T', 'variedad_id': 'VCARC', 'variedad_original': 'Carcione'},
     'correcciones': {
         'cobertura_suelo': (pd.NA, 'Trasplante a suelo'),
         'cantidad_plantas_trasplantadas_parcela': (pd.NA, 'Trasplante a suelo'),
         'fecha_trasplante_suelo': (pd.NA, 'Trasplante a suelo'),
         'superficie_parcela_m2': (pd.NA, 'Trasplante a suelo'),
         'ubicacion_parcela': (pd.NA, 'Trasplante a suelo')
    }},
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AB013T', 'id_sitio': 'AB013T-Derqui'}, 'correcciones': {'ubicacion_contenedor': (pd.NA, 'Trasplante a maceta')}},


    # ACCIÓN 3: AGREGADO DE VARIABLES / OBSERVACIONES FALTANTES

    # --- 3.1 Rescate de cultivos asociados en suelo (H016T) ---
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H016T', 'variedad_id': 'V56', 'sub_ensayo': 1}, 'correcciones': {'cultivo_asociado_suelo': ('Aromáticas y medicinales', 'Trasplante a suelo')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H016T', 'variedad_id': 'V7',  'sub_ensayo': 1}, 'correcciones': {'cultivo_asociado_suelo': ('Aromáticas y medicinales, Frutas y hortalizas', 'Trasplante a suelo')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H016T', 'variedad_id': 'V56', 'sub_ensayo': 2}, 'correcciones': {'cultivo_asociado_suelo': ('Frutas y hortalizas, Aromáticas y medicinales', 'Trasplante a suelo')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'H016T', 'variedad_id': 'V7',  'sub_ensayo': 2}, 'correcciones': {'cultivo_asociado_suelo': ('Frutas y hortalizas, Aromáticas y medicinales', 'Trasplante a suelo')}},

    # --- 3.2 Incorporación de fechas de siembra no procesadas inicialmente (AA020T) ---
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'AA020T', 'variedad_id': 'V56'},  'correcciones': {'fecha_siembra': ('2023-08-30', 'Siembra y emergencia')}},
    {'filtros': {'campaña': '2023-2024', 'id_participante': 'AA020T', 'variedad_id': 'V172'}, 'correcciones': {'fecha_siembra': ('2023-08-30', 'Siembra y emergencia')}},

    # --- 3.3 Carga de observaciones de maceta faltantes (AB013T - UNSAM) ---
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AB013T', 'id_sitio': 'AB013T-UNSAM', 'variedad_id': 'VCARC'},
     'correcciones': {
         'ubicacion_contenedor': ('En invernadero', 'Trasplante a maceta'),
         'profundidad_contenedor_cm': (12, 'Trasplante a maceta'),
         'sustrato': (f'{CAT_CON_MO}, {CAT_CON_MIN}', 'Trasplante a maceta'),
         'fecha_trasplante_maceta': ('2025-09-22', 'Trasplante a maceta'),
         'cantidad_plantines_trasplantados': (7, 'Trasplante a maceta'),
         'cantidad_plantines_implantados': (7, 'Trasplante a maceta'),

         'fecha_primera_floracion': ('2025-12-10', 'Crecimiento y desarrollo')
     }},
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AB013T', 'id_sitio': 'AB013T-UNSAM', 'variedad_id': 'V56'},
     'correcciones': {
         'ubicacion_contenedor': ('En invernadero', 'Trasplante a maceta'),
         'profundidad_contenedor_cm': (12, 'Trasplante a maceta'),
         'sustrato': (f'{CAT_CON_MO}, {CAT_CON_MIN}', 'Trasplante a maceta'),
         'fecha_trasplante_maceta': ('2025-09-22', 'Trasplante a maceta'),
         'cantidad_plantines_trasplantados': (1, 'Trasplante a maceta'),
         'cantidad_plantines_implantados': (1, 'Trasplante a maceta'),

         'fecha_primera_floracion': ('2025-12-10', 'Crecimiento y desarrollo')
     }},
    {'filtros': {'campaña': '2025-2026', 'id_participante': 'AB013T', 'id_sitio': 'AB013T-UNSAM', 'variedad_id': 'V113'},
     'correcciones': {
         'ubicacion_contenedor': ('En invernadero', 'Trasplante a maceta'),
         'profundidad_contenedor_cm': (12, 'Trasplante a maceta'),
         'sustrato': (f'{CAT_CON_MO}, {CAT_CON_MIN}', 'Trasplante a maceta'),
         'fecha_trasplante_maceta': ('2025-09-22', 'Trasplante a maceta'),
         'cantidad_plantines_trasplantados': (6, 'Trasplante a maceta'),
         'cantidad_plantines_implantados': (6, 'Trasplante a maceta'),

         'fecha_primera_floracion': ('2025-12-10', 'Crecimiento y desarrollo')
     }}
]

Se modificó la función del inicio `corregir_observaciones_manuales`, para que pueda ser utilizado con las transformaciones realizadas sobre el df_final_recuperado.
Por eso la función que usamos aca es la misma que se usa al inicio de la limpieza

Además de las correcciones detectadas arriba, borramos las observaciones que se pasaron a null y los datos de un usuario (H016T) de la etapa transplante a maceta porque la completó, pero en los comentarios mencionó que plantó directamente a suelo.

In [215]:
# Solo estas variables separan una cadena con comas en varias filas.
# El resto conserva el texto completo, aunque contenga comas.
VARIABLES_MULTIVALOR_CORRECCION_FINAL = {
    'sustrato',
    'cultivo_asociado_maceta',
    'cultivo_asociado_suelo',
    'sanidad_observada',
    'fertilizacion',
    'control_plagas',
    'causa_desuniformidad'
}

for caso in CASOS_A_CORREGIR:
    df_final_recuperado = corregir_observaciones_manuales(
        df_final_recuperado,
        caso['filtros'],
        caso['correcciones'],
        col_variable='variable',
        modo='reemplazar',
        variables_multivalor=VARIABLES_MULTIVALOR_CORRECCION_FINAL,
        filtro='contiene'
    )

mask_borrar_maceta_h016t = (
    (df_final_recuperado['campaña'] == '2023-2024') & 
    (df_final_recuperado['id_participante'] == 'H016T') & 
    (df_final_recuperado['variedad_id'].isin(['V56', 'V7'])) & 
    (df_final_recuperado['etapa'] == 'Trasplante a maceta')
)
df_final_recuperado = df_final_recuperado[~mask_borrar_maceta_h016t]
sin_valor = df_final_recuperado['valor'].isna()
print(f"Filas convertidas a NA que se eliminan: {sin_valor.sum()}")
df_final_recuperado = df_final_recuperado[~sin_valor].reset_index(drop=True)

print(f"Proceso finalizado correctamente. Registros resultantes: {len(df_final_recuperado)}")

Filas convertidas a NA que se eliminan: 10
Proceso finalizado correctamente. Registros resultantes: 11332


#### 2.3.14 Cierre: etiquetas de variedad, variedad_original, sub-ensayos y validación final

Antes de exportar se cierran dos cosas que se detectaron al revisar el dataset final y se hace una validación final:

- **Mismo participante, misma campaña y misma variedad pero semillas de distinto año** (`56` vs `56 (2023)` / `56 cosecha 2024`, AA023T): son experimentos distintos con los mismos datos cargados dos veces, así que van en `sub_ensayo` 1 y 2 (diccionario `SUBENSAYO_POR_ETIQUETA`). 
- **variedad_original**: a lo largo del proceso se utilizó como identificador de las unidades, debido a que fue un inventó del modelo de IA para diferenciar ciertos casos. Por esto, ahora en la etapa final vamos a borrar esta variable y reemplazarla por `anio_multiplicacion` que se empezó a tomar recién en esta campaña y que las únicas filas que van a contener un dato no nulo son los experimentos del item de arriba.

**Validación final** (`validar_dataset_final`): sin nulos, sin filas repetidas, identidad de ensayo única, variables de un valor con un solo valor por ensayo, sin contradicciones "ninguno / no hubo" + dato, rangos y formatos. Si algo falla, se corta antes de exportar.


In [216]:
CAMBIOS_SUBENSAYO_Y_ETIQUETA = {
    # (campaña, id_participante, variedad_id, variedad_original_actual): (nuevo_sub_ensayo, nueva_variedad_original)
    ('2024-2025', 'AA023T', 'V172', '172 (2023)'):       (2, '2024'),
    ('2024-2025', 'AA023T', 'V56',  '56 (2023)'):        (2, '2024'),
    ('2025-2026', 'AA023T', 'V56',  '56 cosecha 2024'): (2, '2025'),
}

for (camp, part, var, etiqueta_vieja), (sub, etiqueta_nueva) in CAMBIOS_SUBENSAYO_Y_ETIQUETA.items():
    mask = (
        (df_final_recuperado['campaña'] == camp) & 
        (df_final_recuperado['id_participante'] == part) & 
        (df_final_recuperado['variedad_id'] == var) & 
        (df_final_recuperado['variedad_original'] == etiqueta_vieja)
    )
    assert mask.any(), f"No se encontró {(camp, part, var, etiqueta_vieja)}: ¿cambió la etiqueta previa?"
    df_final_recuperado.loc[mask, 'sub_ensayo'] = sub
    df_final_recuperado.loc[mask, 'variedad_original'] = etiqueta_nueva
    print(f"sub_ensayo={sub} | nueva variedad_original='{etiqueta_nueva}': {camp} {part} {var} ({mask.sum()} filas)")


anio = pd.to_numeric(df_final_recuperado['variedad_original'], errors='coerce')

# Filtro por rango: evita que una etiqueta numérica tipo "172" o "56" se confunda con un año
anio = anio.where(anio.between(1990, 2100))

df_final_recuperado['variedad_original'] = anio.astype('Int64')   # entero nullable (acepta NA)
df_final_recuperado = df_final_recuperado.rename(columns={'variedad_original': 'anio_multiplicacion' })
print("\n")

# Chequeo: tienen que quedar solo las filas que corregimos arriba
print(df_final_recuperado['anio_multiplicacion'].value_counts(dropna=False))
assert df_final_recuperado['anio_multiplicacion'].notna().sum() > 0, "Se borraron todos los años: revisar el paso 2"

sub_ensayo=2 | nueva variedad_original='2024': 2024-2025 AA023T V172 (37 filas)
sub_ensayo=2 | nueva variedad_original='2024': 2024-2025 AA023T V56 (39 filas)
sub_ensayo=2 | nueva variedad_original='2025': 2025-2026 AA023T V56 (50 filas)


anio_multiplicacion
<NA>    11206
2024       76
2025       50
Name: count, dtype: Int64


In [217]:
# 4. Validación final: si algo de esto falla, NO se exporta
VARIABLES_MULTIVALOR_FINAL = {'sustrato', 'cultivo_asociado_maceta', 'cultivo_asociado_suelo', 'sanidad_observada',
                              'fertilizacion', 'control_plagas', 'causa_desuniformidad'}
VALORES_NINGUNO_FINAL = {'sanidad_observada': {'No se observó ningún problema'}, 'fertilizacion': {'No utilicé'}, 'control_plagas': {'No realicé control'},
                         'cultivo_asociado_maceta': {'No hubo'}, 'cultivo_asociado_suelo': {'No hubo'}}
VARIABLES_SI_NO = ['desbrote', 'tutorado', 'raleo_frutos', 'eliminacion_brote_apical']
VARIABLES_ESCALA_1_5 = ['sanidad', 'tolerancia_estres_hidrico', 'tolerancia_quemadura_sol', 'uniformidad']


def validar_dataset_final(df, verbose=True):
    """Devuelve un DataFrame con cada chequeo y cuántos casos hay (0 = OK). Los chequeos ERROR frenan la exportación."""
    clave = ['campaña', 'id_participante', 'id_sitio', 'sub_ensayo', 'variedad_id']      # identidad del ensayo
    datos = df[df['variable'] != 'comentarios_generales']
    checks = []

    def agregar(nombre, n, nivel='ERROR'):
        checks.append({'chequeo': nombre, 'casos': int(n), 'nivel': nivel, 'estado': 'OK' if n == 0 else nivel})

    agregar('filas con valor/clave nula', df[clave + ['etapa', 'variable', 'valor']].isna().any(axis=1).sum())
    agregar('filas repetidas (ensayo + etapa + variable + valor)', df.duplicated(clave + ['etapa', 'variable', 'valor']).sum())

    # variables de un solo valor con 2 o + valores en el mismo ensayo
    por_ensayo = datos.groupby(clave + ['variable'])['valor'].nunique()
    simples = por_ensayo[(por_ensayo > 1) & ~por_ensayo.index.get_level_values('variable').isin(VARIABLES_MULTIVALOR_FINAL)]
    agregar('variables de 1 valor con 2+ valores en el mismo ensayo', len(simples))

    # "ninguno / no hubo" conviviendo con otro dato de la misma variable 
    n_contra = 0
    for var, vals_ninguno in VALORES_NINGUNO_FINAL.items():
        g = datos[datos['variable'] == var].groupby(clave)['valor'].apply(set)
        n_contra += sum(1 for s in g if (s & vals_ninguno) and (s - vals_ninguno))
    agregar("contradicciones 'ninguno / no hubo' + otro valor", n_contra)

    # tipos y rangos
    si_no = datos[datos['variable'].isin(VARIABLES_SI_NO)]
    agregar("variables Sí/No con valores distintos de 'Sí' / 'No'", (~si_no['valor'].isin(['Sí', 'No'])).sum())
    for nombre, variables, lo, hi in [('escala 1-5', VARIABLES_ESCALA_1_5, 1, 5), ('gusto_general 1-9', ['gusto_general'], 1, 9)]:
        x = pd.to_numeric(datos.loc[datos['variable'].isin(variables), 'valor'], errors='coerce')
        agregar(f'{nombre}: valores no numéricos o fuera de rango', (x.isna() | (x < lo) | (x > hi)).sum())
    fechas = datos[datos['variable'].str.startswith('fecha_')]
    agregar('fechas que no son YYYY-MM-DD', pd.to_datetime(fechas['valor'], format='%Y-%m-%d', errors='coerce').isna().sum())
    

    resultado = pd.DataFrame(checks)
    if verbose:
        display(resultado)
    errores = resultado[(resultado['nivel'] == 'ERROR') & (resultado['casos'] > 0)]
    assert errores.empty, f"La validación final encontró problemas:\n{errores.to_string(index=False)}"
    return resultado


df_validacion = validar_dataset_final(df_final_recuperado)
print(df_final_recuperado.shape)

,chequeo,casos,nivel,estado
0,filas con valor/clave nula,0,ERROR,OK
1,filas repetidas (ensayo + etapa + variable + v...,0,ERROR,OK
2,variables de 1 valor con 2+ valores en el mism...,0,ERROR,OK
3,contradicciones 'ninguno / no hubo' + otro valor,0,ERROR,OK
4,variables Sí/No con valores distintos de 'Sí' ...,0,ERROR,OK
5,escala 1-5: valores no numéricos o fuera de rango,0,ERROR,OK
6,gusto_general 1-9: valores no numéricos o fuer...,0,ERROR,OK
7,fechas que no son YYYY-MM-DD,0,ERROR,OK


(11332, 10)


## 3. Resultados del preprocesamiento y limpieza

Variables ya resueltas completamente (mapeadas y validadas) con la cantidad de observaciones de cada una. 

*/*: Separa el slug del nombre 


##### Siembra y emergencia — 5 $\rightarrow$ 5
- Método de siembra / metodo_siembra: 143
- Fecha de siembra / fecha_siembra: 346
- Cantidad de semillas sembradas / cantidad_semillas_sembradas: 342
- Cantidad de plántulas emergidas / cantidad_plantulas_emergidas: 302
- Fecha de emergencia (cuando emerge el 70% de las plántulas) / fecha_emergencia: 271

##### Trasplante a maceta — 9 $\rightarrow$ 7
- Ubicación del contenedor / ubicacion_contenedor: 167
- Profundidad del contenedor (cm) / profundidad_contenedor_cm: 134
- Sustrato utilizado (...) / sustrato: 287
- Fecha de trasplante (maceta) / fecha_trasplante_maceta: 146
- Cantidad de plantines trasplantados / cantidad_plantines_trasplantados: 159
- Cantidad de plantines implantados / cantidad_plantines_implantados: 56
- Cultivo asociado / cultivo_asociado_maceta: 70
- Volumen: Discontinuada
- #por contenedor: Discontinuada

##### Trasplante a suelo — 12 $\rightarrow$ 8
- Ubicación de la parcela / ubicacion_parcela: 190
- Superficie de la parcela (m²) / superficie_parcela_m2: 139
- Fecha de trasplante (suelo) / fecha_trasplante_suelo: 117
- Cantidad de plantas trasplantadas a la parcela / cantidad_plantas_trasplantadas_parcela: 116
- Densidad (plantas/m²) / densidad_plantas_m2: 114
- Cobertura de suelo / cobertura_suelo: 184
- Cultivo asociado / cultivo_asociado_suelo: 207
- Cantidad de plantas implantadas / cantidad_plantas_implantadas: 55
- Surcos: Discontinuada
- Distancia surcos: Discontinuada
- Distancia plantas: Discontinuada
- Cultivo anteceso: Discontinuada

##### Labores — 7 $\rightarrow$ 7
- Desbrote / desbrote: 241
- Tutorado / tutorado: 247
- Conducción / conduccion: 201
- Fertilización / fertilizacion: 258
- Control de malezas / control_malezas: 231
- Riego / riego: 256
- Control de plagas / control_plagas: 242

##### Crecimiento y desarrollo — 8 $\rightarrow$ 7
- Fecha de primera floración (día en la que al menos la mitad de las plantas están en su primera floración) / fecha_primera_floracion: 168
- Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante) / tolerancia_estres_hidrico: 216
- Tolerancia a la quemadura del sol (1 = nada tolerante, 5 = muy tolerante) / tolerancia_quemadura_sol: 209
- Sanidad (1 = nada sana, 5 = muy sana) / sanidad: 217
- Muerte por virus rugoso del tomate: Discontinuada
- Uniformidad (1 = nada uniforme, 5 = muy uniforme) / uniformidad: 184
- ¿A qué crees que se debe la uniformidad observada? / causa_desuniformidad: 18
- Plagas observadas / sanidad_observada: 116

##### Labores durante fructificación — 2 $\rightarrow$ 2
- Raleo de frutos / raleo_frutos: 183
- Eliminación del brote apical / eliminacion_brote_apical: 220

##### Cosecha y poscosecha — 20 $\rightarrow$ 7
- Fecha de cosecha #1 a #10: Discontinuadas
- Rendimiento 1 A 6: Discontinuadas
- Color del fruto al momento de cosechar / color_fruto_cosecha: 197
- Duración poscosecha (...) / duracion_poscosecha_dias: 93
- Cantidad de frutos cosechados / total_frutos_cosechados: 81
- Peso de todos los frutos cosechados (KG) / peso_total_cosechado_kg: 83
- fecha_ultima_cosecha: 146
- fecha_primera_cosecha: 145
- condicion_poscosecha: 103

##### Degustación — 16 $\rightarrow$ 16
- ¿Cuánto te gusta este tomate? (...) / gusto_general: 163
- Dulce / dulce: 191
- Ácido / acido: 192
- Jugoso / jugoso: 192
- Persistente / persistente: 184
- Olor a tomate maduro / olor_tomate_maduro: 184
- Olor a tomate verde / olor_tomate_verde: 186
- Mucha pulpa / mucha_pulpa: 190
- Poca pulpa / poca_pulpa: 184
- Pulpa dura / pulpa_dura: 184
- Pulpa blanda / pulpa_blanda: 184
- Mucha semilla / mucha_semilla: 191
- Poca semilla / poca_semilla: 182
- Aguachento / aguachento: 186
- Pastoso / pastoso: 177
- Piel dura / piel_dura: 187

##### Comentarios
- Comentarios generales / comentarios_generales: 776

In [218]:
inspeccionar_valores(df_final_recuperado, 'fecha_primera_cosecha')


----------------------------------------------------------------------
🔸 DESPUÉS (datos limpios)
----------------------------------------------------------------------

=== fecha_primera_cosecha ===
Valor: 2023-02-09, Frecuencia: 5
Valor: 2023-01-25, Frecuencia: 4
Valor: 2023-01-08, Frecuencia: 3
Valor: 2022-12-18, Frecuencia: 3
Valor: 2026-02-01, Frecuencia: 3
Valor: 2022-12-23, Frecuencia: 2
Valor: 2023-03-12, Frecuencia: 2
Valor: 2023-02-08, Frecuencia: 2
Valor: 2023-01-31, Frecuencia: 2
Valor: 2023-02-28, Frecuencia: 2
Valor: 2023-02-10, Frecuencia: 2
Valor: 2023-01-20, Frecuencia: 2
Valor: 2023-02-01, Frecuencia: 2
Valor: 2026-01-27, Frecuencia: 2
Valor: 2026-01-19, Frecuencia: 2
Valor: 2024-12-23, Frecuencia: 2
Valor: 2024-02-09, Frecuencia: 2
Valor: 2023-12-06, Frecuencia: 2
Valor: 2024-03-01, Frecuencia: 2
Valor: 2024-01-27, Frecuencia: 2
Valor: 2026-01-28, Frecuencia: 2
Valor: 2026-01-29, Frecuencia: 2
Valor: 2026-03-04, Frecuencia: 2
Valor: 2026-01-02, Frecuencia: 2
Valor: 2

## 4. Armado del csv con las localides y provincias por sitio_id del ensayo: `sitios_ubicacion.csv`

Utilizando el archivo original sitios.csv (en "data/raw"), se modifica para que quede únicamente la información relevante y estandarizada porque actualmente:
- Mismas zonas aparecen escritas con distintas variantes.
- No esta normalizado, por lo que aparece el mismo usuario con el mimos sitio_id pero con provincias y localidades distintas.




### 4.1 Funciones utilizadas

In [219]:
def sin_tildes(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()

def norm(s):
    if pd.isna(s): return s
    s = re.sub(r"[.\-]", " ", sin_tildes(str(s)).lower())
    s = re.sub(r"\bgral\b", "general", s)
    return re.sub(r"\s+", " ", s).strip()

def mejor_nombre(serie):
    """Nos quedamos como estándar el valor más frecuente por los usuarios; en empate se prefiere con tildes."""
    c = serie.value_counts()
    top = c[c == c.max()].index
    return max(top, key=lambda x: (x != x.upper(), x != sin_tildes(x)))

### 4.2 Configuración inicial (provincias validas, estandarizaciones de nombres, recuperaciones y correcciones de sitios distintos para el mismo participante según la campaña)

In [220]:
# Configuración
RUTA_ORIG = "data/raw/sitios.csv"
CIUDAD_ALIAS = {
    "cipoletti": "cipolletti",
    "fernandez oro": "general fernandez oro",
    "c a b a": "ciudad autonoma de buenos aires",
    "caba": "ciudad autonoma de buenos aires",
}
CORRECCIONES_SITIO = {
    ("AA023T", "san isidro"): "martinez",
}
MUDANZAS = {
    ("AA001T", "2025-2026"): "AA001T-Maipu",
    ("H030T", "2023-2024"): "H030T-Rufino",
    ("AA013T", "2025-2026"): "AA013T-SantaFe",
}
PROVINCIAS_VALIDAS = {
    "buenos aires","caba","catamarca","chaco","chubut","cordoba","corrientes","entre rios",
    "formosa","jujuy","la pampa","la rioja","mendoza","misiones","neuquen","rio negro","salta",
    "san juan","san luis","santa cruz","santa fe","santiago del estero","tierra del fuego","tucuman",
}

### 4.3 Generación del archivo mapeado de los sitios y de un catálogo inicial de posibilidades

In [221]:
sitios_raw = pd.read_csv(RUTA_ORIG, dtype=str, encoding="utf-8-sig")
for c in ["provincia", "ciudad"]:
    sitios_raw[c] = sitios_raw[c].str.strip().replace({"s/d": pd.NA, "": pd.NA})

sitios_raw["prov_key"] = sitios_raw["provincia"].map(norm)
sitios_raw["ciu_key"] = sitios_raw["ciudad"].map(norm).replace(CIUDAD_ALIAS)
sitios_raw["ciu_key"] = [CORRECCIONES_SITIO.get((s, k), k) for s, k in zip(sitios_raw["id_sitio"], sitios_raw["ciu_key"])]
sitios_raw.loc[sitios_raw["ciu_key"] == "ciudad autonoma de buenos aires", "prov_key"] = "caba"

# nuevo id_sitio según mudanzas
sitios_raw["sitio_id"] = [MUDANZAS.get((s, c), s) for s, c in zip(sitios_raw["id_sitio"], sitios_raw["campaña"])]

# Verificaicón de mismo sitio, ubicación distinta -> resolver a mano
g = sitios_raw.groupby("sitio_id")[["prov_key", "ciu_key"]].nunique(dropna=True)
conf = sitios_raw[sitios_raw["sitio_id"].isin(g[(g > 1).any(axis=1)].index)]
if len(conf):
    raise ValueError("Resolver en MUDANZAS / CIUDAD_ALIAS:\n"
                     + conf.sort_values(["sitio_id", "campaña"])[["sitio_id", "campaña", "provincia", "ciudad"]].to_string())

# Verificación de provincias inválidas
invalidas = set(sitios_raw["prov_key"].dropna()) - PROVINCIAS_VALIDAS
assert not invalidas, f"Provincias no reconocidas: {invalidas}"

# nombres canónicos (se carga la grafía más frecuente, con tildes)
nombre_ciudad = sitios_raw.dropna(subset=["ciudad"]).groupby("ciu_key")["ciudad"].agg(mejor_nombre)
nombre_prov = sitios_raw.dropna(subset=["provincia"]).groupby("prov_key")["provincia"].agg(mejor_nombre)
sitios_raw["localidad_f"] = sitios_raw["ciu_key"].map(nombre_ciudad)
sitios_raw["provincia_f"] = sitios_raw["prov_key"].map(nombre_prov)

# una fila por sitio 
sitios = (sitios_raw.sort_values("campaña")
            .groupby("sitio_id", as_index=False)
            .agg(participante_id=("id_participante", "first"),
                 provincia=("provincia_f", "last"),
                 localidad=("localidad_f", "last")))
assert sitios["sitio_id"].is_unique
assert (sitios["localidad"].isna() | sitios["provincia"].notna()).all()  # no hay localidad sin provincia

sitios.to_csv("data/processed/sitios_ubicacion.csv", index=False)

### 4.4 Cambiamos en el conjunto de datos de las *observaciones agronómicas*, los valores de la columna id_sitio por los corregidos recién.

En observaciones agronómicas crudo, aparecen participantes con la misma clave en id_sitio a pesar de que registraron direcciones distintas según la campaña.

In [222]:
mapeo = sitios_raw[["id_sitio", "campaña", "sitio_id"]].drop_duplicates()

# Hacer el merge para traer la columna con el nuevo ID
df_final_recuperado = df_final_recuperado.merge(
    mapeo.rename(columns={"id_sitio": "sitio_id_orig", "sitio_id": "sitio_id_nuevo"}),
    left_on=["id_sitio", "campaña"],
    right_on=["sitio_id_orig", "campaña"],
    how="left"
)

df_final_recuperado["id_sitio"] = df_final_recuperado["sitio_id_nuevo"].fillna(df_final_recuperado["id_sitio"])
df_final_recuperado = df_final_recuperado.drop(columns=["sitio_id_orig", "sitio_id_nuevo"])

## 5 Guardamos el conjunto de datos final de las observaciones agronómicas: `observaciones_agronomicas_limpio.csv`

In [223]:
df_final_recuperado.to_csv('data/processed/observaciones_agronomicas_limpio.csv', index=False)
print(f"Exportado: {df_final_recuperado.shape[0]} filas, {df_final_recuperado.shape[1]} columnas")

Exportado: 11332 filas, 10 columnas


## 6. Armamos el conjunto de datos de los participantes `participantes.csv`

Simplemente va a ser una copia del actual, pero extrayendo solamente la columna del género para poder subirlo a la base de datos y consumirlo en el tablero a pedido de la dirección.

In [12]:
participantes_raw = pd.read_csv('data/raw/participantes.csv')
participantes_processed = participantes_raw[['id_participante', 'genero']]
participantes_processed = participantes_processed.rename(columns={'id_participante': 'participante_id'})
participantes_processed['genero'] = participantes_processed['genero'].replace('s/d', np.nan)
participantes_processed.head(2)

,participante_id,genero
0,AA001T,F
1,AA002T,F


In [13]:
participantes_processed.to_csv('data/processed/participantes.csv', index=False)

## 6. Pendientes

- Visualizar relaciones entre variables

- No es prioritario, pero diferenciar los null estructurales (no se preguntó en esa campaña) de los null reales (se preguntó, pero el evaluador no respondió).



